# Airbnb — Plantilla CNN explicada y lista para producción

Esta plantilla cubre la parte visual del trabajo: descarga de fotos, etiquetado, entrenamiento de CNN, comparación de arquitecturas, validación final y uso del modelo ganador en una nueva tanda de anuncios.

La lógica de particiones queda fija:

| Grupo | Uso |
|---|---|
| `train` | Fotos usadas para que el modelo aprenda. |
| `test` | Fotos usadas para comparar modelos y elegir el mejor. |
| `validacion` | Fotos usadas al final para comprobar el modelo ganador. |

No se vuelve a dividir la data después de la celda 8. El módulo CNN usa directamente la columna `split_calidad`.


In [24]:
# ============================================================
# CELDA 1 - Verificar entorno
# NO instalar ni actualizar librerías aquí
# ============================================================

import sys
import numpy as np
import pandas as pd

print("Python usado por el notebook:")
print(sys.executable)

print("\nVersiones principales:")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

if np.__version__.startswith("2."):
    raise RuntimeError("Tu notebook está usando NumPy 2.x. No continúes.")

print("\nEntorno OK. Puedes continuar.")

Python usado por el notebook:
c:\TF_DL\.venv\Scripts\python.exe

Versiones principales:
NumPy: 1.26.4
Pandas: 2.2.2

Entorno OK. Puedes continuar.


In [25]:
# ============================================================
# CELDA 2 - Verificar entorno
# ============================================================

import sys
print("Python usado por este notebook:")
print(sys.version)
print(sys.executable)

# En Windows + Jupyter, esto ayuda a evitar problemas de Playwright
import asyncio
if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    print("WindowsSelectorEventLoopPolicy activado.")

Python usado por este notebook:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
c:\TF_DL\.venv\Scripts\python.exe
WindowsSelectorEventLoopPolicy activado.


## Importante

Coloca tu Excel en la misma carpeta del notebook. La plantilla intentará encontrar automáticamente un archivo tipo `G4*.xlsx`.

Si tienes varios Excel en la carpeta, modifica manualmente `EXCEL_PATH` dentro del script.

In [ ]:
# ============================================================
# CELDA 3 - Crear script robusto de descarga
# ============================================================

from pathlib import Path

script = r'''
# ============================================================
# SCRIPT: descarga de fotos desde recorrido de Airbnb
# Se ejecuta desde el notebook, pero corre como proceso .py externo.
# Eso evita problemas de Playwright dentro del kernel de Jupyter.
# ============================================================

from pathlib import Path
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
import re
import json
import time
import hashlib
import shutil
import mimetypes
import base64
from datetime import datetime

import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError

# ============================================================
# CONFIGURACIÓN
# ============================================================

# Si lo dejas en None, busca automáticamente un Excel G4*.xlsx
EXCEL_PATH = None

SHEET_NAME = "Principal"
HEADER_ROW = 1
URL_COL = "URL Canónica"
ID_COL = "ID Airbnb"

# None = todos los links. Para prueba rápida usa 1, 2 o 3.
MAX_ALOJAMIENTOS = None

# Cantidad de scrolls en la vista de fotos.
# Si faltan fotos, sube este número.
SCROLLS_GALERIA = 140

# Visible para validar que realmente esté entrando al recorrido de fotos.
HEADLESS = False

# Esperas
ESPERA_INICIAL_SEGUNDOS = 6
ESPERA_SCROLL_SEGUNDOS = 0.8

# Tamaño mínimo renderizado para aceptar una imagen.
# Esto ayuda a ignorar íconos, avatars y miniaturas chicas.
MIN_RENDER_W = 180
MIN_RENDER_H = 120

# Salida solicitada por el enunciado: img/<ID Airbnb>/
OUT_DIR = Path("salida_airbnb_fotos")
IMG_DIR = Path("img")
DEBUG_DIR = OUT_DIR / "debug"
OUT_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)
DEBUG_DIR.mkdir(exist_ok=True)

METADATA_XLSX = OUT_DIR / "metadata_descarga_fotos.xlsx"
MANIFEST_CSV = OUT_DIR / "dataset_manifest_airbnb.csv"
MANIFEST_XLSX = OUT_DIR / "dataset_manifest_airbnb.xlsx"
REPORTE_HTML = OUT_DIR / "reporte_visual_fotos.html"
ZIP_NAME = "salida_airbnb_fotos"
ZIP_PATH = f"{ZIP_NAME}.zip"

# ============================================================
# UTILIDADES
# ============================================================

def encontrar_excel():
    if EXCEL_PATH:
        p = Path(EXCEL_PATH)
        if not p.exists():
            raise FileNotFoundError(f"No existe EXCEL_PATH: {p}")
        return p

    candidatos = sorted(Path(".").glob("G4*.xlsx"))
    candidatos = [p for p in candidatos if not p.name.startswith("~$")]
    if not candidatos:
        candidatos = sorted(Path(".").glob("*.xlsx"))
        candidatos = [p for p in candidatos if not p.name.startswith("~$") and "metadata" not in p.name.lower() and "manifest" not in p.name.lower()]

    if not candidatos:
        raise FileNotFoundError("No encontré ningún Excel. Coloca G4_mod.xlsx en la misma carpeta del notebook.")

    print("Excel detectado automáticamente:", candidatos[0])
    return candidatos[0]


def limpiar_texto(x):
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def id_desde_url(url):
    m = re.search(r"/rooms/(\d+)", str(url))
    return m.group(1) if m else ""


def normalizar_id_airbnb(id_excel, url):
    id_url = id_desde_url(url)
    if id_url:
        return id_url
    x = limpiar_texto(id_excel)
    if x.endswith(".0"):
        x = x[:-2]
    x = re.sub(r"[^0-9A-Za-z_-]", "_", x)
    return x or "sin_id"


def construir_url_recorrido_fotos(url):
    """Construye la URL directa al recorrido de fotos, sin extraer nada de la página inicial."""
    parsed = urlparse(str(url))
    query = parse_qs(parsed.query)
    query["modal"] = ["PHOTO_TOUR_SCROLLABLE"]
    nueva_query = urlencode(query, doseq=True)
    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, nueva_query, parsed.fragment))


def quitar_query_url(url):
    try:
        p = urlparse(url)
        return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))
    except Exception:
        return str(url).split("?")[0]


def es_url_foto_airbnb(url):
    if not url:
        return False
    u = str(url).lower()
    if not u.startswith("http"):
        return False

    # Fotos reales de Airbnb suelen venir de muscache.
    if "muscache.com" not in u:
        return False

    # Evitar assets, íconos y perfiles.
    excluir = [
        "favicon", "sprite", "icon", "logo", "avatar", "user_pic",
        "profile", "badge", "pdp-magic-carpet", "map", "api/v3"
    ]
    if any(x in u for x in excluir):
        return False

    # Preferir rutas de fotos.
    if "/im/pictures/" in u or "/pictures/" in u or ".jpg" in u or ".jpeg" in u or ".webp" in u or ".png" in u:
        return True

    return False


def elegir_mejores_variantes(items):
    """
    Deduplica por ruta base. Si hay varias versiones con im_w=..., conserva la de mayor ancho.
    items: lista de dicts {url, w, h, ...}
    """
    grupos = {}
    for item in items:
        url = item.get("url")
        if not es_url_foto_airbnb(url):
            continue
        base = quitar_query_url(url)
        grupos.setdefault(base, []).append(item)

    elegidos = []
    for base, variantes in grupos.items():
        mejor = variantes[0]
        mejor_score = -1
        for item in variantes:
            u = item.get("url", "")
            m = re.search(r"im_w=(\d+)", u)
            im_w = int(m.group(1)) if m else 0
            render_w = int(item.get("w") or 0)
            render_h = int(item.get("h") or 0)
            score = im_w * 10 + render_w + render_h
            if score > mejor_score:
                mejor = item
                mejor_score = score
        elegidos.append(mejor)

    return elegidos


def descargar_imagen(url, ruta_destino):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        img = Image.open(BytesIO(r.content)).convert("RGB")
        img.save(ruta_destino, quality=92)
        return True, ""
    except Exception as e:
        return False, str(e)


def escape_html(x):
    x = limpiar_texto(x)
    return (
        x.replace("&", "&amp;")
         .replace("<", "&lt;")
         .replace(">", "&gt;")
         .replace('"', "&quot;")
         .replace("'", "&#039;")
    )


def imagen_tag(ruta, width=180):
    try:
        p = Path(ruta)
        if not p.exists():
            return "<span style='color:red'>No existe</span>"
        mime = mimetypes.guess_type(str(p))[0] or "image/jpeg"
        with open(p, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        data = f"data:{mime};base64,{b64}"
        return f"<a href='{data}' target='_blank'><img src='{data}' style='width:{width}px;border-radius:6px;border:1px solid #ddd'></a>"
    except Exception as e:
        return f"<span style='color:red'>{escape_html(e)}</span>"

# ============================================================
# EXTRACCIÓN ESTRICTA DESDE EL RECORRIDO DE FOTOS
# ============================================================

def extraer_imagenes_visibles_grandes(page):
    """
    Extrae únicamente imágenes grandes actualmente visibles en la vista del recorrido.
    No se llama nunca desde la página inicial.
    """
    return page.evaluate(
        r"""
        ({minW, minH}) => {
            const out = [];

            function absUrl(u) {
                if (!u) return null;
                try { return new URL(String(u).trim(), location.href).href; }
                catch(e) { return null; }
            }

            function bestFromSrcset(srcset) {
                if (!srcset) return null;
                let best = null;
                let bestScore = -1;
                const parts = String(srcset).split(',');
                for (const part of parts) {
                    const bits = part.trim().split(/\s+/);
                    const u = bits[0];
                    let score = 0;
                    if (bits.length > 1) {
                        const m = bits[1].match(/(\d+)/);
                        if (m) score = parseInt(m[1]);
                    }
                    if (score > bestScore) {
                        best = u;
                        bestScore = score;
                    }
                }
                return best;
            }

            function addCandidate(url, rect, source, alt) {
                url = absUrl(url);
                if (!url) return;
                if (!rect) return;

                const w = rect.width || 0;
                const h = rect.height || 0;
                const x = rect.left || 0;
                const y = rect.top || 0;

                // Solo fotos grandes visibles en pantalla.
                if (w < minW || h < minH) return;
                if (rect.bottom < 0 || rect.top > window.innerHeight) return;

                out.push({
                    url,
                    x,
                    y,
                    w,
                    h,
                    source,
                    alt: alt || '',
                    page_url: location.href
                });
            }

            // IMG visibles grandes
            document.querySelectorAll('img').forEach(img => {
                const rect = img.getBoundingClientRect();
                const url = img.currentSrc || img.src || bestFromSrcset(img.srcset);
                addCandidate(url, rect, 'img', img.alt || '');
            });

            // SOURCE dentro de picture
            document.querySelectorAll('picture source').forEach(src => {
                const pic = src.closest('picture');
                const img = pic ? pic.querySelector('img') : null;
                if (!img) return;
                const rect = img.getBoundingClientRect();
                const url = bestFromSrcset(src.srcset);
                addCandidate(url, rect, 'source_srcset', img.alt || '');
            });

            // Background images visibles grandes
            document.querySelectorAll('*').forEach(el => {
                const rect = el.getBoundingClientRect();
                if ((rect.width || 0) < minW || (rect.height || 0) < minH) return;
                if (rect.bottom < 0 || rect.top > window.innerHeight) return;

                const bg = window.getComputedStyle(el).backgroundImage;
                if (!bg || bg === 'none') return;
                const matches = [...bg.matchAll(/url\(["']?(.*?)["']?\)/g)];
                for (const m of matches) {
                    addCandidate(m[1], rect, 'background', '');
                }
            });

            return out;
        }
        """,
        {"minW": MIN_RENDER_W, "minH": MIN_RENDER_H}
    )


def cerrar_popup_traduccion(page):
    """Cierra el aviso de traducción sin cerrar el recorrido fotográfico."""
    patron = re.compile(
        r"traducci[oó]n activada|traducci[oó]n automática|translation (?:is )?(?:on|enabled)",
        re.IGNORECASE,
    )

    try:
        texto = page.get_by_text(patron).first
        if not texto.is_visible(timeout=2500):
            return False

        dialogo = texto.locator(
            "xpath=ancestor::*[@role='dialog' or @aria-modal='true'][1]"
        )

        if dialogo.count() > 0:
            boton_cerrar = dialogo.locator(
                "button[aria-label*='errar' i], button[aria-label*='lose' i]"
            ).first
            if boton_cerrar.count() > 0:
                boton_cerrar.click(timeout=3000)
            else:
                botones = dialogo.locator("button")
                if botones.count() > 0:
                    botones.first.click(timeout=3000)
                else:
                    page.keyboard.press("Escape")
        else:
            page.keyboard.press("Escape")

        texto.wait_for(state="hidden", timeout=3000)
        page.wait_for_timeout(500)
        print("Aviso de traducción cerrado.")
        return True
    except Exception as e:
        print("Aviso: no se pudo cerrar automáticamente el popup de traducción:", e)
        return False


def mover_scroll_recorrido(page, ir_arriba=False):
    """Mueve el elemento desplazable real y devuelve su estado."""
    return page.evaluate(
        r"""
        ({irArriba}) => {
            const visibles = [...document.querySelectorAll('*')].filter(el => {
                const style = getComputedStyle(el);
                const rect = el.getBoundingClientRect();
                const permiteScroll = /(auto|scroll)/.test(style.overflowY);
                return permiteScroll &&
                       el.scrollHeight > el.clientHeight + 80 &&
                       rect.width > 300 && rect.height > 250 &&
                       rect.bottom > 0 && rect.top < window.innerHeight;
            });

            visibles.sort((a, b) =>
                (b.scrollHeight - b.clientHeight) -
                (a.scrollHeight - a.clientHeight)
            );

            const doc = document.scrollingElement || document.documentElement;
            const el = visibles[0] || doc;
            const esDocumento = el === doc || el === document.body || el === document.documentElement;
            const antes = esDocumento ? window.scrollY : el.scrollTop;
            const viewport = esDocumento ? window.innerHeight : el.clientHeight;

            if (irArriba) {
                if (esDocumento) window.scrollTo(0, 0);
                else el.scrollTop = 0;
            } else {
                const paso = Math.max(300, Math.floor(viewport * 0.82));
                if (esDocumento) window.scrollBy(0, paso);
                else el.scrollBy({top: paso, behavior: 'auto'});
            }

            const despues = esDocumento ? window.scrollY : el.scrollTop;
            const total = esDocumento
                ? Math.max(document.body.scrollHeight, document.documentElement.scrollHeight)
                : el.scrollHeight;

            return {
                antes,
                despues,
                viewport,
                total,
                avanzo: irArriba || despues > antes + 1,
                al_final: despues + viewport >= total - 80,
                tipo: esDocumento ? 'documento' : 'contenedor'
            };
        }
        """,
        {"irArriba": ir_arriba},
    )


def llegar_arriba(page):
    try:
        mover_scroll_recorrido(page, ir_arriba=True)
        page.wait_for_timeout(1000)
    except Exception:
        pass


def scroll_y_capturar(page, id_airbnb):
    """
    Hace scroll por la URL modal=PHOTO_TOUR_SCROLLABLE y captura fotos visibles grandes.
    """
    todas = []
    vistas = set()
    sin_nuevas = 0
    sin_avanzar = 0

    llegar_arriba(page)

    debug_path = DEBUG_DIR / f"{id_airbnb}_recorrido_inicio.png"
    try:
        page.screenshot(path=str(debug_path), full_page=False)
    except Exception:
        pass

    for i in range(SCROLLS_GALERIA):
        try:
            items = extraer_imagenes_visibles_grandes(page)
        except Exception as e:
            print("Error extrayendo visibles:", e)
            items = []

        nuevos = 0
        for item in items:
            url = item.get("url")
            if not es_url_foto_airbnb(url):
                continue
            base = quitar_query_url(url)
            if base not in vistas:
                vistas.add(base)
                todas.append(item)
                nuevos += 1

        sin_nuevas = sin_nuevas + 1 if nuevos == 0 else 0

        if i % 10 == 0:
            print(f"  Scroll {i+1}/{SCROLLS_GALERIA} | fotos únicas visibles acumuladas: {len(vistas)} | nuevas: {nuevos}")

        try:
            info = mover_scroll_recorrido(page)
            sin_avanzar = 0 if info["avanzo"] else sin_avanzar + 1
            page.wait_for_timeout(int(ESPERA_SCROLL_SEGUNDOS * 1000))

            # Airbnb carga fotos de forma diferida. Solo termina después de
            # varias comprobaciones sin movimiento y sin imágenes nuevas.
            if info["al_final"] and sin_nuevas >= 3:
                print("  Fin real del recorrido confirmado.")
                break
            if sin_avanzar >= 5 and sin_nuevas >= 5:
                print("  El recorrido ya no avanza ni carga fotos nuevas.")
                break
        except Exception as e:
            print("Error desplazando el recorrido:", e)
            page.wait_for_timeout(int(ESPERA_SCROLL_SEGUNDOS * 1000))

    return elegir_mejores_variantes(todas)


def procesar_un_airbnb(playwright, row, idx):
    id_airbnb = normalizar_id_airbnb(row.get(ID_COL), row.get(URL_COL))
    url_original = str(row.get(URL_COL)).strip()
    url_recorrido = construir_url_recorrido_fotos(url_original)

    print("\n" + "=" * 70)
    print(f"Fila Excel: {idx}")
    print("ID Airbnb:", id_airbnb)
    print("URL original:", url_original)
    print("URL recorrido:", url_recorrido)

    carpeta = IMG_DIR / id_airbnb
    carpeta.mkdir(parents=True, exist_ok=True)

    browser = playwright.chromium.launch(
        headless=HEADLESS,
        args=["--disable-blink-features=AutomationControlled"]
    )
    context = browser.new_context(
        viewport={"width": 1400, "height": 950},
        locale="es-PE",
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    )
    page = context.new_page()

    registros = []

    try:
        # Punto clave: NO entramos a la página original para extraer imágenes.
        # Entramos directamente al recorrido de fotos.
        page.goto(url_recorrido, wait_until="domcontentloaded", timeout=90000)
        page.wait_for_timeout(ESPERA_INICIAL_SEGUNDOS * 1000)

        print("Página cargada:", page.url)

        # Si Airbnb redirige a otra URL, guardamos evidencia.
        if "PHOTO_TOUR_SCROLLABLE" not in page.url:
            print("Aviso: la URL final no contiene PHOTO_TOUR_SCROLLABLE. Puede que Airbnb haya redirigido.")

        cerrar_popup_traduccion(page)
        items = scroll_y_capturar(page, id_airbnb)
        print("Total fotos detectadas en recorrido:", len(items))

        for j, item in enumerate(items, start=1):
            img_url = item.get("url")
            hash_img = hashlib.md5(img_url.encode("utf-8")).hexdigest()[:10]
            nombre = f"{id_airbnb}_foto_{j:03d}_{hash_img}.jpg"
            ruta_local = carpeta / nombre
            ok, error = descargar_imagen(img_url, ruta_local)

            registros.append({
                "id_airbnb": id_airbnb,
                "fila_excel": idx,
                "url_original": url_original,
                "url_recorrido": url_recorrido,
                "page_url_final": page.url,
                "nro_foto": j,
                "image_url": img_url,
                "ruta_local": str(ruta_local),
                "image_path": str(ruta_local),
                "descarga_ok": ok,
                "error_descarga": error,
                "render_x": item.get("x"),
                "render_y": item.get("y"),
                "render_w": item.get("w"),
                "render_h": item.get("h"),
                "source": item.get("source"),
                "alt": item.get("alt"),
                "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "metodo_extraccion": "URL directa con modal=PHOTO_TOUR_SCROLLABLE; captura de imágenes visibles grandes durante scroll; no se extraen imágenes de la página inicial"
            })

        if not registros:
            registros.append({
                "id_airbnb": id_airbnb,
                "fila_excel": idx,
                "url_original": url_original,
                "url_recorrido": url_recorrido,
                "page_url_final": page.url,
                "nro_foto": None,
                "image_url": None,
                "ruta_local": None,
                "image_path": None,
                "descarga_ok": False,
                "error_descarga": "No se detectaron fotos en el recorrido",
                "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "metodo_extraccion": "Intento en modal=PHOTO_TOUR_SCROLLABLE sin fotos detectadas"
            })

    except Exception as e:
        print("Error procesando:", e)
        registros.append({
            "id_airbnb": id_airbnb,
            "fila_excel": idx,
            "url_original": url_original,
            "url_recorrido": url_recorrido,
            "page_url_final": None,
            "nro_foto": None,
            "image_url": None,
            "ruta_local": None,
            "image_path": None,
            "descarga_ok": False,
            "error_descarga": str(e),
            "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "metodo_extraccion": "Error al abrir/capturar modal=PHOTO_TOUR_SCROLLABLE"
        })

    finally:
        context.close()
        browser.close()

    return registros

# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    excel = encontrar_excel()
    df = pd.read_excel(excel, sheet_name=SHEET_NAME, header=HEADER_ROW)

    print("Columnas detectadas:", list(df.columns))
    if URL_COL not in df.columns:
        raise ValueError(f"No encuentro la columna {URL_COL}")
    if ID_COL not in df.columns:
        raise ValueError(f"No encuentro la columna {ID_COL}")

    base = df.dropna(subset=[URL_COL]).copy()
    base = base[base[URL_COL].astype(str).str.contains("airbnb", case=False, na=False)].copy()

    print("Links válidos detectados:", len(base))

    if MAX_ALOJAMIENTOS is not None:
        base = base.head(MAX_ALOJAMIENTOS)
        print("Modo prueba. Se procesarán:", len(base))
    else:
        print("Modo completo. Se procesarán todos:", len(base))

    resultados = []

    with sync_playwright() as p:
        for idx, row in base.iterrows():
            regs = procesar_un_airbnb(p, row, idx)
            resultados.extend(regs)

    metadata = pd.DataFrame(resultados)
    metadata.to_excel(METADATA_XLSX, index=False)

    # Manifest para ML: solo fotos descargadas correctamente
    manifest = metadata[metadata.get("descarga_ok") == True].copy()
    if not manifest.empty:
        manifest["label_ambiente"] = ""
        manifest["label_calidad"] = ""
        manifest["split"] = ""
        manifest["observacion"] = ""

        columnas = [
            "id_airbnb", "url_original", "url_recorrido", "nro_foto",
            "image_url", "ruta_local", "image_path", "label_ambiente",
            "label_calidad", "split", "observacion", "fecha_extraccion", "metodo_extraccion"
        ]
        columnas = [c for c in columnas if c in manifest.columns]
        manifest = manifest[columnas]

    manifest.to_csv(MANIFEST_CSV, index=False, encoding="utf-8-sig")
    manifest.to_excel(MANIFEST_XLSX, index=False)

    # Reporte HTML
    rows = []
    for _, r in metadata.iterrows():
        img = imagen_tag(r.get("ruta_local")) if r.get("descarga_ok") else ""
        rows.append(f"""
        <tr>
            <td>{escape_html(r.get('id_airbnb'))}</td>
            <td>{escape_html(r.get('nro_foto'))}</td>
            <td>{img}</td>
            <td><a href='{escape_html(r.get('url_recorrido'))}' target='_blank'>Abrir recorrido</a></td>
            <td style='font-size:11px'>{escape_html(r.get('ruta_local'))}</td>
            <td style='font-size:11px'>{escape_html(r.get('error_descarga'))}</td>
        </tr>
        """)

    html = f"""
    <!doctype html>
    <html><head><meta charset='utf-8'><title>Reporte fotos Airbnb</title>
    <style>
    body {{ font-family: Arial, sans-serif; margin: 20px; }}
    table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
    th {{ background:#222; color:white; padding:8px; position: sticky; top: 0; }}
    td {{ border:1px solid #ddd; padding:8px; vertical-align: middle; }}
    tr:nth-child(even) {{ background:#f5f5f5; }}
    </style></head><body>
    <h1>Reporte de fotos Airbnb</h1>
    <p>Extracción desde <b>modal=PHOTO_TOUR_SCROLLABLE</b>. No se extraen imágenes de la página inicial.</p>
    <table>
    <thead><tr><th>ID Airbnb</th><th>N° foto</th><th>Imagen</th><th>URL recorrido</th><th>Ruta local</th><th>Error</th></tr></thead>
    <tbody>{''.join(rows)}</tbody>
    </table>
    </body></html>
    """

    with open(REPORTE_HTML, "w", encoding="utf-8") as f:
        f.write(html)

    # ZIP con todo
    if Path(ZIP_PATH).exists():
        Path(ZIP_PATH).unlink()
    shutil.make_archive(ZIP_NAME, "zip", ".", base_dir=str(OUT_DIR))

    print("\nProceso terminado.")
    print("Metadata:", METADATA_XLSX)
    print("Manifest CSV:", MANIFEST_CSV)
    print("Manifest Excel:", MANIFEST_XLSX)
    print("Reporte HTML:", REPORTE_HTML)
    print("ZIP:", ZIP_PATH)
    print("Fotos descargadas correctamente:", len(manifest))

if __name__ == "__main__":
    main()
'''

Path("airbnb_descargar_solo_recorrido.py").write_text(script, encoding="utf-8")
print("Script creado: airbnb_descargar_solo_recorrido.py")

Script creado: airbnb_descargar_solo_recorrido.py


In [ ]:
# ============================================================
# CELDA 4 - Ejecutar descarga
# ============================================================
# Importante: deja G4_mod.xlsx en la misma carpeta del notebook.
# Si se abre un navegador, déjalo trabajar hasta que termine.

import sys
import subprocess

subprocess.check_call([sys.executable, "airbnb_descargar_solo_recorrido.py"])

0

In [10]:
# ============================================================
# CELDA 5 - Revisar resultados
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

metadata_path = Path("salida_airbnb_fotos/metadata_descarga_fotos.xlsx")
manifest_path = Path("salida_airbnb_fotos/dataset_manifest_airbnb.csv")
reporte_path = Path("salida_airbnb_fotos/reporte_visual_fotos.html")

if metadata_path.exists():
    metadata = pd.read_excel(metadata_path)
    print("Metadata:", metadata.shape)
    display(metadata.head())
    if "id_airbnb" in metadata.columns and "descarga_ok" in metadata.columns:
        display(metadata.groupby(["id_airbnb", "descarga_ok"]).size().reset_index(name="n"))
else:
    print("No existe metadata todavía.")

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print("Manifest para ML:", manifest.shape)
    display(manifest.head())
else:
    print("No existe manifest todavía.")

if reporte_path.exists():
    print("Reporte HTML:", reporte_path)
    display(HTML(f"<a href='{reporte_path.as_posix()}' target='_blank'>Abrir reporte visual</a>"))

Metadata: (2282, 19)


,id_airbnb,fila_excel,url_original,url_recorrido,page_url_final,nro_foto,image_url,ruta_local,image_path,descarga_ok,error_descarga,render_x,render_y,render_w,render_h,source,alt,fecha_extraccion,metodo_extraccion
0,708973846265169428,0,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,1,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,True,NaN,132.5,158.0,560.0,506.0,img,NaN,2026-06-21 19:45:42,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...
1,708973846265169428,0,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,2,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,True,NaN,700.5,158.0,272.0,253.0,img,NaN,2026-06-21 19:45:42,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...
2,708973846265169428,0,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,3,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,True,NaN,700.5,419.0,272.0,245.0,img,NaN,2026-06-21 19:45:42,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...
3,708973846265169428,0,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,4,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,True,NaN,980.5,158.0,272.0,253.0,img,NaN,2026-06-21 19:45:42,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...
4,708973846265169428,0,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,5,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,True,NaN,980.5,419.0,272.0,245.0,img,NaN,2026-06-21 19:45:43,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...


,id_airbnb,descarga_ok,n
0,20738256,True,20
1,51265033,True,34
2,51764629,True,54
3,51813337,True,28
4,564368895233936119,True,20
5,708973846265169428,True,30
6,742541719033641514,True,49
7,800208014785752524,True,74
8,827026659851983853,True,25
9,838633176578704267,True,33


Manifest para ML: (2282, 13)


,id_airbnb,url_original,url_recorrido,nro_foto,image_url,ruta_local,image_path,label_ambiente,label_calidad,split,observacion,fecha_extraccion,metodo_extraccion
0,708973846265169428,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,1,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,NaN,NaN,NaN,NaN,2026-06-21 19:45:42,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...
1,708973846265169428,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,2,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,NaN,NaN,NaN,NaN,2026-06-21 19:45:42,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...
2,708973846265169428,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,3,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,NaN,NaN,NaN,NaN,2026-06-21 19:45:42,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...
3,708973846265169428,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,4,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,NaN,NaN,NaN,NaN,2026-06-21 19:45:42,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...
4,708973846265169428,https://www.airbnb.com.pe/rooms/70897384626516...,https://www.airbnb.com.pe/rooms/70897384626516...,5,https://a0.muscache.com/im/pictures/hosting/Ho...,img\708973846265169428\708973846265169428_foto...,img\708973846265169428\708973846265169428_foto...,NaN,NaN,NaN,NaN,2026-06-21 19:45:43,URL directa con modal=PHOTO_TOUR_SCROLLABLE; c...


Reporte HTML: salida_airbnb_fotos\reporte_visual_fotos.html


## Etiquetado de calidad fotográfica

En esta parte se etiquetan **todas las fotos necesarias para el CNN**: entrenamiento, prueba y validación.  
La validación también se etiqueta manualmente porque luego servirá como referencia para comprobar si el mejor modelo acertó o no.

La idea es que el flujo sea secuencial:

1. Se preparan las fotos y sus grupos.
2. Se etiquetan las fotos una sola vez.
3. Se consolida el dataset.
4. Se entrena, compara y valida el CNN.

No tendrás que regresar después a una celda especial para etiquetar validación.

In [26]:
# ============================================================
# CELDA 6 - Preparar manifest para etiquetado automatico y separar splits
# ============================================================
# Nombres usados en esta plantilla:
# - train      : se usa para entrenar la CNN
# - test       : se usa para probar/comparar modelos durante el desarrollo
# - validacion : se reserva para la validacion final del modelo elegido
#
# IMPORTANTE:
# El split se hace por ID Airbnb completo, no por foto, para evitar que
# imagenes del mismo alojamiento caigan en grupos diferentes.

from pathlib import Path
import pandas as pd
import random
from IPython.display import display

# ------------------------------------------------------------
# CONFIGURACION SIMPLE
# ------------------------------------------------------------
MANIFEST_ORIGINAL = Path("salida_airbnb_fotos/dataset_manifest_airbnb.csv")

# Archivos nuevos que se crearan
MANIFEST_SPLIT_CSV = Path("salida_airbnb_fotos/dataset_manifest_con_split_calidad.csv")
MANIFEST_SPLIT_XLSX = Path("salida_airbnb_fotos/dataset_manifest_con_split_calidad.xlsx")

# Cantidad de Airbnb completos para validacion final.
# Este grupo se etiqueta automaticamente al final y NO se usa para entrenar ni elegir modelos.
N_VALIDACION_IDS = 10

# Cantidad de Airbnb completos para test.
# Este grupo sirve para probar/comparar modelos durante el desarrollo.
N_TEST_IDS = 7

RANDOM_STATE = 33

# ------------------------------------------------------------
# CARGA DEL MANIFEST
# ------------------------------------------------------------
if not MANIFEST_ORIGINAL.exists():
    raise FileNotFoundError(
        "No encontre el manifest original. Primero ejecuta la descarga de fotos y revisa que exista: "
        f"{MANIFEST_ORIGINAL}"
    )

df = pd.read_csv(MANIFEST_ORIGINAL)

# Normalizar columna de ruta de imagen
if "image_path" not in df.columns:
    if "ruta_local" in df.columns:
        df["image_path"] = df["ruta_local"]
    else:
        raise ValueError("No encuentro columna image_path ni ruta_local en el manifest.")

# Mantener solo imagenes que existan en disco
df["image_path"] = df["image_path"].astype(str)
df["existe_imagen"] = df["image_path"].apply(lambda x: Path(x).exists())
df = df[df["existe_imagen"]].copy()

if df.empty:
    raise ValueError("El manifest no tiene imagenes existentes. Revisa la descarga.")

# ID unico por foto para guardar etiquetas sin confundirse
df["id_airbnb"] = df["id_airbnb"].astype(str)
df["registro_id"] = df["id_airbnb"].astype(str) + "||" + df["image_path"].astype(str)

# Crear split por ID Airbnb, no por foto.
ids = sorted(df["id_airbnb"].dropna().astype(str).unique().tolist())
random.seed(RANDOM_STATE)
random.shuffle(ids)

n_validacion = min(N_VALIDACION_IDS, len(ids))
ids_validacion = set(ids[:n_validacion])

restantes = ids[n_validacion:]
n_test = min(N_TEST_IDS, len(restantes))
ids_test = set(restantes[:n_test])

ids_train = set(restantes[n_test:])

def asignar_split(id_airbnb):
    x = str(id_airbnb)
    if x in ids_validacion:
        return "validacion"
    elif x in ids_test:
        return "test"
    else:
        return "train"

df["split_calidad"] = df["id_airbnb"].apply(asignar_split)

# Columnas que llenara la CELDA 7 con el scoring automatico equivalente a la app.
for col in [
    "iluminacion", "encuadre", "nitidez", "atractivo_visual", "utilidad_huesped",
    "puntaje_total", "calidad_final", "estado_etiquetado", "observacion",
    "score_iluminacion", "score_contraste", "score_nitidez", "score_color_natural", "score_cobertura_util"
]:
    if col not in df.columns:
        df[col] = ""

df.to_csv(MANIFEST_SPLIT_CSV, index=False, encoding="utf-8-sig")
df.to_excel(MANIFEST_SPLIT_XLSX, index=False)

print("Manifest preparado para etiquetado automatico:")
print(MANIFEST_SPLIT_CSV)
print(MANIFEST_SPLIT_XLSX)
print()

print("Resumen por split:")
resumen = df.groupby("split_calidad").agg(
    airbnb=("id_airbnb", "nunique"),
    fotos=("image_path", "count")
).reset_index()

orden = {"train": 1, "test": 2, "validacion": 3}
resumen["orden"] = resumen["split_calidad"].map(orden)
resumen = resumen.sort_values("orden").drop(columns="orden")
display(resumen)

print("IDs usados para VALIDACION FINAL:")
display(pd.DataFrame({"id_airbnb_validacion": sorted(ids_validacion)}))

print("IDs usados para TEST:")
display(pd.DataFrame({"id_airbnb_test": sorted(ids_test)}))

print("""
Nota:
- train se usa para entrenar.
- test se usa para probar/comparar modelos.
- validacion se deja para la evaluacion final del modelo elegido.
- La CELDA 7 ya no abre una ventanita manual: calcula etiquetas automaticamente desde los pixeles.
- Si luego quieres cambiar cantidades, modifica N_VALIDACION_IDS y N_TEST_IDS y vuelve a ejecutar esta celda.
""")


Manifest preparado para etiquetado automatico:
salida_airbnb_fotos\dataset_manifest_con_split_calidad.csv
salida_airbnb_fotos\dataset_manifest_con_split_calidad.xlsx

Resumen por split:


,split_calidad,airbnb,fotos
1,train,35,3032
0,test,7,502
2,validacion,10,946


IDs usados para VALIDACION FINAL:


,id_airbnb_validacion
0,1086343485196324895
1,1118142286350817818
2,1158688913933082401
3,1202250552806361613
4,1412366769894758932
5,1422671086540527802
6,1648119432301960151
7,742541719033641514
8,827026659851983853
9,976060981254094446


IDs usados para TEST:


,id_airbnb_test
0,1142890900267610713
1,1453092459905498863
2,1480403540824670560
3,1492336827661910996
4,51265033
5,51764629
6,871231364866066627



Nota:
- train se usa para entrenar.
- test se usa para probar/comparar modelos.
- validacion se deja para la evaluacion final del modelo elegido.
- La CELDA 7 ya no abre una ventanita manual: calcula etiquetas automaticamente desde los pixeles.
- Si luego quieres cambiar cantidades, modifica N_VALIDACION_IDS y N_TEST_IDS y vuelve a ejecutar esta celda.



In [27]:
# ============================================================
# CELDA 7 - Etiquetado automatico de calidad fotografica
# Misma logica base usada en la aplicacion Angular:
# - iluminacion por luminancia media
# - contraste por desviacion de luminancia
# - nitidez por energia de bordes
# - color natural por saturacion moderada
# - cobertura util por pixeles sin extremos negros/quemados
# - calidad_final se clasifica por mediana del puntaje_total:
#   alta = puntaje_total >= mediana; media = puntaje_total < mediana
# ============================================================

from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from IPython.display import display

# ------------------------------------------------------------
# CONFIGURACION
# ------------------------------------------------------------
MANIFEST_SPLIT_CSV = Path("salida_airbnb_fotos/dataset_manifest_con_split_calidad.csv")
ETIQUETAS_CSV = Path("salida_airbnb_fotos/etiquetas_calidad_fotos.csv")
ETIQUETAS_XLSX = Path("salida_airbnb_fotos/etiquetas_calidad_fotos.xlsx")

# Si es True, recalcula todo desde cero para que el etiquetado sea reproducible.
# Si quieres conservar etiquetas manuales antiguas, cambia a False.
SOBREESCRIBIR_ETIQUETAS_EXISTENTES = True

PESOS_SCORE_VISUAL = {
    "iluminacion": 0.24,
    "contraste": 0.22,
    "nitidez": 0.22,
    "color_natural": 0.14,
    "cobertura_util": 0.18,
}

COLUMNAS_ETIQUETAS = [
    "registro_id", "id_airbnb", "split_calidad", "nro_foto", "image_path",
    "iluminacion", "encuadre", "nitidez", "atractivo_visual", "utilidad_huesped",
    "puntaje_total", "calidad_final", "estado_etiquetado", "observacion", "fecha_etiquetado",
    "score_iluminacion", "score_contraste", "score_nitidez", "score_color_natural", "score_cobertura_util"
]


def clamp(value, min_value=0.0, max_value=1.0):
    return max(min_value, min(max_value, float(value)))


def round_one(value):
    return round(float(value) + 1e-12, 1)


def factor(value):
    return round_one(clamp(value) * 100)


def clasificar_score_visual(score, mediana_referencia=None):
    if mediana_referencia is None or pd.isna(mediana_referencia):
        return "media"
    return "alta" if score >= mediana_referencia else "media"


def normalizar_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    return x[:-2] if x.endswith(".0") else x


def resolver_ruta_local(x):
    x_str = str(x).strip()
    candidatos = [
        Path(x_str),
        Path(x_str.replace("\\", "/")),
        Path("salida_airbnb_fotos") / x_str,
        Path("salida_airbnb_fotos") / x_str.replace("\\", "/"),
    ]
    for candidato in candidatos:
        if candidato.exists():
            return candidato
    return Path(x_str)


def analizar_pixeles_imagen(path_imagen, max_width=260):
    """Port Python de analyzeImagePixels + scoreImageMetrics de src/app/scoring.ts."""
    imagen = PILImage.open(path_imagen).convert("RGB")

    width_original, height_original = imagen.size
    if width_original <= 0 or height_original <= 0:
        raise ValueError("imagen sin dimensiones validas")

    ratio = height_original / max(width_original, 1)
    width = max_width
    height = max(1, int(round(max_width * ratio)))
    imagen = imagen.resize((width, height), PILImage.Resampling.LANCZOS)

    arr = np.asarray(imagen, dtype=np.float32) / 255.0
    sampled = arr[::2, ::2, :]

    r = sampled[:, :, 0]
    g = sampled[:, :, 1]
    b = sampled[:, :, 2]

    max_rgb = np.maximum(np.maximum(r, g), b)
    min_rgb = np.minimum(np.minimum(r, g), b)
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    saturation = np.where(max_rgb == 0, 0, (max_rgb - min_rgb) / max_rgb)
    balanced = (luma > 0.08) & (luma < 0.94)

    # Equivalente al barrido x + 2 de la app, aplicado sobre la imagen muestreada.
    if luma.shape[1] > 1:
        edge_energy = float(np.abs(luma[:, 1:] - luma[:, :-1]).sum() / luma.size)
    else:
        edge_energy = 0.0

    mean_luma = float(luma.mean())
    contrast = float(luma.std())
    mean_saturation = float(saturation.mean())
    usable_coverage = float(balanced.mean())

    metricas = {
        "score_iluminacion": factor(1 - abs(mean_luma - 0.58) / 0.45),
        "score_contraste": factor(contrast / 0.24),
        "score_nitidez": factor(edge_energy / 0.08),
        "score_color_natural": factor(1 - abs(mean_saturation - 0.32) / 0.42),
        "score_cobertura_util": factor(usable_coverage),
    }

    score_visual = round_one(
        metricas["score_iluminacion"] * PESOS_SCORE_VISUAL["iluminacion"]
        + metricas["score_contraste"] * PESOS_SCORE_VISUAL["contraste"]
        + metricas["score_nitidez"] * PESOS_SCORE_VISUAL["nitidez"]
        + metricas["score_color_natural"] * PESOS_SCORE_VISUAL["color_natural"]
        + metricas["score_cobertura_util"] * PESOS_SCORE_VISUAL["cobertura_util"]
    )

    detalles = (
        f"auto app: luma={mean_luma:.2f}; contraste={contrast:.2f}; "
        f"bordes={edge_energy:.2f}; saturacion={mean_saturation:.2f}; "
        f"cobertura={usable_coverage:.2f}"
    )

    return metricas, score_visual, detalles


def etiquetar_foto(row):
    path_imagen = resolver_ruta_local(row["image_path"])

    base = {
        "registro_id": row.get("registro_id", f"{row['id_airbnb']}||{row['image_path']}"),
        "id_airbnb": normalizar_id(row["id_airbnb"]),
        "split_calidad": row["split_calidad"],
        "nro_foto": row.get("nro_foto", None),
        "image_path": str(row["image_path"]),
        "fecha_etiquetado": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    try:
        metricas, score_visual, detalles = analizar_pixeles_imagen(path_imagen)
        calidad = clasificar_score_visual(score_visual)

        # Columnas historicas del notebook: 1 si cumple, 0 si no cumple.
        base.update({
            "iluminacion": int(metricas["score_iluminacion"] >= 50),
            "encuadre": int(metricas["score_cobertura_util"] >= 50),
            "nitidez": int(metricas["score_nitidez"] >= 50),
            "atractivo_visual": int(score_visual >= 75),
            "utilidad_huesped": int(metricas["score_cobertura_util"] >= 50 and metricas["score_iluminacion"] >= 50),
            "puntaje_total": score_visual,
            "calidad_final": calidad,
            "estado_etiquetado": "etiquetado",
            "observacion": detalles,
            **metricas,
        })
    except Exception as exc:
        base.update({
            "iluminacion": 0,
            "encuadre": 0,
            "nitidez": 0,
            "atractivo_visual": 0,
            "utilidad_huesped": 0,
            "puntaje_total": 0.0,
            "calidad_final": "no_evaluable",
            "estado_etiquetado": "no_evaluable",
            "observacion": f"error automatico: {exc}",
            "score_iluminacion": 0.0,
            "score_contraste": 0.0,
            "score_nitidez": 0.0,
            "score_color_natural": 0.0,
            "score_cobertura_util": 0.0,
        })

    return base


if not MANIFEST_SPLIT_CSV.exists():
    raise FileNotFoundError("No encontre el manifest con split. Ejecuta primero la CELDA 6.")

manifest = pd.read_csv(MANIFEST_SPLIT_CSV)
manifest["id_airbnb"] = manifest["id_airbnb"].apply(normalizar_id)
manifest["image_path"] = manifest["image_path"].astype(str)

if "registro_id" not in manifest.columns:
    manifest["registro_id"] = manifest["id_airbnb"].astype(str) + "||" + manifest["image_path"].astype(str)

manifest["ruta_resuelta"] = manifest["image_path"].apply(resolver_ruta_local)
manifest = manifest[manifest["ruta_resuelta"].apply(lambda p: Path(p).exists())].copy()

if manifest.empty:
    raise ValueError("No hay imagenes existentes para etiquetar automaticamente.")

if ETIQUETAS_CSV.exists() and not SOBREESCRIBIR_ETIQUETAS_EXISTENTES:
    etiquetas_previas = pd.read_csv(ETIQUETAS_CSV)
    if "registro_id" not in etiquetas_previas.columns and not etiquetas_previas.empty:
        etiquetas_previas["registro_id"] = etiquetas_previas["id_airbnb"].astype(str) + "||" + etiquetas_previas["image_path"].astype(str)
    ya_hechas = set(etiquetas_previas["registro_id"].astype(str))
    pendientes = manifest[~manifest["registro_id"].astype(str).isin(ya_hechas)].copy()
else:
    etiquetas_previas = pd.DataFrame(columns=COLUMNAS_ETIQUETAS)
    pendientes = manifest.copy()

print("Fotos en manifest:", len(manifest))
print("Fotos a etiquetar automaticamente:", len(pendientes))
print("Archivo de salida:", ETIQUETAS_CSV)

nuevas = [etiquetar_foto(row) for _, row in pendientes.iterrows()]
etiquetas_nuevas = pd.DataFrame(nuevas, columns=COLUMNAS_ETIQUETAS)

if ETIQUETAS_CSV.exists() and not SOBREESCRIBIR_ETIQUETAS_EXISTENTES:
    etiquetas = pd.concat([etiquetas_previas, etiquetas_nuevas], ignore_index=True)
    etiquetas = etiquetas.drop_duplicates(subset=["registro_id"], keep="last")
else:
    etiquetas = etiquetas_nuevas

for col in COLUMNAS_ETIQUETAS:
    if col not in etiquetas.columns:
        etiquetas[col] = None
etiquetas = etiquetas[COLUMNAS_ETIQUETAS].copy()

# Reclasificacion final por mediana del puntaje_total.
# Las imagenes no evaluables se conservan como no_evaluable.
mask_evaluable = etiquetas["estado_etiquetado"].astype(str).str.lower().eq("etiquetado")
puntajes_evaluables = pd.to_numeric(etiquetas.loc[mask_evaluable, "puntaje_total"], errors="coerce")
mediana_puntaje_total = float(puntajes_evaluables.median()) if puntajes_evaluables.notna().any() else np.nan

if pd.notna(mediana_puntaje_total):
    etiquetas.loc[mask_evaluable, "calidad_final"] = np.where(
        pd.to_numeric(etiquetas.loc[mask_evaluable, "puntaje_total"], errors="coerce") >= mediana_puntaje_total,
        "alta",
        "media"
    )

ETIQUETAS_CSV.parent.mkdir(parents=True, exist_ok=True)
etiquetas.to_csv(ETIQUETAS_CSV, index=False, encoding="utf-8-sig")
etiquetas.to_excel(ETIQUETAS_XLSX, index=False)

print("Etiquetado automatico terminado.")
print(ETIQUETAS_CSV)
print(ETIQUETAS_XLSX)
print(f"Mediana puntaje_total usada para calidad_final: {mediana_puntaje_total:.2f}" if pd.notna(mediana_puntaje_total) else "Mediana puntaje_total no disponible")

print("\nDistribucion por calidad_final:")
display(etiquetas["calidad_final"].value_counts(dropna=False).reset_index(name="n_fotos").rename(columns={"index": "calidad_final"}))

print("\nResumen por split y calidad:")
display(etiquetas.groupby(["split_calidad", "calidad_final"], dropna=False).size().reset_index(name="n_fotos"))

print("\nMetricas automaticas promedio:")
metric_cols = ["puntaje_total", "score_iluminacion", "score_contraste", "score_nitidez", "score_color_natural", "score_cobertura_util"]
display(etiquetas[metric_cols].describe().round(2))


Fotos en manifest: 4480
Fotos a etiquetar automaticamente: 4480
Archivo de salida: salida_airbnb_fotos\etiquetas_calidad_fotos.csv


C:\Users\andre\AppData\Local\Temp\ipykernel_37524\1723014138.py:110: RuntimeWarning: invalid value encountered in divide
  saturation = np.where(max_rgb == 0, 0, (max_rgb - min_rgb) / max_rgb)


Etiquetado automatico terminado.
salida_airbnb_fotos\etiquetas_calidad_fotos.csv
salida_airbnb_fotos\etiquetas_calidad_fotos.xlsx
Mediana puntaje_total usada para calidad_final: 79.70

Distribucion por calidad_final:


,calidad_final,n_fotos
0,alta,2256
1,media,2224



Resumen por split y calidad:


,split_calidad,calidad_final,n_fotos
0,test,alta,252
1,test,media,250
2,train,alta,1536
3,train,media,1496
4,validacion,alta,468
5,validacion,media,478



Metricas automaticas promedio:


,puntaje_total,score_iluminacion,score_contraste,score_nitidez,score_color_natural,score_cobertura_util
count,4480.00,4480.00,4480.00,4480.00,4480.00,4480.00
mean,79.13,82.17,89.29,59.03,71.37,93.24
std,8.08,14.90,12.16,22.19,17.37,7.87
min,31.70,2.70,28.10,11.20,0.00,20.80
25%,74.70,75.80,81.90,41.80,58.30,91.90
50%,79.70,85.50,93.30,56.00,72.70,95.60
75%,84.40,93.30,100.00,74.40,85.40,97.70
max,98.80,100.00,100.00,100.00,100.00,100.00


In [ ]:
## para re etiquetar

##from pathlib import Path

##SALIDA_DIR = Path("salida_airbnb_fotos")
##archivo_etiquetas = SALIDA_DIR / "etiquetas_calidad_fotos.csv"

##if archivo_etiquetas.exists():
##    archivo_etiquetas.unlink()
##    print("Archivo de etiquetas eliminado. Puedes re-etiquetar desde cero.")
##else:
##    print("No había archivo de etiquetas.")

In [28]:

# ============================================================
# CELDA 8 - Consolidar etiquetas y generar dataset para CNN
# ============================================================
# Esta celda toma:
# - salida_airbnb_fotos/dataset_manifest_con_split_calidad.csv
# - salida_airbnb_fotos/etiquetas_calidad_fotos.csv
#
# Para que el flujo sea secuencial, antes de ejecutar esta celda ya deben estar
# etiquetadas las fotos de train, test y validacion en la CELDA 7.
#
# Y genera:
# - dataset completo etiquetado
# - dataset CNN solo con alta/media, consistente con la mediana de la CELDA 7
# - archivos separados train/test/validacion
# - score visual por Airbnb

import pandas as pd
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. Rutas de trabajo
# ------------------------------------------------------------
SALIDA_DIR = Path("salida_airbnb_fotos")

MANIFEST_SPLIT_CSV = SALIDA_DIR / "dataset_manifest_con_split_calidad.csv"
ETIQUETAS_CSV = SALIDA_DIR / "etiquetas_calidad_fotos.csv"

DATASET_COMPLETO_CSV = SALIDA_DIR / "dataset_calidad_etiquetado_completo.csv"
DATASET_COMPLETO_XLSX = SALIDA_DIR / "dataset_calidad_etiquetado_completo.xlsx"

DATASET_CNN_CSV = SALIDA_DIR / "dataset_cnn_calidad_etiquetado.csv"
DATASET_CNN_XLSX = SALIDA_DIR / "dataset_cnn_calidad_etiquetado.xlsx"

DATASET_CNN_TRAIN_CSV = SALIDA_DIR / "dataset_cnn_train.csv"
DATASET_CNN_TEST_CSV = SALIDA_DIR / "dataset_cnn_test.csv"
DATASET_CNN_VALIDACION_CSV = SALIDA_DIR / "dataset_cnn_validacion.csv"

SCORE_AIRBNB_CSV = SALIDA_DIR / "score_visual_airbnb_etiquetado.csv"
SCORE_AIRBNB_XLSX = SALIDA_DIR / "score_visual_airbnb_etiquetado.xlsx"

# ------------------------------------------------------------
# 2. Validar archivos
# ------------------------------------------------------------
if not MANIFEST_SPLIT_CSV.exists():
    raise FileNotFoundError(f"No encontré el manifest: {MANIFEST_SPLIT_CSV}")

if not ETIQUETAS_CSV.exists():
    raise FileNotFoundError(
        f"No encontré etiquetas: {ETIQUETAS_CSV}. Primero debes etiquetar las fotos con la CELDA 7."
    )

# ------------------------------------------------------------
# 3. Cargar archivos
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_SPLIT_CSV)
etiquetas = pd.read_csv(ETIQUETAS_CSV)

print("Manifest cargado:", manifest.shape)
print("Etiquetas cargadas:", etiquetas.shape)

# ------------------------------------------------------------
# 4. Normalizar columnas base
# ------------------------------------------------------------
if "image_path" not in manifest.columns and "ruta_local" in manifest.columns:
    manifest["image_path"] = manifest["ruta_local"]

if "image_path" not in etiquetas.columns and "ruta_local" in etiquetas.columns:
    etiquetas["image_path"] = etiquetas["ruta_local"]

if "id_airbnb" not in manifest.columns:
    raise ValueError("El manifest no tiene columna id_airbnb.")

if "id_airbnb" not in etiquetas.columns:
    raise ValueError("El archivo de etiquetas no tiene columna id_airbnb.")

if "image_path" not in manifest.columns:
    raise ValueError("El manifest no tiene columna image_path ni ruta_local.")

if "image_path" not in etiquetas.columns:
    raise ValueError("El archivo de etiquetas no tiene columna image_path ni ruta_local.")


def normalizar_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x


def normalizar_ruta(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = x.replace("\\", "/")
    x = x.replace("//", "/")
    x = x.replace("./", "")
    return x.strip()


def resolver_ruta_local(x):
    if pd.isna(x):
        return ""

    x_str = str(x).strip()

    p1 = Path(x_str)
    if p1.exists():
        return str(p1)

    x_norm = normalizar_ruta(x_str)
    p2 = Path(x_norm)
    if p2.exists():
        return str(p2)

    return x_str


manifest["id_airbnb"] = manifest["id_airbnb"].apply(normalizar_id)
etiquetas["id_airbnb"] = etiquetas["id_airbnb"].apply(normalizar_id)

manifest["image_path"] = manifest["image_path"].astype(str)
etiquetas["image_path"] = etiquetas["image_path"].astype(str)

manifest["image_path_norm"] = manifest["image_path"].apply(normalizar_ruta)
etiquetas["image_path_norm"] = etiquetas["image_path"].apply(normalizar_ruta)

manifest["merge_key"] = manifest["id_airbnb"] + "||" + manifest["image_path_norm"]
etiquetas["merge_key"] = etiquetas["id_airbnb"] + "||" + etiquetas["image_path_norm"]

# ------------------------------------------------------------
# 5. Limpiar etiquetas válidas
# ------------------------------------------------------------
if "calidad_final" not in etiquetas.columns:
    raise ValueError("El archivo de etiquetas no tiene la columna calidad_final.")

etiquetas_validas = etiquetas.copy()

etiquetas_validas = etiquetas_validas[
    etiquetas_validas["calidad_final"].notna()
].copy()

etiquetas_validas["calidad_final"] = (
    etiquetas_validas["calidad_final"]
    .astype(str)
    .str.strip()
    .str.lower()
)

etiquetas_validas = etiquetas_validas[
    etiquetas_validas["calidad_final"] != ""
].copy()

if "estado_etiquetado" in etiquetas_validas.columns:
    etiquetas_validas["estado_etiquetado"] = (
        etiquetas_validas["estado_etiquetado"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    etiquetas_validas = etiquetas_validas[
        etiquetas_validas["estado_etiquetado"].isin(["etiquetado", "no_evaluable"])
    ].copy()

print("\nEtiquetas válidas:", etiquetas_validas.shape)

print("\nDistribución calidad_final en etiquetas:")
display(
    etiquetas_validas["calidad_final"]
    .value_counts(dropna=False)
    .reset_index(name="n_fotos")
    .rename(columns={"index": "calidad_final"})
)

if "estado_etiquetado" in etiquetas_validas.columns:
    print("\nDistribución estado_etiquetado:")
    display(
        etiquetas_validas["estado_etiquetado"]
        .value_counts(dropna=False)
        .reset_index(name="n_fotos")
        .rename(columns={"index": "estado_etiquetado"})
    )

# ------------------------------------------------------------
# 6. Revisar si las llaves cruzan
# ------------------------------------------------------------
keys_manifest = set(manifest["merge_key"])
keys_etiquetas = set(etiquetas_validas["merge_key"])
keys_comunes = keys_manifest.intersection(keys_etiquetas)

print("\nLlaves en manifest:", len(keys_manifest))
print("Llaves en etiquetas:", len(keys_etiquetas))
print("Llaves comunes:", len(keys_comunes))

if len(keys_comunes) == 0:
    print("\nNo cruzó ninguna imagen. Mostrando ejemplos para revisar:")

    print("\nEjemplo manifest:")
    display(manifest[["id_airbnb", "image_path", "image_path_norm", "merge_key"]].head())

    print("\nEjemplo etiquetas:")
    display(etiquetas_validas[["id_airbnb", "image_path", "image_path_norm", "merge_key"]].head())

    raise ValueError(
        "No se pudo cruzar manifest con etiquetas. Revisa que ambos usen las mismas rutas de imágenes."
    )

# ------------------------------------------------------------
# 7. Preparar manifest limpio antes del merge
# ------------------------------------------------------------
columnas_etiquetado_que_sobran = [
    "iluminacion",
    "encuadre",
    "nitidez",
    "atractivo_visual",
    "utilidad_huesped",
    "puntaje_total",
    "calidad_final",
    "estado_etiquetado",
    "observacion",
    "fecha_etiquetado",
    "label_calidad"
]

manifest_limpio = manifest.drop(
    columns=[c for c in columnas_etiquetado_que_sobran if c in manifest.columns],
    errors="ignore"
).copy()

# ------------------------------------------------------------
# 8. Cruzar manifest + etiquetas
# ------------------------------------------------------------
cols_etiquetas = [
    "merge_key",
    "iluminacion",
    "encuadre",
    "nitidez",
    "atractivo_visual",
    "utilidad_huesped",
    "puntaje_total",
    "calidad_final",
    "estado_etiquetado",
    "observacion",
    "fecha_etiquetado"
]

cols_etiquetas = [c for c in cols_etiquetas if c in etiquetas_validas.columns]

df = manifest_limpio.merge(
    etiquetas_validas[cols_etiquetas],
    on="merge_key",
    how="inner"
)

print("\nDataset cruzado manifest + etiquetas:", df.shape)

if "calidad_final" not in df.columns:
    if "calidad_final_y" in df.columns:
        df["calidad_final"] = df["calidad_final_y"]
    elif "calidad_final_x" in df.columns:
        df["calidad_final"] = df["calidad_final_x"]
    else:
        raise KeyError("No existe calidad_final después del cruce.")

print("\nDistribución calidad_final después del cruce:")
display(
    df["calidad_final"]
    .value_counts(dropna=False)
    .reset_index(name="n_fotos")
    .rename(columns={"index": "calidad_final"})
)

print("\nAvance por split:")
display(
    df.groupby(["split_calidad", "calidad_final"], dropna=False)
    .size()
    .reset_index(name="n_fotos")
)

# ------------------------------------------------------------
# 9. Resolver ruta local y validar existencia de imágenes
# ------------------------------------------------------------
df["ruta_imagen_final"] = df["image_path"].apply(resolver_ruta_local)
df["existe_imagen"] = df["ruta_imagen_final"].apply(lambda x: Path(x).exists())

print("\nExistencia de imágenes:")
display(
    df["existe_imagen"]
    .value_counts(dropna=False)
    .reset_index(name="n_fotos")
    .rename(columns={"index": "existe_imagen"})
)

no_existen = df[df["existe_imagen"] == False].copy()
if len(no_existen) > 0:
    print("\nEjemplos de imágenes que no existen:")
    display(no_existen[["id_airbnb", "image_path", "ruta_imagen_final"]].head(10))

df = df[df["existe_imagen"] == True].copy()

# ------------------------------------------------------------
# 10. Crear etiqueta final para CNN
# ------------------------------------------------------------
df["label_calidad"] = (
    df["calidad_final"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Guardar dataset completo, incluyendo no_evaluable
df.to_csv(DATASET_COMPLETO_CSV, index=False, encoding="utf-8-sig")
df.to_excel(DATASET_COMPLETO_XLSX, index=False)

print("\nDataset completo etiquetado guardado:")
print(DATASET_COMPLETO_CSV)
print(DATASET_COMPLETO_XLSX)

# Para entrenar CNN excluimos no_evaluable
df_cnn = df[df["label_calidad"].isin(["alta", "media"])].copy()

df_cnn.to_csv(DATASET_CNN_CSV, index=False, encoding="utf-8-sig")
df_cnn.to_excel(DATASET_CNN_XLSX, index=False)

# Archivos separados por split para la CNN
df_cnn[df_cnn["split_calidad"] == "train"].to_csv(DATASET_CNN_TRAIN_CSV, index=False, encoding="utf-8-sig")
df_cnn[df_cnn["split_calidad"] == "test"].to_csv(DATASET_CNN_TEST_CSV, index=False, encoding="utf-8-sig")
df_cnn[df_cnn["split_calidad"] == "validacion"].to_csv(DATASET_CNN_VALIDACION_CSV, index=False, encoding="utf-8-sig")

print("\nDataset CNN guardado:")
print(DATASET_CNN_CSV)
print(DATASET_CNN_XLSX)

print("\nArchivos separados para CNN:")
print(DATASET_CNN_TRAIN_CSV)
print(DATASET_CNN_TEST_CSV)
print(DATASET_CNN_VALIDACION_CSV)

print("\nFilas para entrenar/probar/validar CNN:", len(df_cnn))

print("\nDistribución final para CNN:")
if len(df_cnn) > 0:
    display(
        df_cnn.groupby(["split_calidad", "label_calidad"])
        .size()
        .reset_index(name="n_fotos")
    )
else:
    print("No hay filas para entrenar CNN.")

clases_presentes = set(df_cnn["label_calidad"].dropna().unique())
clases_esperadas = {"alta", "media"}
clases_faltantes = clases_esperadas - clases_presentes

print("\nEntrenamiento binario por mediana: alta vs media.")
if clases_faltantes:
    print("AVISO: faltan clases en el dataset CNN:", sorted(clases_faltantes))
print("Las fotos no_evaluable se quedan como descarte, no como clase principal.")


# ------------------------------------------------------------
# 11. Score visual por Airbnb
# ------------------------------------------------------------
mapa_puntos = {
    "alta": 1,
    "media": 0
}

if len(df_cnn) > 0:
    df_score = df_cnn.copy()
    df_score["puntos_calidad"] = df_score["label_calidad"].map(mapa_puntos)

    score_airbnb = (
        df_score
        .groupby(["id_airbnb", "split_calidad"], as_index=False)
        .agg(
            n_fotos_validas=("ruta_imagen_final", "count"),
            n_altas=("label_calidad", lambda x: (x == "alta").sum()),
            n_medias=("label_calidad", lambda x: (x == "media").sum()),
            score_visual=("puntos_calidad", "mean")
        )
    )

    score_airbnb["pct_altas"] = score_airbnb["n_altas"] / score_airbnb["n_fotos_validas"]
    score_airbnb["pct_medias"] = score_airbnb["n_medias"] / score_airbnb["n_fotos_validas"]

    # Score visual binario: 1 = alta, 0 = media.
    # El Airbnb queda como alta si al menos la mitad de sus fotos quedaron por encima de la mediana.
    score_airbnb["calidad_visual_airbnb"] = score_airbnb["pct_altas"].apply(
        lambda x: "alta" if x >= 0.50 else "media"
    )

    score_airbnb.to_csv(SCORE_AIRBNB_CSV, index=False, encoding="utf-8-sig")
    score_airbnb.to_excel(SCORE_AIRBNB_XLSX, index=False)

    print("\nScore visual por Airbnb guardado:")
    print(SCORE_AIRBNB_CSV)
    print(SCORE_AIRBNB_XLSX)

    print("\nVista previa score visual por Airbnb:")
    display(score_airbnb.head(20))

    print("\nDistribución calidad visual por Airbnb:")
    display(
        score_airbnb.groupby(["split_calidad", "calidad_visual_airbnb"])
        .size()
        .reset_index(name="n_airbnb")
    )

else:
    print("\nNo se generó score por Airbnb porque no hay fotos válidas para CNN.")

# ------------------------------------------------------------
# 12. Resumen final
# ------------------------------------------------------------
print("\nRESUMEN FINAL")
print("Fotos etiquetadas totales cruzadas:", len(df))
print("Fotos útiles para CNN:", len(df_cnn))
print("Fotos no evaluables:", (df["label_calidad"] == "no_evaluable").sum())

print("\nUso correcto de splits:")
print("- train      -> entrenar")
print("- test       -> probar/comparar modelos")
print("- validacion -> validación final")

print("\nArchivos principales:")
print("- Dataset completo:", DATASET_COMPLETO_CSV)
print("- Dataset CNN:", DATASET_CNN_CSV)
print("- Train:", DATASET_CNN_TRAIN_CSV)
print("- Test:", DATASET_CNN_TEST_CSV)
print("- Validación:", DATASET_CNN_VALIDACION_CSV)
print("- Score Airbnb:", SCORE_AIRBNB_CSV)


Manifest cargado: (4480, 26)
Etiquetas cargadas: (4480, 20)

Etiquetas válidas: (4480, 22)

Distribución calidad_final en etiquetas:


,calidad_final,n_fotos
0,alta,2256
1,media,2224



Distribución estado_etiquetado:


,estado_etiquetado,n_fotos
0,etiquetado,4480



Llaves en manifest: 4480
Llaves en etiquetas: 4480
Llaves comunes: 4480

Dataset cruzado manifest + etiquetas: (4480, 28)

Distribución calidad_final después del cruce:


...


Entrenamiento binario por mediana: alta vs media.
Las fotos no_evaluable se quedan como descarte, no como clase principal.

Score visual por Airbnb guardado:
salida_airbnb_fotos\score_visual_airbnb_etiquetado.csv
salida_airbnb_fotos\score_visual_airbnb_etiquetado.xlsx

Vista previa score visual por Airbnb:


,id_airbnb,split_calidad,n_fotos_validas,n_altas,n_medias,score_visual,pct_altas,pct_medias,calidad_visual_airbnb
0,1086343485196324895,validacion,30,10,20,0.333333,0.333333,0.666667,media
1,1089836481621094489,train,64,36,28,0.562500,0.562500,0.437500,alta
2,1118142286350817818,validacion,62,16,46,0.258065,0.258065,0.741935,media
3,1132618139969268206,train,93,50,43,0.537634,0.537634,0.462366,alta
4,1135034995673292208,train,125,54,71,0.432000,0.432000,0.568000,media
5,1142890900267610713,test,67,23,44,0.343284,0.343284,0.656716,media
6,1158688913933082401,validacion,66,31,35,0.469697,0.469697,0.530303,media
7,1173214265061225196,train,52,28,24,0.538462,0.538462,0.461538,alta
8,1173484430269756134,train,104,70,34,0.673077,0.673077,0.326923,alta
9,1190612003084495196,train,84,50,34,0.595238,0.595238,0.404762,alta



Distribución calidad visual por Airbnb:


,split_calidad,calidad_visual_airbnb,n_airbnb
0,test,alta,3
1,test,media,4
2,train,alta,20
3,train,media,15
4,validacion,alta,4
5,validacion,media,6



RESUMEN FINAL
Fotos etiquetadas totales cruzadas: 4480
Fotos útiles para CNN: 4480
Fotos no evaluables: 0

Uso correcto de splits:
- train      -> entrenar
- test       -> probar/comparar modelos
- validacion -> validación final

Archivos principales:
- Dataset completo: salida_airbnb_fotos\dataset_calidad_etiquetado_completo.csv
- Dataset CNN: salida_airbnb_fotos\dataset_cnn_calidad_etiquetado.csv
- Train: salida_airbnb_fotos\dataset_cnn_train.csv
- Test: salida_airbnb_fotos\dataset_cnn_test.csv
- Validación: salida_airbnb_fotos\dataset_cnn_validacion.csv
- Score Airbnb: salida_airbnb_fotos\score_visual_airbnb_etiquetado.csv


## Guía rápida del módulo CNN

Ejecuta las celdas en orden. La plantilla deja evidencia para la rúbrica sin que tengas que armar tablas manualmente.

| Paso | Celda | Resultado |
|---|---|---|
| 1 | Celda 8 | Consolida etiquetas y crea `dataset_cnn_calidad_etiquetado.csv`. |
| 2 | Celda 9 | Prepara imágenes y DataLoaders. |
| 3 | Celda 10 | Define modelos, entrenamiento y métricas. |
| 4 | Celda 11 | Entrena varias arquitecturas y selecciona el mejor modelo. |
| 5 | Celda 12 | Comprueba el modelo ganador en `validacion`. |
| 6 | Celda 14 | Genera tablas para explicar la CNN en el informe. |
| 7 | Celda 15 | Calcula proporciones de fotos `alta` y `media` por Airbnb. |
| 8 | Celda 16 | Aplica el modelo a una nueva tanda desde otro Excel tipo G4. |

La idea es que el notebook funcione como tutorial: cada bloque explica qué hace, qué archivo usa y qué salida genera.


## Inicio del módulo CNN

La CNN recibe una fotograf?a del alojamiento y la clasifica en dos niveles. Estas clases vienen de la Celda 7: `alta` corresponde a fotos con `puntaje_total` en o por encima de la mediana, y `media` a fotos por debajo de esa mediana.

| Etiqueta | Lectura práctica |
|---|---|
| `alta` | Foto de buena calidad visual o alta utilidad comercial. |
| `media` | Foto útil, pero menos atractiva o menos informativa. |

Las fotos `no_evaluable` se excluyen del entrenamiento porque no muestran claramente el inmueble, por ejemplo carteles, mapas, reglas, logos o capturas.

El objetivo final no es solo clasificar fotos sueltas. El objetivo operativo es resumir cada Airbnb con dos indicadores:

| Indicador | Interpretación |
|---|---|
| `prop_fotos_buena_calidad` | Proporción de fotos clasificadas como `alta`. |
| `prop_fotos_calidad_media` | Proporción de fotos clasificadas como `media`. |

Estos indicadores luego pueden unirse al Excel principal del proyecto.


### Celda 9 — Preparar imágenes para la CNN

Esta celda toma `dataset_cnn_calidad_etiquetado.csv`, filtra las fotos `alta` y `media`, verifica que los archivos existan y arma los DataLoaders.

También define los hiperparámetros principales: tamaño de imagen, batch size, epochs, learning rates y data augmentation. El aumento de datos solo se aplica al entrenamiento para que el modelo no memorice exactamente las mismas fotos.


In [29]:
# ============================================================
# CELDA 9 - Preparar dataset e imágenes para la CNN
# Lee dataset_cnn_calidad_etiquetado.csv y arma los DataLoaders.
# Parámetros clave: IMG_SIZE, BATCH_SIZE, learning rates y epochs.
# No modifica las particiones: usa directamente split_calidad.
# Versión corregida: SIN sklearn.metrics para evitar error scipy/_propack.
# ============================================================

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Métricas manuales SIN sklearn
# Esto reemplaza accuracy_score, precision_score, recall_score,
# f1_score, classification_report, confusion_matrix y ConfusionMatrixDisplay.
# ------------------------------------------------------------

def accuracy_score(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    if len(y_true) == 0:
        return 0
    return float((y_true == y_pred).mean())


def precision_score(y_true, y_pred, pos_label=1, zero_division=0, **kwargs):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == pos_label) & (y_pred == pos_label))
    fp = np.sum((y_true != pos_label) & (y_pred == pos_label))

    if (tp + fp) == 0:
        return zero_division

    return float(tp / (tp + fp))


def recall_score(y_true, y_pred, pos_label=1, zero_division=0, **kwargs):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == pos_label) & (y_pred == pos_label))
    fn = np.sum((y_true == pos_label) & (y_pred != pos_label))

    if (tp + fn) == 0:
        return zero_division

    return float(tp / (tp + fn))


def f1_score(y_true, y_pred, pos_label=1, zero_division=0, **kwargs):
    p = precision_score(y_true, y_pred, pos_label=pos_label, zero_division=zero_division)
    r = recall_score(y_true, y_pred, pos_label=pos_label, zero_division=zero_division)

    if (p + r) == 0:
        return zero_division

    return float(2 * p * r / (p + r))


def confusion_matrix(y_true, y_pred, labels=None):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if labels is None:
        labels = sorted(list(set(y_true.tolist()) | set(y_pred.tolist())))

    label_to_idx = {label: i for i, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for real, pred in zip(y_true, y_pred):
        if real in label_to_idx and pred in label_to_idx:
            cm[label_to_idx[real], label_to_idx[pred]] += 1

    return cm


def classification_report(y_true, y_pred, target_names=None, digits=4, output_dict=False, **kwargs):
    labels = sorted(list(set(np.array(y_true).tolist()) | set(np.array(y_pred).tolist())))

    if target_names is None:
        target_names = [str(x) for x in labels]

    rows = {}
    for label, name in zip(labels, target_names):
        p = precision_score(y_true, y_pred, pos_label=label, zero_division=0)
        r = recall_score(y_true, y_pred, pos_label=label, zero_division=0)
        f1 = f1_score(y_true, y_pred, pos_label=label, zero_division=0)
        support = int(np.sum(np.array(y_true) == label))

        rows[name] = {
            "precision": p,
            "recall": r,
            "f1-score": f1,
            "support": support
        }

    acc = accuracy_score(y_true, y_pred)

    if output_dict:
        rows["accuracy"] = acc
        return rows

    texto = ""
    texto += f"{'clase':<15}{'precision':>12}{'recall':>12}{'f1-score':>12}{'support':>12}\n"
    texto += "-" * 63 + "\n"

    for name, vals in rows.items():
        texto += (
            f"{name:<15}"
            f"{vals['precision']:>12.{digits}f}"
            f"{vals['recall']:>12.{digits}f}"
            f"{vals['f1-score']:>12.{digits}f}"
            f"{vals['support']:>12}\n"
        )

    texto += "\n"
    texto += f"accuracy: {acc:.{digits}f}\n"

    return texto


class ConfusionMatrixDisplay:
    def __init__(self, confusion_matrix, display_labels=None):
        self.confusion_matrix = confusion_matrix
        self.display_labels = display_labels

    def plot(self, cmap=None, values_format="d", ax=None, **kwargs):
        cm = self.confusion_matrix

        if ax is None:
            fig, ax = plt.subplots(figsize=(5, 4))

        im = ax.imshow(cm, cmap=cmap)

        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")
        ax.set_title("Matriz de confusión")

        n = cm.shape[0]

        if self.display_labels is None:
            labels = list(range(n))
        else:
            labels = self.display_labels

        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)

        for i in range(n):
            for j in range(n):
                ax.text(j, i, format(cm[i, j], values_format), ha="center", va="center")

        plt.colorbar(im, ax=ax)
        plt.tight_layout()

        return self


# ------------------------------------------------------------
# Reproducibilidad
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------
SALIDA_DIR = Path("salida_airbnb_fotos")
DATASET_CNN_CSV = SALIDA_DIR / "dataset_cnn_calidad_etiquetado.csv"

BATCH_SIZE = 64
NUM_EPOCHS_HEAD = 8       # primera fase: solo cabeza clasificadora
NUM_EPOCHS_FINE = 25      # segunda fase: fine-tuning
LR_HEAD = 1e-3
LR_FINE = 1e-5
IMG_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

# ------------------------------------------------------------
# Cargar dataset
# ------------------------------------------------------------
df = pd.read_csv(DATASET_CNN_CSV)

print("Dataset cargado:", df.shape)
display(df.head())

# Nos quedamos solo con las 2 clases reales
df = df[df["label_calidad"].isin(["alta", "media"])].copy()

# Asegurar rutas
if "ruta_imagen_final" in df.columns:
    df["path_modelo"] = df["ruta_imagen_final"].astype(str)
else:
    df["path_modelo"] = df["image_path"].astype(str)

df["existe"] = df["path_modelo"].apply(lambda x: Path(x).exists())
df = df[df["existe"] == True].copy()

print("\nDataset filtrado para CNN:", df.shape)

print("\nDistribución por split y clase:")
display(
    df.groupby(["split_calidad", "label_calidad"])
    .size()
    .reset_index(name="n_fotos")
)

# ------------------------------------------------------------
# Mapeo de etiquetas
# ------------------------------------------------------------
label_to_idx = {
    "media": 0,
    "alta": 1
}

idx_to_label = {
    0: "media",
    1: "alta"
}

df["label_idx"] = df["label_calidad"].map(label_to_idx)

# ------------------------------------------------------------
# Separar splits
# ------------------------------------------------------------
df_train = df[df["split_calidad"] == "train"].copy()
df_test = df[df["split_calidad"] == "test"].copy()
df_validacion = df[df["split_calidad"] == "validacion"].copy()

print("\nCantidad por split:")
print("Train:", len(df_train))
print("Test:", len(df_test))
print("Validación:", len(df_validacion))

if len(df_train) == 0:
    raise ValueError("No hay fotos en train.")

if len(df_test) == 0:
    raise ValueError("No hay fotos en test.")

if len(df_validacion) == 0:
    print("Advertencia: todavía no hay fotos etiquetadas en validación final.")

# ------------------------------------------------------------
# Dataset personalizado
# ------------------------------------------------------------
class AirbnbImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["path_modelo"]
        label = int(row["label_idx"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# ------------------------------------------------------------
# Transformaciones
# ------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------
train_dataset = AirbnbImageDataset(df_train, transform=train_transform)
test_dataset = AirbnbImageDataset(df_test, transform=eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

if len(df_validacion) > 0:
    validacion_dataset = AirbnbImageDataset(df_validacion, transform=eval_transform)
    validacion_loader = DataLoader(
        validacion_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )
else:
    validacion_loader = None

print("\nDataLoaders listos.")

Dispositivo: cuda
Dataset cargado: (4480, 30)


,id_airbnb,nro_foto,image_path,ruta_local,path_modelo,label_ambiente,split,metodo_extraccion,existe_imagen,registro_id,...,nitidez,atractivo_visual,utilidad_huesped,puntaje_total,calidad_final,estado_etiquetado,observacion,fecha_etiquetado,ruta_imagen_final,label_calidad
0,1086343485196324895,1,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1086343485196324895||img\1086343485196324895\1...,...,0,0,1,74.1,media,etiquetado,auto app: luma=0.46; contraste=0.19; bordes=0....,2026-06-22 03:12:44,img\1086343485196324895\1086343485196324895_fo...,media
1,1086343485196324895,2,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1086343485196324895||img\1086343485196324895\1...,...,0,1,1,75.6,media,etiquetado,auto app: luma=0.63; contraste=0.19; bordes=0....,2026-06-22 03:12:44,img\1086343485196324895\1086343485196324895_fo...,media
2,1086343485196324895,3,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1086343485196324895||img\1086343485196324895\1...,...,0,0,1,74.1,media,etiquetado,auto app: luma=0.47; contraste=0.19; bordes=0....,2026-06-22 03:12:44,img\1086343485196324895\1086343485196324895_fo...,media
3,1086343485196324895,4,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1086343485196324895||img\1086343485196324895\1...,...,0,0,1,73.4,media,etiquetado,auto app: luma=0.49; contraste=0.18; bordes=0....,2026-06-22 03:12:44,img\1086343485196324895\1086343485196324895_fo...,media
4,1086343485196324895,5,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,img\1086343485196324895\1086343485196324895_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1086343485196324895||img\1086343485196324895\1...,...,0,0,1,68.4,media,etiquetado,auto app: luma=0.45; contraste=0.17; bordes=0....,2026-06-22 03:12:44,img\1086343485196324895\1086343485196324895_fo...,media



Dataset filtrado para CNN: (4480, 31)

Distribución por split y clase:


,split_calidad,label_calidad,n_fotos
0,test,alta,252
1,test,media,250
2,train,alta,1536
3,train,media,1496
4,validacion,alta,468
5,validacion,media,478



Cantidad por split:
Train: 3032
Test: 502
Validación: 946

DataLoaders listos.


### Celda 10 — Modelos, entrenamiento y métricas

Aquí se definen las arquitecturas que se van a comparar:

| Modelo | Tipo | Para qué sirve |
|---|---|---|
| CNN simple | Entrenada desde cero | Línea base para comparar. |
| MobileNetV2 | Preentrenado | Modelo liviano y rápido. |
| ResNet18 | Preentrenado | Buen equilibrio entre precisión y costo. |
| ResNet50 | Preentrenado | Más profundo, con mayor capacidad. |
| EfficientNet-B0 | Preentrenado | Arquitectura eficiente. |
| ConvNeXt Tiny | Preentrenado | CNN moderna y potente. |

La celda calcula accuracy, precision, recall, F1-score, loss y matriz de confusión. Estas métricas permiten comparar modelos sin elegir arbitrariamente.


In [30]:
# ============================================================
# CELDA 10 - Funciones para experimentar con CNN
# ============================================================
# Esta celda prepara todo lo necesario para comparar modelos CNN.
#
# Versión corregida:
# - NO usa sklearn.metrics
# - Calcula accuracy, precision_macro, recall_macro, f1_macro
#   y matriz de confusión manualmente.
# - Evita el error scipy/sklearn/_propack.
# ============================================================

import os
import gc
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print("Advertencia: matplotlib no está instalado. Se mostrarán matrices como texto.")

# ------------------------------------------------------------
# 0. Métricas manuales SIN sklearn
# ------------------------------------------------------------

def _to_numpy_1d(x):
    return np.array(x).reshape(-1)


def accuracy_score(y_true, y_pred):
    y_true = _to_numpy_1d(y_true)
    y_pred = _to_numpy_1d(y_pred)

    if len(y_true) == 0:
        return 0.0

    return float((y_true == y_pred).mean())


def confusion_matrix(y_true, y_pred, labels=None):
    y_true = _to_numpy_1d(y_true)
    y_pred = _to_numpy_1d(y_pred)

    if labels is None:
        labels = [0, 1]

    label_to_idx = {label: i for i, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for real, pred in zip(y_true, y_pred):
        if real in label_to_idx and pred in label_to_idx:
            cm[label_to_idx[real], label_to_idx[pred]] += 1

    return cm


def _precision_recall_f1_por_clase(y_true, y_pred, labels=None, zero_division=0):
    y_true = _to_numpy_1d(y_true)
    y_pred = _to_numpy_1d(y_pred)

    if labels is None:
        labels = [0, 1]

    resultados = {}

    for label in labels:
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))

        precision = tp / (tp + fp) if (tp + fp) > 0 else zero_division
        recall = tp / (tp + fn) if (tp + fn) > 0 else zero_division

        if (precision + recall) > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = zero_division

        support = int(np.sum(y_true == label))

        resultados[label] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": support
        }

    return resultados


def precision_score(y_true, y_pred, average="binary", pos_label=1, labels=None, zero_division=0, **kwargs):
    if labels is None:
        labels = [0, 1]

    resultados = _precision_recall_f1_por_clase(
        y_true,
        y_pred,
        labels=labels,
        zero_division=zero_division
    )

    if average == "macro":
        return float(np.mean([resultados[label]["precision"] for label in labels]))

    if average is None:
        return np.array([resultados[label]["precision"] for label in labels])

    return float(resultados[pos_label]["precision"])


def recall_score(y_true, y_pred, average="binary", pos_label=1, labels=None, zero_division=0, **kwargs):
    if labels is None:
        labels = [0, 1]

    resultados = _precision_recall_f1_por_clase(
        y_true,
        y_pred,
        labels=labels,
        zero_division=zero_division
    )

    if average == "macro":
        return float(np.mean([resultados[label]["recall"] for label in labels]))

    if average is None:
        return np.array([resultados[label]["recall"] for label in labels])

    return float(resultados[pos_label]["recall"])


def f1_score(y_true, y_pred, average="binary", pos_label=1, labels=None, zero_division=0, **kwargs):
    if labels is None:
        labels = [0, 1]

    resultados = _precision_recall_f1_por_clase(
        y_true,
        y_pred,
        labels=labels,
        zero_division=zero_division
    )

    if average == "macro":
        return float(np.mean([resultados[label]["f1"] for label in labels]))

    if average is None:
        return np.array([resultados[label]["f1"] for label in labels])

    return float(resultados[pos_label]["f1"])


def classification_report(y_true, y_pred, target_names=None, labels=None, digits=4, zero_division=0, output_dict=False, **kwargs):
    if labels is None:
        labels = [0, 1]

    if target_names is None:
        target_names = [str(label) for label in labels]

    resultados = _precision_recall_f1_por_clase(
        y_true,
        y_pred,
        labels=labels,
        zero_division=zero_division
    )

    acc = accuracy_score(y_true, y_pred)

    if output_dict:
        salida = {}

        for label, nombre in zip(labels, target_names):
            salida[nombre] = {
                "precision": resultados[label]["precision"],
                "recall": resultados[label]["recall"],
                "f1-score": resultados[label]["f1"],
                "support": resultados[label]["support"]
            }

        salida["accuracy"] = acc
        salida["macro avg"] = {
            "precision": precision_score(y_true, y_pred, average="macro", labels=labels, zero_division=zero_division),
            "recall": recall_score(y_true, y_pred, average="macro", labels=labels, zero_division=zero_division),
            "f1-score": f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=zero_division),
            "support": int(len(y_true))
        }

        return salida

    texto = ""
    texto += f"{'clase':<15}{'precision':>12}{'recall':>12}{'f1-score':>12}{'support':>12}\n"
    texto += "-" * 63 + "\n"

    for label, nombre in zip(labels, target_names):
        vals = resultados[label]
        texto += (
            f"{nombre:<15}"
            f"{vals['precision']:>12.{digits}f}"
            f"{vals['recall']:>12.{digits}f}"
            f"{vals['f1']:>12.{digits}f}"
            f"{vals['support']:>12}\n"
        )

    texto += "-" * 63 + "\n"
    texto += f"{'accuracy':<15}{'':>12}{'':>12}{acc:>12.{digits}f}{len(y_true):>12}\n"

    macro_precision = precision_score(y_true, y_pred, average="macro", labels=labels, zero_division=zero_division)
    macro_recall = recall_score(y_true, y_pred, average="macro", labels=labels, zero_division=zero_division)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=zero_division)

    texto += (
        f"{'macro avg':<15}"
        f"{macro_precision:>12.{digits}f}"
        f"{macro_recall:>12.{digits}f}"
        f"{macro_f1:>12.{digits}f}"
        f"{len(y_true):>12}\n"
    )

    return texto


class ConfusionMatrixDisplay:
    def __init__(self, confusion_matrix, display_labels=None):
        self.confusion_matrix = confusion_matrix
        self.display_labels = display_labels

    def plot(self, values_format="d", ax=None, cmap=None, **kwargs):
        cm = self.confusion_matrix

        if plt is None:
            print("Matriz de confusión:")
            print(cm)
            return self

        if ax is None:
            fig, ax = plt.subplots(figsize=(5, 4))

        im = ax.imshow(cm, cmap=cmap)

        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")

        n = cm.shape[0]

        if self.display_labels is None:
            labels = list(range(n))
        else:
            labels = self.display_labels

        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)

        for i in range(n):
            for j in range(n):
                ax.text(
                    j,
                    i,
                    format(cm[i, j], values_format),
                    ha="center",
                    va="center"
                )

        plt.colorbar(im, ax=ax)
        plt.tight_layout()

        return self


# ------------------------------------------------------------
# 1. Semilla para reproducibilidad
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USAR_AMP = torch.cuda.is_available()

print("Dispositivo:", device)
print("Mixed precision activado:", USAR_AMP)

# ------------------------------------------------------------
# 2. Verificación de variables que vienen de la CELDA 9
# ------------------------------------------------------------
variables_necesarias = ["df_train", "df_test", "df_validacion", "SALIDA_DIR"]

for var in variables_necesarias:
    if var not in globals():
        raise ValueError(f"Falta ejecutar la celda anterior. No existe la variable: {var}")

SALIDA_DIR = Path(SALIDA_DIR)

# ------------------------------------------------------------
# 3. Etiquetas del problema
# ------------------------------------------------------------
label_to_idx = {
    "media": 0,
    "alta": 1
}

idx_to_label = {
    0: "media",
    1: "alta"
}

# ------------------------------------------------------------
# 4. Detectar columna de ID Airbnb
# ------------------------------------------------------------
def detectar_columna_id(df):
    posibles = [
        "id_airbnb",
        "ID Airbnb",
        "ID_Airbnb",
        "airbnb_id",
        "room_id",
        "listing_id"
    ]

    for col in posibles:
        if col in df.columns:
            return col

    raise ValueError(
        "No encontré una columna de ID Airbnb. Revisa si existe id_airbnb, ID Airbnb, room_id o listing_id."
    )

ID_COL = detectar_columna_id(df_train)
print("Columna de ID Airbnb detectada:", ID_COL)

# ------------------------------------------------------------
# 5. Asegurar que los dataframes tengan las columnas necesarias
# ------------------------------------------------------------
def asegurar_columnas_cnn(df_local):
    """
    Esta función deja el dataframe listo para CNN.

    Debe tener:
    - path_modelo: ruta de la imagen.
    - label_calidad: etiqueta humana alta/media.
    - label_idx: etiqueta numérica 0/1.
    """

    df_local = df_local.copy()

    if "path_modelo" not in df_local.columns:
        if "ruta_imagen_final" in df_local.columns:
            df_local["path_modelo"] = df_local["ruta_imagen_final"].astype(str)
        elif "image_path" in df_local.columns:
            df_local["path_modelo"] = df_local["image_path"].astype(str)
        elif "ruta_imagen" in df_local.columns:
            df_local["path_modelo"] = df_local["ruta_imagen"].astype(str)
        else:
            raise ValueError(
                "No encontré columna de ruta de imagen. Se esperaba ruta_imagen_final, image_path o ruta_imagen."
            )

    if "label_calidad" not in df_local.columns:
        if "calidad_final" in df_local.columns:
            df_local["label_calidad"] = df_local["calidad_final"].astype(str)
        elif "label" in df_local.columns:
            df_local["label_calidad"] = df_local["label"].astype(str)
        else:
            raise ValueError(
                "No encontré columna de etiqueta. Se esperaba label_calidad, calidad_final o label."
            )

    df_local["label_calidad"] = df_local["label_calidad"].astype(str).str.lower().str.strip()
    df_local = df_local[df_local["label_calidad"].isin(["alta", "media"])].copy()

    df_local["label_idx"] = df_local["label_calidad"].map(label_to_idx)

    df_local["existe_imagen"] = df_local["path_modelo"].apply(lambda x: Path(str(x)).exists())
    df_local = df_local[df_local["existe_imagen"] == True].copy()

    return df_local.reset_index(drop=True)

df_train = asegurar_columnas_cnn(df_train)
df_test = asegurar_columnas_cnn(df_test)
df_validacion = asegurar_columnas_cnn(df_validacion)

print("Train:", df_train.shape)
print("Test:", df_test.shape)
print("Validación:", df_validacion.shape)

print("\nDistribución train:")
display(
    df_train["label_calidad"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "clase", "label_calidad": "n"})
)

print("\nDistribución test:")
display(
    df_test["label_calidad"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "clase", "label_calidad": "n"})
)

print("\nDistribución validación:")
display(
    df_validacion["label_calidad"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "clase", "label_calidad": "n"})
)

# ------------------------------------------------------------
# 6. Dataset personalizado de imágenes
# ------------------------------------------------------------
class AirbnbImageDatasetCNN(Dataset):
    """
    Dataset para cargar imágenes de Airbnb.

    Cada fila representa una foto.
    Devuelve:
    - imagen transformada
    - etiqueta numérica
    """

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["path_modelo"]
        label = int(row["label_idx"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# ------------------------------------------------------------
# 7. Transformaciones según configuración
# ------------------------------------------------------------
def crear_transforms(img_size=224, augmentation="base_color"):
    """
    Crea las transformaciones de imagen.

    augmentation:
    - base_color
    - limpia
    - crop_suave
    """

    resize_size = max(img_size + 32, 256)

    if augmentation == "base_color":
        train_transform = transforms.Compose([
            transforms.Resize((resize_size, resize_size)),
            transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(
                brightness=0.15,
                contrast=0.15,
                saturation=0.10
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    elif augmentation == "limpia":
        train_transform = transforms.Compose([
            transforms.Resize((resize_size, resize_size)),
            transforms.CenterCrop(img_size),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    elif augmentation == "crop_suave":
        train_transform = transforms.Compose([
            transforms.Resize((resize_size, resize_size)),
            transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    else:
        raise ValueError(f"augmentation no reconocido: {augmentation}")

    eval_transform = transforms.Compose([
        transforms.Resize((resize_size, resize_size)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return train_transform, eval_transform

# ------------------------------------------------------------
# 8. Crear loaders según configuración
# ------------------------------------------------------------
def crear_loaders(config):
    """
    Crea train_loader, test_loader y validacion_loader usando la configuración indicada.
    """

    img_size = config["img_size"]
    batch_size = config["batch_size"]
    augmentation = config["augmentation"]

    train_transform, eval_transform = crear_transforms(
        img_size=img_size,
        augmentation=augmentation
    )

    train_dataset = AirbnbImageDatasetCNN(df_train, transform=train_transform)
    test_dataset = AirbnbImageDatasetCNN(df_test, transform=eval_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    if len(df_validacion) > 0:
        validacion_dataset = AirbnbImageDatasetCNN(df_validacion, transform=eval_transform)
        validacion_loader = DataLoader(
            validacion_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )
    else:
        validacion_loader = None

    return train_loader, test_loader, validacion_loader

# ------------------------------------------------------------
# 9. Pesos de clase
# ------------------------------------------------------------
def crear_criterion():
    """
    CrossEntropyLoss con pesos de clase.
    """

    conteo_train = df_train["label_idx"].value_counts().sort_index()

    class_counts = torch.tensor(
        [conteo_train.get(0, 1), conteo_train.get(1, 1)],
        dtype=torch.float
    )

    class_weights = class_counts.sum() / (len(class_counts) * class_counts)
    class_weights = class_weights.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    return criterion, class_weights

criterion, class_weights = crear_criterion()
print("Pesos de clase:", class_weights)

# ------------------------------------------------------------
# 10. CNN simple desde cero
# ------------------------------------------------------------
class CNN_Simple(nn.Module):
    """
    CNN simple creada desde cero.
    """

    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ------------------------------------------------------------
# 11. Funciones auxiliares para transfer learning
# ------------------------------------------------------------
def congelar_todo(model):
    """
    Congela las capas preentrenadas.
    """

    for p in model.parameters():
        p.requires_grad = False


def descongelar_ultimas_capas(model, nombre_modelo):
    """
    Descongela solo las últimas capas del modelo.
    """

    nombre_modelo = nombre_modelo.lower()

    if "resnet" in nombre_modelo:
        for p in model.layer4.parameters():
            p.requires_grad = True
        for p in model.fc.parameters():
            p.requires_grad = True

    elif "mobilenet" in nombre_modelo:
        for p in model.features[-4:].parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True

    elif "efficientnet" in nombre_modelo:
        for p in model.features[-3:].parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True

    elif "convnext" in nombre_modelo:
        for p in model.features[-2:].parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True


def crear_modelo(nombre_modelo):
    """
    Crea una arquitectura CNN.
    """

    nombre_modelo = nombre_modelo.lower()

    if nombre_modelo == "cnn_simple":
        model = CNN_Simple(num_classes=2)
        return model

    if nombre_modelo == "mobilenet_v2":
        try:
            weights = models.MobileNet_V2_Weights.DEFAULT
            model = models.mobilenet_v2(weights=weights)
            print("MobileNetV2 con pesos preentrenados.")
        except Exception as e:
            print("MobileNetV2 sin pesos preentrenados:", e)
            model = models.mobilenet_v2(weights=None)

        congelar_todo(model)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, 2)
        return model

    if nombre_modelo == "resnet18":
        try:
            weights = models.ResNet18_Weights.DEFAULT
            model = models.resnet18(weights=weights)
            print("ResNet18 con pesos preentrenados.")
        except Exception as e:
            print("ResNet18 sin pesos preentrenados:", e)
            model = models.resnet18(weights=None)

        congelar_todo(model)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, 2)
        return model

    if nombre_modelo == "efficientnet_b0":
        try:
            weights = models.EfficientNet_B0_Weights.DEFAULT
            model = models.efficientnet_b0(weights=weights)
            print("EfficientNet-B0 con pesos preentrenados.")
        except Exception as e:
            print("EfficientNet-B0 sin pesos preentrenados:", e)
            model = models.efficientnet_b0(weights=None)

        congelar_todo(model)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, 2)
        return model

    if nombre_modelo == "convnext_tiny":
        try:
            weights = models.ConvNeXt_Tiny_Weights.DEFAULT
            model = models.convnext_tiny(weights=weights)
            print("ConvNeXt Tiny con pesos preentrenados.")
        except Exception as e:
            print("ConvNeXt Tiny sin pesos preentrenados:", e)
            model = models.convnext_tiny(weights=None)

        congelar_todo(model)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, 2)
        return model

    raise ValueError(f"Modelo no reconocido: {nombre_modelo}")

# ------------------------------------------------------------
# 12. Evaluación
# ------------------------------------------------------------
def evaluar_modelo(model, loader, split_name="test"):
    """
    Evalúa el modelo en un loader.
    """

    if loader is None:
        raise ValueError(f"No existe loader para el split: {split_name}")

    model.eval()

    y_true = []
    y_pred = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.cuda.amp.autocast(enabled=USAR_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)

            preds = torch.argmax(outputs, dim=1)

            running_loss += loss.item() * images.size(0)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)

    metrics = {
        "split": split_name,
        "loss": avg_loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred
    }

    return metrics


def imprimir_reporte(model, loader, nombre_modelo, split_name):
    """
    Imprime métricas y matriz de confusión.
    """

    metrics = evaluar_modelo(model, loader, split_name=split_name)

    print("=" * 70)
    print(f"Reporte: {nombre_modelo} - {split_name}")
    print("=" * 70)

    print(f"Loss:      {metrics['loss']:.4f}")
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision_macro']:.4f}")
    print(f"Recall:    {metrics['recall_macro']:.4f}")
    print(f"F1 macro:  {metrics['f1_macro']:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            metrics["y_true"],
            metrics["y_pred"],
            target_names=["media", "alta"],
            zero_division=0
        )
    )

    cm = confusion_matrix(metrics["y_true"], metrics["y_pred"], labels=[0, 1])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["media", "alta"]
    )

    disp.plot(values_format="d")

    if plt is not None:
        plt.title(f"Matriz de confusión - {nombre_modelo} - {split_name}")
        plt.show()

    return metrics

# ------------------------------------------------------------
# 13. Entrenamiento por experimento
# ------------------------------------------------------------
def contar_parametros_entrenables(model):
    """
    Cuenta cuántos parámetros serán actualizados durante el entrenamiento.
    """

    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def cpu_state_dict(model):
    """
    Guarda los pesos del modelo en CPU para evitar problemas de memoria GPU.
    """

    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def entrenar_experimento(nombre_modelo, config, experimento_id, etapa):
    """
    Entrena un modelo bajo una configuración específica.
    """

    print("\n" + "=" * 90)
    print(f"EXPERIMENTO: {experimento_id}")
    print(f"ETAPA: {etapa}")
    print(f"MODELO: {nombre_modelo}")
    print("CONFIGURACIÓN:")
    print(config)
    print("=" * 90)

    train_loader, test_loader, _ = crear_loaders(config)

    model = crear_modelo(nombre_modelo).to(device)

    historial = []

    mejor_f1 = -1
    mejor_estado = None
    mejor_epoch = None
    mejor_fase = None

    scaler = torch.cuda.amp.GradScaler(enabled=USAR_AMP)

    def entrenar_fase(fase, num_epochs, learning_rate):
        nonlocal mejor_f1, mejor_estado, mejor_epoch, mejor_fase, historial

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=learning_rate,
            weight_decay=config["weight_decay"]
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=3
        )

        paciencia = config["paciencia"]
        sin_mejora = 0

        print(f"\nFase: {fase}")
        print("Parámetros entrenables:", contar_parametros_entrenables(model))

        for epoch in range(num_epochs):
            model.train()

            train_loss = 0.0
            y_true_train = []
            y_pred_train = []

            for images, labels in train_loader:
                images = images.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.cuda.amp.autocast(enabled=USAR_AMP):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                train_loss += loss.item() * images.size(0)

                preds = torch.argmax(outputs, dim=1)
                y_true_train.extend(labels.cpu().numpy())
                y_pred_train.extend(preds.cpu().numpy())

            train_loss = train_loss / len(train_loader.dataset)

            train_acc = accuracy_score(y_true_train, y_pred_train)
            train_f1 = f1_score(y_true_train, y_pred_train, average="macro", zero_division=0)

            test_metrics = evaluar_modelo(model, test_loader, split_name="test")

            scheduler.step(test_metrics["f1_macro"])

            fila = {
                "experimento_id": experimento_id,
                "etapa": etapa,
                "modelo": nombre_modelo,
                "fase": fase,
                "epoch": epoch + 1,
                "img_size": config["img_size"],
                "batch_size": config["batch_size"],
                "augmentation": config["augmentation"],
                "epochs_head": config["epochs_head"],
                "epochs_fine": config["epochs_fine"],
                "lr_head": config["lr_head"],
                "lr_fine": config["lr_fine"],
                "weight_decay": config["weight_decay"],
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "train_f1_macro": train_f1,
                "test_loss": test_metrics["loss"],
                "test_accuracy": test_metrics["accuracy"],
                "test_precision_macro": test_metrics["precision_macro"],
                "test_recall_macro": test_metrics["recall_macro"],
                "test_f1_macro": test_metrics["f1_macro"]
            }

            historial.append(fila)

            print(
                f"{experimento_id} | {nombre_modelo} | {fase} | "
                f"Epoch {epoch+1:02d}/{num_epochs} | "
                f"train F1 {train_f1:.3f} | test F1 {test_metrics['f1_macro']:.3f}"
            )

            if test_metrics["f1_macro"] > mejor_f1:
                mejor_f1 = test_metrics["f1_macro"]
                mejor_estado = cpu_state_dict(model)
                mejor_epoch = epoch + 1
                mejor_fase = fase
                sin_mejora = 0
            else:
                sin_mejora += 1

            if sin_mejora >= paciencia:
                print(f"Early stopping en {fase}. Mejor F1 test: {mejor_f1:.4f}")
                break

    entrenar_fase(
        fase="head",
        num_epochs=config["epochs_head"],
        learning_rate=config["lr_head"]
    )

    if nombre_modelo != "cnn_simple" and config["epochs_fine"] > 0:
        descongelar_ultimas_capas(model, nombre_modelo)

        entrenar_fase(
            fase="fine_tuning",
            num_epochs=config["epochs_fine"],
            learning_rate=config["lr_fine"]
        )

    if mejor_estado is None:
        raise ValueError("No se pudo guardar un mejor estado del modelo. Revisa el entrenamiento.")

    model.load_state_dict(mejor_estado)

    test_loader = crear_loaders(config)[1]
    test_metrics = evaluar_modelo(model, test_loader, split_name="test")

    resumen = {
        "experimento_id": experimento_id,
        "etapa": etapa,
        "modelo": nombre_modelo,
        "img_size": config["img_size"],
        "batch_size": config["batch_size"],
        "augmentation": config["augmentation"],
        "epochs_head": config["epochs_head"],
        "epochs_fine": config["epochs_fine"],
        "lr_head": config["lr_head"],
        "lr_fine": config["lr_fine"],
        "weight_decay": config["weight_decay"],
        "mejor_fase": mejor_fase,
        "mejor_epoch": mejor_epoch,
        "test_loss": test_metrics["loss"],
        "accuracy": test_metrics["accuracy"],
        "precision_macro": test_metrics["precision_macro"],
        "recall_macro": test_metrics["recall_macro"],
        "f1_macro": test_metrics["f1_macro"]
    }

    return model, pd.DataFrame(historial), resumen

# ------------------------------------------------------------
# 14. Predicción sobre un dataframe
# ------------------------------------------------------------
def predecir_dataframe(model, dataframe, config):
    """
    Aplica el modelo a un dataframe de fotos.
    """

    model.eval()

    df_pred = dataframe.copy().reset_index(drop=True)

    _, eval_transform = crear_transforms(
        img_size=config["img_size"],
        augmentation=config["augmentation"]
    )

    pred_labels = []
    prob_media = []
    prob_alta = []

    with torch.no_grad():
        for _, row in df_pred.iterrows():
            image = Image.open(row["path_modelo"]).convert("RGB")
            image_tensor = eval_transform(image).unsqueeze(0).to(device)

            with torch.cuda.amp.autocast(enabled=USAR_AMP):
                output = model(image_tensor)
                probs = torch.softmax(output, dim=1).cpu().numpy()[0]

            pred_idx = int(np.argmax(probs))
            pred_label = idx_to_label[pred_idx]

            pred_labels.append(pred_label)
            prob_media.append(float(probs[0]))
            prob_alta.append(float(probs[1]))

    df_pred["label_calidad_real"] = df_pred["label_calidad"]
    df_pred["label_calidad_predicha"] = pred_labels
    df_pred["prob_media"] = prob_media
    df_pred["prob_alta"] = prob_alta

    df_pred["label_calidad_modelo_final"] = df_pred["label_calidad_predicha"]

    return df_pred

print("CELDA 10 lista.")
print("Ya se pueden ejecutar los experimentos de la CELDA 11.")

Dispositivo: cuda
Mixed precision activado: True
Columna de ID Airbnb detectada: id_airbnb
Train: (3032, 32)
Test: (502, 32)
Validación: (946, 32)

Distribución train:


,n,count
0,alta,1536
1,media,1496



Distribución test:


,n,count
0,alta,252
1,media,250



Distribución validación:


,n,count
0,media,478
1,alta,468


Pesos de clase: tensor([1.0134, 0.9870], device='cuda:0')
CELDA 10 lista.
Ya se pueden ejecutar los experimentos de la CELDA 11.


In [31]:
# ============================================================
# CELDA 10A - Optimización para aprovechar mejor la PC / GPU
# ============================================================
# Esta celda no entrena modelos.
# Solo ajusta PyTorch para usar mejor la computadora.
#
# ¿Qué hace?
# - Detecta si hay GPU disponible.
# - Activa optimizaciones de CUDA.
# - Mejora la carga de imágenes con DataLoader.
# - Sobrescribe la función crear_loaders() de la celda 10.
#
# En Windows/Jupyter se usa NUM_WORKERS = 0 para evitar cuelgues del DataLoader.
# ============================================================

import torch
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# 1. Detectar dispositivo
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USAR_AMP = torch.cuda.is_available()

print("Dispositivo usado:", device)

if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))

    # Acelera convoluciones cuando las imágenes tienen tamaño fijo
    torch.backends.cudnn.benchmark = True

    # Permite operaciones más rápidas en GPUs NVIDIA compatibles
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

    # En Windows/Jupyter, 0 es lo mas estable.
    NUM_WORKERS = 0
    PIN_MEMORY = True

else:
    print("No se detectó CUDA. El entrenamiento correrá en CPU y será más lento.")
    NUM_WORKERS = 0
    PIN_MEMORY = False

print("Mixed precision activado:", USAR_AMP)
print("NUM_WORKERS:", NUM_WORKERS)
print("PIN_MEMORY:", PIN_MEMORY)

# ------------------------------------------------------------
# 2. Sobrescribir función crear_loaders()
# ------------------------------------------------------------
def crear_loaders(config):
    """
    Crea los DataLoaders para train, test y validacion.

    Esta versión está optimizada para PC/GPU:
    - num_workers ayuda a cargar imágenes en paralelo.
    - pin_memory acelera el paso de datos hacia la GPU.
    - persistent_workers evita reiniciar workers en cada época.
    """

    img_size = config["img_size"]
    batch_size = config["batch_size"]
    augmentation = config["augmentation"]

    train_transform, eval_transform = crear_transforms(
        img_size=img_size,
        augmentation=augmentation
    )

    train_dataset = AirbnbImageDatasetCNN(df_train, transform=train_transform)
    test_dataset = AirbnbImageDatasetCNN(df_test, transform=eval_transform)
    validacion_dataset = AirbnbImageDatasetCNN(df_validacion, transform=eval_transform)

    usar_workers_persistentes = NUM_WORKERS > 0

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=usar_workers_persistentes
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=usar_workers_persistentes
    )

    validacion_loader = DataLoader(
        validacion_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=usar_workers_persistentes
    )

    return train_loader, test_loader, validacion_loader

print("CELDA 10A lista.")
print("Se optimizó crear_loaders() para aprovechar mejor la PC.")

Dispositivo usado: cuda
GPU detectada: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Mixed precision activado: True
NUM_WORKERS: 0
PIN_MEMORY: True
CELDA 10A lista.
Se optimizó crear_loaders() para aprovechar mejor la PC.


### Celda 11 — Entrenar y comparar modelos

Esta celda entrena todas las alternativas y guarda una tabla comparativa. El modelo ganador se elige con **F1-score macro** porque equilibra precision y recall entre las dos clases creadas por mediana: `alta` y `media`.

La CNN simple sirve como comparación mínima. Los modelos preentrenados muestran si transfer learning mejora el resultado.


In [13]:
# ============================================================
# CELDA 11 - Comparación de arquitecturas CNN usando GPU real
# ============================================================
# Esta celda compara arquitecturas y configuraciones.
#
# Correcciones incluidas:
# - Crea SALIDA_DIR si no existía.
# - Usa DEVICE definido en la CELDA 10A.
# - Verifica que CUDA esté disponible.
# - Mantiene compatibilidad con tu flujo anterior.
# - Muestra si el modelo devuelto está en GPU o CPU.
# - Guarda resultados, errores, hiperparámetros y candidatos.
# ============================================================

import gc
import copy
import re
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

# ------------------------------------------------------------
# 0. Carpeta de salida
# ------------------------------------------------------------
try:
    SALIDA_DIR
except NameError:
    SALIDA_DIR = Path("salida_airbnb_fotos")
    SALIDA_DIR.mkdir(parents=True, exist_ok=True)
    print("SALIDA_DIR no existía. Se creó:", SALIDA_DIR)

# ------------------------------------------------------------
# 0.1. Verificar DEVICE / GPU
# ------------------------------------------------------------

# Si quieres que la celda se detenga cuando no hay GPU, deja True.
# Si quieres que corra por CPU aunque sea lento, cambia a False.
EXIGIR_GPU = True

try:
    DEVICE
except NameError:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = DEVICE

USAR_GPU_REAL = DEVICE.type == "cuda" and torch.cuda.is_available()
USAR_AMP = USAR_GPU_REAL

# Dejamos estas variables como globales por si otras funciones las usan
globals()["DEVICE"] = DEVICE
globals()["device"] = device
globals()["USAR_AMP"] = USAR_AMP

print("============================================================")
print("VERIFICACIÓN ANTES DE ENTRENAR")
print("============================================================")
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA PyTorch:", torch.version.cuda)
print("DEVICE:", DEVICE)
print("USAR_GPU_REAL:", USAR_GPU_REAL)
print("USAR_AMP:", USAR_AMP)

if USAR_GPU_REAL:
    print("GPU detectada:", torch.cuda.get_device_name(0))
    memoria_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print("Memoria total GPU:", round(memoria_total, 2), "GB")

    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

else:
    print("No se detectó GPU CUDA real.")
    print("Si tienes NVIDIA, probablemente PyTorch no está usando CUDA.")

    if EXIGIR_GPU:
        raise RuntimeError(
            "CUDA no está disponible. La celda se detuvo para evitar entrenar por CPU. "
            "Si quieres correr por CPU, cambia EXIGIR_GPU = False."
        )


def mostrar_memoria_gpu(mensaje=""):
    """
    Muestra memoria usada/reservada en GPU.
    """
    if torch.cuda.is_available():
        memoria_usada = torch.cuda.memory_allocated(0) / (1024**3)
        memoria_reservada = torch.cuda.memory_reserved(0) / (1024**3)

        print(f"\nGPU memoria {mensaje}")
        print("Memoria usada:", round(memoria_usada, 3), "GB")
        print("Memoria reservada:", round(memoria_reservada, 3), "GB")


mostrar_memoria_gpu("al inicio")

# ------------------------------------------------------------
# 0.2. Verificar objetos necesarios de celdas anteriores
# ------------------------------------------------------------

objetos_necesarios = [
    "df_train",
    "df_test",
    "df_validacion",
    "crear_transforms",
    "AirbnbImageDatasetCNN",
    "entrenar_experimento",
    "label_to_idx",
    "idx_to_label"
]

faltantes = [nombre for nombre in objetos_necesarios if nombre not in globals()]

if len(faltantes) > 0:
    raise NameError(
        "Faltan objetos de celdas anteriores. "
        "Debes correr primero las celdas previas del notebook. "
        f"Faltantes: {faltantes}"
    )

# ------------------------------------------------------------
# 0.3. Función auxiliar para copiar pesos a CPU
# ------------------------------------------------------------
# Si ya existe en una celda anterior, se respeta.
# Si no existe, la creamos aquí.

if "cpu_state_dict" not in globals():

    def cpu_state_dict(model):
        """
        Copia el state_dict del modelo hacia CPU.
        Sirve para guardar checkpoints sin ocupar memoria GPU.
        """
        return {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

# ------------------------------------------------------------
# 1. Configuración general de corrida
# ------------------------------------------------------------

INCLUIR_CONVNEXT = True

# Si tu máquina es fuerte, deja True.
# Si quieres una corrida rápida, cambia a False.
USAR_TODAS_LAS_CONFIGURACIONES = True

# Si quieres probar SOLO la configuración fuerte, cambia esto a True.
SOLO_CONFIG_MONSTRUO = False

# Cantidad de candidatos que se guardarán para validación final
TOP_CANDIDATOS_TEST = 5

CHECKPOINTS_CANDIDATOS_DIR = SALIDA_DIR / "checkpoints_candidatos_cnn"
CHECKPOINTS_CANDIDATOS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Configuraciones de arquitectura / hiperparámetros
# ------------------------------------------------------------

CONFIG_ARQUITECTURA_BASE = {
    "img_size": 224,
    "batch_size": 64 if USAR_GPU_REAL else 16,
    "augmentation": "limpia",
    "epochs_head": 5,
    "epochs_fine": 12,
    "lr_head": 1e-3,
    "lr_fine": 1e-5,
    "weight_decay": 1e-4,
    "paciencia": 4
}

CONFIG_ARQUITECTURA_2 = {
    "img_size": 224,
    "batch_size": 32 if USAR_GPU_REAL else 8,
    "augmentation": "base_color",
    "epochs_head": 6,
    "epochs_fine": 15,
    "lr_head": 8e-4,
    "lr_fine": 8e-6,
    "weight_decay": 1e-4,
    "paciencia": 5
}

CONFIG_ARQUITECTURA_3 = {
    "img_size": 256,
    "batch_size": 32 if USAR_GPU_REAL else 8,
    "augmentation": "crop_suave",
    "epochs_head": 5,
    "epochs_fine": 14,
    "lr_head": 1e-3,
    "lr_fine": 5e-6,
    "weight_decay": 5e-5,
    "paciencia": 5
}

CONFIG_ARQUITECTURA_4 = {
    "img_size": 192,
    "batch_size": 64 if USAR_GPU_REAL else 16,
    "augmentation": "limpia",
    "epochs_head": 4,
    "epochs_fine": 10,
    "lr_head": 1e-3,
    "lr_fine": 2e-5,
    "weight_decay": 1e-4,
    "paciencia": 4
}

CONFIG_ARQUITECTURA_5_MONSTRUO = {
    "img_size": 256,
    "batch_size": 64 if USAR_GPU_REAL else 16,
    "augmentation": "crop_suave",
    "epochs_head": 8,
    "epochs_fine": 25,
    "lr_head": 8e-4,
    "lr_fine": 5e-6,
    "weight_decay": 1e-4,
    "paciencia": 5
}

CONFIGURACIONES_ARQUITECTURA = {
    "config_base_224_limpia": CONFIG_ARQUITECTURA_BASE,
    "config_2_224_color": CONFIG_ARQUITECTURA_2,
    "config_3_256_crop": CONFIG_ARQUITECTURA_3,
    "config_4_192_limpia": CONFIG_ARQUITECTURA_4,
    "config_5_256_monstruo_controlado": CONFIG_ARQUITECTURA_5_MONSTRUO
}

# Mantiene compatibilidad con código anterior
CONFIG_ARQUITECTURA = CONFIG_ARQUITECTURA_BASE

# ------------------------------------------------------------
# 3. Definir qué configuraciones correr
# ------------------------------------------------------------

if SOLO_CONFIG_MONSTRUO:
    CONFIGURACIONES_A_CORRER = {
        "config_5_256_monstruo_controlado": CONFIG_ARQUITECTURA_5_MONSTRUO
    }
elif USAR_TODAS_LAS_CONFIGURACIONES:
    CONFIGURACIONES_A_CORRER = CONFIGURACIONES_ARQUITECTURA
else:
    CONFIGURACIONES_A_CORRER = {
        "config_base_224_limpia": CONFIG_ARQUITECTURA_BASE
    }

print("\nConfiguraciones a evaluar:")
for nombre_config, config in CONFIGURACIONES_A_CORRER.items():
    print("\n-", nombre_config)
    print(config)

# ------------------------------------------------------------
# 4. Lista de arquitecturas
# ------------------------------------------------------------

modelos_arquitectura = [
    "cnn_simple",
    "mobilenet_v2",
    "resnet18",
    "efficientnet_b0"
]

if INCLUIR_CONVNEXT:
    modelos_arquitectura.append("convnext_tiny")

print("\nArquitecturas a evaluar:")
for m in modelos_arquitectura:
    print("-", m)

# ------------------------------------------------------------
# 5. Contenedores de resultados
# ------------------------------------------------------------

resultados_experimentos = []
historiales_experimentos = []
errores_experimentos = []
candidatos_top = []

mejor_global_test = {
    "f1_macro": -1,
    "modelo": None,
    "configuracion": None,
    "config": None,
    "state_dict": None,
    "resumen": None
}

# ------------------------------------------------------------
# 6. Funciones auxiliares
# ------------------------------------------------------------

def limpiar_nombre_archivo(texto):
    texto = str(texto)
    texto = re.sub(r"[^a-zA-Z0-9_\-]+", "_", texto)
    return texto[:120]


def liberar_memoria():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def valor_float(x, default=-1):
    try:
        return float(x)
    except Exception:
        return default


def verificar_device_modelo(model, nombre_modelo=""):
    """
    Verifica en qué dispositivo quedó el modelo.
    Si sale CPU, el problema está dentro de entrenar_experimento().
    """

    try:
        device_modelo = next(model.parameters()).device
        print("\nDevice del modelo devuelto:", device_modelo)

        if USAR_GPU_REAL and device_modelo.type != "cuda":
            print("ADVERTENCIA:")
            print("El modelo devuelto está en CPU, no en GPU.")
            print("Esto indica que entrenar_experimento() probablemente no está usando .to(DEVICE).")
            print("Hay que corregir la celda donde se define entrenar_experimento().")

        return device_modelo

    except Exception as e:
        print("No se pudo verificar el device del modelo:", e)
        return None


def actualizar_top_candidatos(model, resumen, config, nombre_config):
    """
    Guarda en memoria solo los TOP candidatos según F1 macro en test.
    Luego la CELDA 12 los evaluará en validación final.
    """

    global candidatos_top
    global mejor_global_test

    f1 = valor_float(resumen.get("f1_macro", -1), -1)

    candidato = {
        "f1_macro": f1,
        "modelo": resumen.get("modelo"),
        "configuracion": nombre_config,
        "config": copy.deepcopy(config),
        "state_dict": cpu_state_dict(model),
        "resumen": copy.deepcopy(resumen)
    }

    candidatos_top.append(candidato)

    candidatos_top = sorted(
        candidatos_top,
        key=lambda x: x["f1_macro"],
        reverse=True
    )

    if len(candidatos_top) > TOP_CANDIDATOS_TEST:
        descartados = candidatos_top[TOP_CANDIDATOS_TEST:]
        candidatos_top = candidatos_top[:TOP_CANDIDATOS_TEST]

        for d in descartados:
            if "state_dict" in d:
                del d["state_dict"]

    if f1 > mejor_global_test["f1_macro"]:
        mejor_global_test = copy.deepcopy(candidato)

        print("\nNuevo mejor candidato por TEST hasta ahora")
        print("Modelo:", mejor_global_test["modelo"])
        print("Configuración:", mejor_global_test["configuracion"])
        print("F1 macro test:", round(mejor_global_test["f1_macro"], 4))


def correr_experimento_arquitectura(nombre_modelo, nombre_config, config):
    """
    Entrena una arquitectura CNN con una configuración específica.
    """

    config_local = copy.deepcopy(config)

    experimento_id = f"ARQ_{nombre_modelo}_{nombre_config}"

    print("\nDevice que se enviará al entrenamiento:", DEVICE)
    mostrar_memoria_gpu("antes del experimento")

    model, hist_df, resumen = entrenar_experimento(
        nombre_modelo=nombre_modelo,
        config=config_local,
        experimento_id=experimento_id,
        etapa="comparacion_arquitectura_config"
    )

    verificar_device_modelo(model, nombre_modelo)
    mostrar_memoria_gpu("después del experimento")

    resumen = copy.deepcopy(resumen)
    resumen["configuracion"] = nombre_config
    resumen["img_size"] = config_local["img_size"]
    resumen["batch_size"] = config_local["batch_size"]
    resumen["augmentation"] = config_local["augmentation"]
    resumen["epochs_head"] = config_local["epochs_head"]
    resumen["epochs_fine"] = config_local["epochs_fine"]
    resumen["lr_head"] = config_local["lr_head"]
    resumen["lr_fine"] = config_local["lr_fine"]
    resumen["weight_decay"] = config_local["weight_decay"]
    resumen["paciencia"] = config_local["paciencia"]
    resumen["device_usado_celda_11"] = str(DEVICE)
    resumen["cuda_disponible"] = torch.cuda.is_available()
    resumen["gpu_nombre"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

    hist_df = hist_df.copy()
    hist_df["modelo"] = nombre_modelo
    hist_df["configuracion"] = nombre_config
    hist_df["img_size"] = config_local["img_size"]
    hist_df["batch_size"] = config_local["batch_size"]
    hist_df["augmentation"] = config_local["augmentation"]
    hist_df["device_usado_celda_11"] = str(DEVICE)

    resultados_experimentos.append(resumen)
    historiales_experimentos.append(hist_df)

    actualizar_top_candidatos(
        model=model,
        resumen=resumen,
        config=config_local,
        nombre_config=nombre_config
    )

    del model
    liberar_memoria()
    mostrar_memoria_gpu("después de liberar memoria")


# ------------------------------------------------------------
# 7. Ejecutar comparación de arquitecturas y configuraciones
# ------------------------------------------------------------

total_experimentos = len(modelos_arquitectura) * len(CONFIGURACIONES_A_CORRER)
contador = 0

print("\nTotal de experimentos a ejecutar:", total_experimentos)

for nombre_config, config in CONFIGURACIONES_A_CORRER.items():
    for nombre_modelo in modelos_arquitectura:
        contador += 1

        print("\n" + "#" * 100)
        print(f"Experimento {contador}/{total_experimentos}")
        print("Modelo:", nombre_modelo)
        print("Configuración:", nombre_config)
        print("DEVICE:", DEVICE)
        print("#" * 100)

        try:
            correr_experimento_arquitectura(
                nombre_modelo=nombre_modelo,
                nombre_config=nombre_config,
                config=config
            )

        except RuntimeError as e:
            mensaje_error = str(e)

            errores_experimentos.append({
                "modelo": nombre_modelo,
                "configuracion": nombre_config,
                "tipo_error": "RuntimeError",
                "mensaje": mensaje_error
            })

            print("\nERROR en experimento:")
            print("Modelo:", nombre_modelo)
            print("Configuración:", nombre_config)
            print(mensaje_error)

            if "out of memory" in mensaje_error.lower():
                print("\nParece error de memoria GPU.")
                print("Solución: baja batch_size de esa configuración a 32 o 16.")

            liberar_memoria()

        except Exception as e:
            mensaje_error = str(e)

            errores_experimentos.append({
                "modelo": nombre_modelo,
                "configuracion": nombre_config,
                "tipo_error": type(e).__name__,
                "mensaje": mensaje_error
            })

            print("\nERROR en experimento:")
            print("Modelo:", nombre_modelo)
            print("Configuración:", nombre_config)
            print(type(e).__name__, mensaje_error)

            liberar_memoria()

# ------------------------------------------------------------
# 8. Validar que sí haya resultados
# ------------------------------------------------------------

if len(resultados_experimentos) == 0:
    raise ValueError(
        "No se pudo completar ningún experimento. Revisa los errores impresos arriba."
    )

if len(candidatos_top) == 0:
    raise ValueError(
        "No se generaron candidatos. Revisa si todos los experimentos fallaron."
    )

# ------------------------------------------------------------
# 9. Consolidar resultados
# ------------------------------------------------------------

resultados_experimentos_cnn_df = pd.DataFrame(resultados_experimentos)

historial_experimentos_cnn_df = pd.concat(
    historiales_experimentos,
    ignore_index=True
)

resultados_experimentos_cnn_df = resultados_experimentos_cnn_df.sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

print("\nResultados comparativos de arquitecturas CNN:")
display(resultados_experimentos_cnn_df)

# ------------------------------------------------------------
# 10. Guardar resultados
# ------------------------------------------------------------

resultados_csv = SALIDA_DIR / "resultados_comparacion_arquitecturas_cnn.csv"
resultados_xlsx = SALIDA_DIR / "resultados_comparacion_arquitecturas_cnn.xlsx"

historial_csv = SALIDA_DIR / "historial_comparacion_arquitecturas_cnn.csv"
historial_xlsx = SALIDA_DIR / "historial_comparacion_arquitecturas_cnn.xlsx"

resultados_experimentos_cnn_df.to_csv(resultados_csv, index=False, encoding="utf-8-sig")
resultados_experimentos_cnn_df.to_excel(resultados_xlsx, index=False)

historial_experimentos_cnn_df.to_csv(historial_csv, index=False, encoding="utf-8-sig")
historial_experimentos_cnn_df.to_excel(historial_xlsx, index=False)

print("\nArchivos de resultados guardados:")
print(resultados_csv)
print(resultados_xlsx)
print(historial_csv)
print(historial_xlsx)

# ------------------------------------------------------------
# 11. Guardar errores, si existieron
# ------------------------------------------------------------

if len(errores_experimentos) > 0:
    errores_experimentos_df = pd.DataFrame(errores_experimentos)

    errores_csv = SALIDA_DIR / "errores_experimentos_cnn.csv"
    errores_xlsx = SALIDA_DIR / "errores_experimentos_cnn.xlsx"

    errores_experimentos_df.to_csv(errores_csv, index=False, encoding="utf-8-sig")
    errores_experimentos_df.to_excel(errores_xlsx, index=False)

    print("\nAlgunos experimentos fallaron, pero la celda continuó.")
    print("Errores guardados en:")
    print(errores_csv)
    print(errores_xlsx)

    display(errores_experimentos_df)

# ------------------------------------------------------------
# 12. Tabla de hiperparámetros usados
# ------------------------------------------------------------

filas_hiperparametros = []

for nombre_config, config in CONFIGURACIONES_A_CORRER.items():
    for hiperparametro, valor in config.items():
        filas_hiperparametros.append({
            "configuracion": nombre_config,
            "hiperparametro": hiperparametro,
            "valor": valor
        })

tabla_hiperparametros_cnn = pd.DataFrame(filas_hiperparametros)

explicaciones = {
    "img_size": "Tamaño al que se redimensionan las imágenes antes de entrar a la CNN.",
    "batch_size": "Cantidad de imágenes procesadas en cada paso de entrenamiento.",
    "augmentation": "Estrategia de transformación de imágenes durante entrenamiento.",
    "epochs_head": "Épocas para entrenar la cabeza clasificadora del modelo.",
    "epochs_fine": "Épocas para ajustar las últimas capas del modelo preentrenado.",
    "lr_head": "Tasa de aprendizaje usada al entrenar la cabeza clasificadora.",
    "lr_fine": "Tasa de aprendizaje usada en el fine-tuning.",
    "weight_decay": "Regularización para reducir sobreajuste.",
    "paciencia": "Número de épocas sin mejora antes de detener el entrenamiento."
}

tabla_hiperparametros_cnn["explicacion"] = tabla_hiperparametros_cnn["hiperparametro"].map(explicaciones)

hiperparametros_csv = SALIDA_DIR / "hiperparametros_cnn_rubrica.csv"
hiperparametros_xlsx = SALIDA_DIR / "hiperparametros_cnn_rubrica.xlsx"

tabla_hiperparametros_cnn.to_csv(hiperparametros_csv, index=False, encoding="utf-8-sig")
tabla_hiperparametros_cnn.to_excel(hiperparametros_xlsx, index=False)

print("\nHiperparámetros usados:")
display(tabla_hiperparametros_cnn)

# ------------------------------------------------------------
# 13. Tabla de ventajas y desventajas por arquitectura
# ------------------------------------------------------------

tabla_arquitecturas_cnn = pd.DataFrame([
    {
        "modelo": "cnn_simple",
        "tipo": "CNN desde cero",
        "ventaja": "Es fácil de explicar y sirve como línea base.",
        "desventaja": "Tiene menos conocimiento previo y puede rendir peor con pocas imágenes."
    },
    {
        "modelo": "mobilenet_v2",
        "tipo": "CNN preentrenada liviana",
        "ventaja": "Es rápida y consume pocos recursos.",
        "desventaja": "Puede capturar menos detalles que modelos más complejos."
    },
    {
        "modelo": "resnet18",
        "tipo": "CNN preentrenada intermedia",
        "ventaja": "Equilibra rendimiento y costo computacional.",
        "desventaja": "Puede no captar detalles finos si la diferencia entre clases es muy sutil."
    },
    {
        "modelo": "efficientnet_b0",
        "tipo": "CNN preentrenada eficiente",
        "ventaja": "Suele funcionar bien con buena relación entre precisión y costo.",
        "desventaja": "Puede requerir ajuste fino cuidadoso para no sobreajustarse."
    },
    {
        "modelo": "convnext_tiny",
        "tipo": "CNN moderna preentrenada",
        "ventaja": "Modelo moderno con buena capacidad de representación.",
        "desventaja": "Es más pesado y puede demorar más en entrenamiento."
    }
])

tabla_arquitecturas_cnn = tabla_arquitecturas_cnn[
    tabla_arquitecturas_cnn["modelo"].isin(modelos_arquitectura)
].copy()

arquitecturas_csv = SALIDA_DIR / "ventajas_desventajas_arquitecturas_cnn.csv"
arquitecturas_xlsx = SALIDA_DIR / "ventajas_desventajas_arquitecturas_cnn.xlsx"

tabla_arquitecturas_cnn.to_csv(arquitecturas_csv, index=False, encoding="utf-8-sig")
tabla_arquitecturas_cnn.to_excel(arquitecturas_xlsx, index=False)

print("\nVentajas y desventajas por arquitectura:")
display(tabla_arquitecturas_cnn)

# ------------------------------------------------------------
# 14. Guardar TOP candidatos para evaluación de estabilidad
# ------------------------------------------------------------

candidatos_top = sorted(
    candidatos_top,
    key=lambda x: x["f1_macro"],
    reverse=True
)

metadata_candidatos = []

for i, candidato in enumerate(candidatos_top, start=1):
    modelo = candidato["modelo"]
    configuracion = candidato["configuracion"]

    nombre_seguro = limpiar_nombre_archivo(
        f"candidato_top_{i:02d}_{modelo}_{configuracion}"
    )

    checkpoint_candidato_path = CHECKPOINTS_CANDIDATOS_DIR / f"{nombre_seguro}.pt"

    torch.save(
        {
            "ranking_test": i,
            "modelo": modelo,
            "configuracion": configuracion,
            "config": candidato["config"],
            "state_dict": candidato["state_dict"],
            "resumen_test": candidato["resumen"],
            "label_to_idx": label_to_idx,
            "idx_to_label": idx_to_label
        },
        checkpoint_candidato_path
    )

    fila = copy.deepcopy(candidato["resumen"])
    fila["ranking_test"] = i
    fila["modelo"] = modelo
    fila["configuracion"] = configuracion
    fila["checkpoint_path"] = str(checkpoint_candidato_path)
    fila["nota"] = "Candidato para validación final. No es necesariamente el modelo final."

    metadata_candidatos.append(fila)

candidatos_top_df = pd.DataFrame(metadata_candidatos)

candidatos_csv = SALIDA_DIR / "top_candidatos_cnn_para_validacion.csv"
candidatos_xlsx = SALIDA_DIR / "top_candidatos_cnn_para_validacion.xlsx"

candidatos_top_df.to_csv(candidatos_csv, index=False, encoding="utf-8-sig")
candidatos_top_df.to_excel(candidatos_xlsx, index=False)

print("\nTOP candidatos guardados para validación final:")
display(candidatos_top_df)

print("\nArchivos de candidatos:")
print(candidatos_csv)
print(candidatos_xlsx)

print("\nCheckpoints de candidatos guardados en:")
print(CHECKPOINTS_CANDIDATOS_DIR)

# ------------------------------------------------------------
# 15. Guardar checkpoint del mejor por test
# ------------------------------------------------------------
# Este NO debe interpretarse como ganador final.
# Solo es el mejor candidato preliminar según test.
# La Celda 12 debe elegir el modelo final por estabilidad en validación.
# ------------------------------------------------------------

modelo_ganador_test_nombre = mejor_global_test["modelo"]
config_ganadora_test_nombre = mejor_global_test["configuracion"]
config_ganadora_test = mejor_global_test["config"]

checkpoint_path = SALIDA_DIR / "modelo_ganador_experimentos_cnn.pt"

torch.save(
    {
        "modelo": modelo_ganador_test_nombre,
        "configuracion": config_ganadora_test_nombre,
        "config": config_ganadora_test,
        "state_dict": mejor_global_test["state_dict"],
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "resultados_experimentos": resultados_experimentos_cnn_df.to_dict(orient="records"),
        "hiperparametros": tabla_hiperparametros_cnn.to_dict(orient="records"),
        "arquitecturas": tabla_arquitecturas_cnn.to_dict(orient="records"),
        "errores_experimentos": errores_experimentos,
        "top_candidatos": candidatos_top_df.to_dict(orient="records"),
        "criterio": "Mejor F1 macro en test. Selección preliminar, no definitiva."
    },
    checkpoint_path
)

print("\nMejor candidato preliminar según TEST:")
print("Modelo:", modelo_ganador_test_nombre)
print("Configuración:", config_ganadora_test_nombre)
print("F1 macro test:", round(mejor_global_test["f1_macro"], 4))

print("\nCheckpoint preliminar guardado en:")
print(checkpoint_path)

print("\nIMPORTANTE:")
print("La selección final NO debe hacerse solo con este resultado de test.")
print("Ahora la CELDA 12 debe evaluar los TOP candidatos en validación")
print("y elegir el modelo más estable.")

VERIFICACIÓN ANTES DE ENTRENAR
PyTorch: 2.7.1+cu128
CUDA disponible: True
CUDA PyTorch: 12.8
DEVICE: cuda
USAR_GPU_REAL: True
USAR_AMP: True
GPU detectada: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Memoria total GPU: 11.94 GB

GPU memoria al inicio
Memoria usada: 0.016 GB
Memoria reservada: 0.25 GB

Configuraciones a evaluar:

- config_base_224_limpia
{'img_size': 224, 'batch_size': 64, 'augmentation': 'limpia', 'epochs_head': 5, 'epochs_fine': 12, 'lr_head': 0.001, 'lr_fine': 1e-05, 'weight_decay': 0.0001, 'paciencia': 4}

- config_2_224_color
{'img_size': 224, 'batch_size': 32, 'augmentation': 'base_color', 'epochs_head': 6, 'epochs_fine': 15, 'lr_head': 0.0008, 'lr_fine': 8e-06, 'weight_decay': 0.0001, 'paciencia': 5}

- config_3_256_crop
{'img_size': 256, 'batch_size': 32, 'augmentation': 'crop_suave', 'epochs_head': 5, 'epochs_fine': 14, 'lr_head': 0.001, 'lr_fine': 5e-06, 'weight_decay': 5e-05, 'paciencia': 5}

- config_4_192_limpia
{'img_size': 192, 'batch_size': 64, 'augmentation':

C:\Users\andre\AppData\Local\Temp\ipykernel_37524\845020441.py:890: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USAR_AMP)
C:\Users\andre\AppData\Local\Temp\ipykernel_37524\845020441.py:927: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USAR_AMP):
C:\Users\andre\AppData\Local\Temp\ipykernel_37524\845020441.py:779: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USAR_AMP):


ARQ_cnn_simple_config_base_224_limpia | cnn_simple | head | Epoch 01/5 | train F1 0.749 | test F1 0.352
ARQ_cnn_simple_config_base_224_limpia | cnn_simple | head | Epoch 02/5 | train F1 0.780 | test F1 0.684
ARQ_cnn_simple_config_base_224_limpia | cnn_simple | head | Epoch 03/5 | train F1 0.795 | test F1 0.803
ARQ_cnn_simple_config_base_224_limpia | cnn_simple | head | Epoch 04/5 | train F1 0.797 | test F1 0.771
ARQ_cnn_simple_config_base_224_limpia | cnn_simple | head | Epoch 05/5 | train F1 0.791 | test F1 0.725

Device del modelo devuelto: cuda:0

GPU memoria después del experimento
Memoria usada: 0.019 GB
Memoria reservada: 1.992 GB

Nuevo mejor candidato por TEST hasta ahora
Modelo: cnn_simple
Configuración: config_base_224_limpia
F1 macro test: 0.8031

GPU memoria después de liberar memoria
Memoria usada: 0.016 GB
Memoria reservada: 0.25 GB

####################################################################################################
Experimento 2/25
Modelo: mobilenet_v2
C

C:\Users\andre\AppData\Local\Temp\ipykernel_37524\845020441.py:890: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USAR_AMP)
C:\Users\andre\AppData\Local\Temp\ipykernel_37524\845020441.py:927: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USAR_AMP):
C:\Users\andre\AppData\Local\Temp\ipykernel_37524\845020441.py:779: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USAR_AMP):


ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | head | Epoch 01/5 | train F1 0.623 | test F1 0.676
ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | head | Epoch 02/5 | train F1 0.723 | test F1 0.711
ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | head | Epoch 03/5 | train F1 0.740 | test F1 0.719
ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | head | Epoch 04/5 | train F1 0.766 | test F1 0.663
ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | head | Epoch 05/5 | train F1 0.774 | test F1 0.745

Fase: fine_tuning
Parámetros entrenables: 1528642
ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | fine_tuning | Epoch 01/12 | train F1 0.769 | test F1 0.737
ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | fine_tuning | Epoch 02/12 | train F1 0.785 | test F1 0.742
ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | fine_tuning | Epoch 03/12 | train F1 0.794 | test F1 0.749
ARQ_mobilenet_v2_config_base_224_limpia | mobilenet_v2 | fine

...


Ventajas y desventajas por arquitectura:


,modelo,tipo,ventaja,desventaja
0,cnn_simple,CNN desde cero,Es fácil de explicar y sirve como línea base.,Tiene menos conocimiento previo y puede rendir...
1,mobilenet_v2,CNN preentrenada liviana,Es rápida y consume pocos recursos.,Puede capturar menos detalles que modelos más ...
2,resnet18,CNN preentrenada intermedia,Equilibra rendimiento y costo computacional.,Puede no captar detalles finos si la diferenci...
3,efficientnet_b0,CNN preentrenada eficiente,Suele funcionar bien con buena relación entre ...,Puede requerir ajuste fino cuidadoso para no s...
4,convnext_tiny,CNN moderna preentrenada,Modelo moderno con buena capacidad de represen...,Es más pesado y puede demorar más en entrenami...



TOP candidatos guardados para validación final:


,experimento_id,etapa,modelo,img_size,batch_size,augmentation,epochs_head,epochs_fine,lr_head,lr_fine,...,recall_macro,f1_macro,configuracion,paciencia,device_usado_celda_11,cuda_disponible,gpu_nombre,ranking_test,checkpoint_path,nota
0,ARQ_cnn_simple_config_5_256_monstruo_controlado,comparacion_arquitectura_config,cnn_simple,256,64,crop_suave,8,25,0.0008,0.000005,...,0.830889,0.823989,config_5_256_monstruo_controlado,5,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,1,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...
1,ARQ_cnn_simple_config_base_224_limpia,comparacion_arquitectura_config,cnn_simple,224,64,limpia,5,12,0.0010,0.000010,...,0.804890,0.803089,config_base_224_limpia,4,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,2,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...
2,ARQ_cnn_simple_config_4_192_limpia,comparacion_arquitectura_config,cnn_simple,192,64,limpia,4,10,0.0010,0.000020,...,0.802241,0.795971,config_4_192_limpia,4,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,3,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...
3,ARQ_convnext_tiny_config_4_192_limpia,comparacion_arquitectura_config,convnext_tiny,192,64,limpia,4,10,0.0010,0.000020,...,0.790065,0.790065,config_4_192_limpia,4,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,4,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...
4,ARQ_cnn_simple_config_3_256_crop,comparacion_arquitectura_config,cnn_simple,256,32,crop_suave,5,14,0.0010,0.000005,...,0.798043,0.787834,config_3_256_crop,5,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,5,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...



Archivos de candidatos:
salida_airbnb_fotos\top_candidatos_cnn_para_validacion.csv
salida_airbnb_fotos\top_candidatos_cnn_para_validacion.xlsx

Checkpoints de candidatos guardados en:
salida_airbnb_fotos\checkpoints_candidatos_cnn

Mejor candidato preliminar según TEST:
Modelo: cnn_simple
Configuración: config_5_256_monstruo_controlado
F1 macro test: 0.824

Checkpoint preliminar guardado en:
salida_airbnb_fotos\modelo_ganador_experimentos_cnn.pt

IMPORTANTE:
La selección final NO debe hacerse solo con este resultado de test.
Ahora la CELDA 12 debe evaluar los TOP candidatos en validación
y elegir el modelo más estable.


### Celda 12 — Comprobación final

Después de elegir el mejor modelo en la celda 11, esta celda lo evalúa en `validacion`. Este resultado se puede reportar como la comprobación final del módulo CNN.


In [15]:
# ============================================================
# CELDA 12 - Selección final del modelo más estable
# VERSIÓN ULTRA SEGURA
# ============================================================
# Esta celda evalúa en VALIDACIÓN los TOP candidatos guardados
# por la CELDA 11.
#
# Cambios para evitar caída del kernel:
# - Evalúa un candidato a la vez.
# - No guarda modelos completos en memoria.
# - No reconstruye el modelo final al final.
# - No genera predicciones masivas aquí.
# - Usa batch pequeño.
# - Por defecto fuerza CPU para máxima estabilidad.
#
# Después de esta celda, la CELDA 12B usará:
# modelo_final_estable_cnn.pt
# ============================================================

import gc
import copy
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 0. Configuración segura
# ------------------------------------------------------------
FORZAR_CPU_EN_VALIDACION = True
BATCH_VALIDACION_SEGURO = 2

if FORZAR_CPU_EN_VALIDACION:
    device = torch.device("cpu")
    print("Validación forzada en CPU para evitar caída del kernel.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Validación usando:", device)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 1. Rutas necesarias
# ------------------------------------------------------------
candidatos_csv = SALIDA_DIR / "top_candidatos_cnn_para_validacion.csv"
checkpoints_dir = SALIDA_DIR / "checkpoints_candidatos_cnn"

if not candidatos_csv.exists():
    raise FileNotFoundError(
        f"No encontré el archivo de candidatos:\n{candidatos_csv}\n\n"
        "Primero ejecuta la CELDA 11 corregida."
    )

if not checkpoints_dir.exists():
    raise FileNotFoundError(
        f"No encontré la carpeta de checkpoints:\n{checkpoints_dir}\n\n"
        "Primero ejecuta la CELDA 11 corregida."
    )

df_candidatos = pd.read_csv(candidatos_csv)

if "checkpoint_path" not in df_candidatos.columns:
    raise ValueError(
        "El archivo de candidatos no tiene la columna checkpoint_path. "
        "Vuelve a ejecutar la CELDA 11 corregida."
    )

print("Candidatos encontrados:")
display(df_candidatos)

# ------------------------------------------------------------
# 2. Funciones auxiliares
# ------------------------------------------------------------
def limpiar_memoria():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def cargar_checkpoint_seguro(path_checkpoint):
    try:
        return torch.load(path_checkpoint, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path_checkpoint, map_location="cpu")



def evaluar_modelo_validacion_seguro(model, loader, split_name="validacion"):
    """
    Evalua sin depender del criterion/device global de la celda 10/11.
    Evita mezclar tensores CPU y CUDA.
    """

    if loader is None:
        raise ValueError(f"No existe loader para el split: {split_name}")

    model.eval()

    peso_local = None
    try:
        if "class_weights" in globals() and class_weights is not None:
            peso_local = class_weights.detach().to(device)
    except Exception:
        peso_local = None

    criterion_local = nn.CrossEntropyLoss(weight=peso_local).to(device)

    y_true = []
    y_pred = []
    running_loss = 0.0

    usar_amp_local = bool(device.type == "cuda" and globals().get("USAR_AMP", False))

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=(device.type == "cuda"))
            labels = labels.to(device, non_blocking=(device.type == "cuda"))

            if usar_amp_local:
                with torch.amp.autocast("cuda"):
                    outputs = model(images)
                    loss = criterion_local(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion_local(outputs, labels)

            preds = torch.argmax(outputs, dim=1)

            running_loss += loss.item() * images.size(0)
            y_true.extend(labels.detach().cpu().numpy())
            y_pred.extend(preds.detach().cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)

    return {
        "split": split_name,
        "loss": avg_loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred
    }

def obtener_f1_test(checkpoint, fila_candidato):
    if "resumen_test" in checkpoint:
        resumen_test = checkpoint["resumen_test"]
        if isinstance(resumen_test, dict) and "f1_macro" in resumen_test:
            return float(resumen_test["f1_macro"])

    if "f1_macro" in fila_candidato:
        return float(fila_candidato["f1_macro"])

    return None


def matriz_confusion_segura(y_true, y_pred):
    try:
        return confusion_matrix(y_true, y_pred)
    except Exception:
        clases = sorted(list(set(list(y_true) + list(y_pred))))
        pos = {c: i for i, c in enumerate(clases)}
        cm = [[0 for _ in clases] for _ in clases]

        for real, pred in zip(y_true, y_pred):
            cm[pos[real]][pos[pred]] += 1

        return pd.DataFrame(cm, index=clases, columns=clases).values


def recall_minimo_desde_cm(cm):
    recalls = []

    for i in range(cm.shape[0]):
        total_real = cm[i, :].sum()

        if total_real == 0:
            recalls.append(0)
        else:
            recalls.append(cm[i, i] / total_real)

    if len(recalls) == 0:
        return 0

    return min(recalls)


def evaluar_un_candidato(fila):
    path_checkpoint = Path(str(fila["checkpoint_path"]))

    if not path_checkpoint.exists():
        raise FileNotFoundError(f"No encontré el checkpoint: {path_checkpoint}")

    checkpoint = cargar_checkpoint_seguro(path_checkpoint)

    modelo_nombre = checkpoint["modelo"]
    configuracion_nombre = checkpoint["configuracion"]
    config_original = checkpoint["config"]
    f1_test = obtener_f1_test(checkpoint, fila)

    config_validacion = copy.deepcopy(config_original)
    config_validacion["batch_size"] = min(
        int(config_original.get("batch_size", BATCH_VALIDACION_SEGURO)),
        BATCH_VALIDACION_SEGURO
    )

    print("\n" + "=" * 80)
    print("Evaluando candidato")
    print("Modelo:", modelo_nombre)
    print("Configuración:", configuracion_nombre)
    print("F1 macro test:", f1_test)
    print("Batch validación:", config_validacion["batch_size"])
    print("=" * 80)

    limpiar_memoria()

    model = crear_modelo(modelo_nombre)
    model.load_state_dict(checkpoint["state_dict"])
    model = model.to(device)
    model.eval()

    _, _, validacion_loader = crear_loaders(config_validacion)

    metricas_validacion = evaluar_modelo_validacion_seguro(
        model=model,
        loader=validacion_loader,
        split_name="validacion"
    )

    f1_validacion = float(metricas_validacion["f1_macro"])

    if f1_test is None:
        brecha_test_validacion = None
        puntaje_estabilidad = f1_validacion
    else:
        brecha_test_validacion = abs(float(f1_test) - f1_validacion)

        cm = matriz_confusion_segura(
            metricas_validacion["y_true"],
            metricas_validacion["y_pred"]
        )

        recall_minimo = recall_minimo_desde_cm(cm)

        puntaje_estabilidad = (
            0.70 * f1_validacion
            + 0.20 * (1 - brecha_test_validacion)
            + 0.10 * recall_minimo
        )

    cm = matriz_confusion_segura(
        metricas_validacion["y_true"],
        metricas_validacion["y_pred"]
    )

    recall_minimo = recall_minimo_desde_cm(cm)

    fila_resultado = {
        "ranking_test": fila.get("ranking_test", None),
        "modelo": modelo_nombre,
        "configuracion": configuracion_nombre,
        "checkpoint_path": str(path_checkpoint),

        "f1_macro_test": f1_test,
        "f1_macro_validacion": f1_validacion,
        "brecha_abs_test_validacion": brecha_test_validacion,

        "accuracy_validacion": float(metricas_validacion["accuracy"]),
        "precision_macro_validacion": float(metricas_validacion["precision_macro"]),
        "recall_macro_validacion": float(metricas_validacion["recall_macro"]),
        "loss_validacion": float(metricas_validacion["loss"]),

        "recall_minimo_clase_validacion": recall_minimo,
        "puntaje_estabilidad": puntaje_estabilidad,

        "img_size": config_original.get("img_size"),
        "batch_size_entrenamiento": config_original.get("batch_size"),
        "batch_size_validacion": config_validacion.get("batch_size"),
        "augmentation": config_original.get("augmentation"),
        "epochs_head": config_original.get("epochs_head"),
        "epochs_fine": config_original.get("epochs_fine"),
        "lr_head": config_original.get("lr_head"),
        "lr_fine": config_original.get("lr_fine"),
        "weight_decay": config_original.get("weight_decay"),
        "paciencia": config_original.get("paciencia"),

        "matriz_confusion": str(cm.tolist())
    }

    del model
    del checkpoint
    del validacion_loader

    limpiar_memoria()

    return fila_resultado


# ------------------------------------------------------------
# 3. Evaluar candidatos uno por uno
# ------------------------------------------------------------
resultados_validacion = []
errores_validacion = []

for i, fila in df_candidatos.iterrows():
    try:
        resultado = evaluar_un_candidato(fila)
        resultados_validacion.append(resultado)

    except Exception as e:
        error = {
            "modelo": fila.get("modelo", ""),
            "configuracion": fila.get("configuracion", ""),
            "checkpoint_path": fila.get("checkpoint_path", ""),
            "tipo_error": type(e).__name__,
            "mensaje": str(e)
        }

        errores_validacion.append(error)

        print("\nError evaluando candidato:")
        print(error)

        limpiar_memoria()

if len(resultados_validacion) == 0:
    raise ValueError(
        "No se pudo evaluar ningún candidato en validación. "
        "Revisa los errores impresos arriba."
    )

# ------------------------------------------------------------
# 4. Comparación de estabilidad
# ------------------------------------------------------------
resultados_validacion_df = pd.DataFrame(resultados_validacion)

resultados_validacion_df = resultados_validacion_df.sort_values(
    ["puntaje_estabilidad", "f1_macro_validacion", "brecha_abs_test_validacion"],
    ascending=[False, False, True]
).reset_index(drop=True)

resultados_validacion_df["ranking_estabilidad"] = resultados_validacion_df.index + 1

print("\nComparación de candidatos en validación:")
display(resultados_validacion_df)

metricas_candidatos_csv = SALIDA_DIR / "metricas_validacion_top_candidatos_cnn.csv"
metricas_candidatos_xlsx = SALIDA_DIR / "metricas_validacion_top_candidatos_cnn.xlsx"

resultados_validacion_df.to_csv(metricas_candidatos_csv, index=False, encoding="utf-8-sig")
resultados_validacion_df.to_excel(metricas_candidatos_xlsx, index=False)

print("\nMétricas de candidatos guardadas:")
print(metricas_candidatos_csv)
print(metricas_candidatos_xlsx)

# ------------------------------------------------------------
# 5. Guardar errores si existieron
# ------------------------------------------------------------
if len(errores_validacion) > 0:
    errores_validacion_df = pd.DataFrame(errores_validacion)

    errores_validacion_csv = SALIDA_DIR / "errores_validacion_top_candidatos_cnn.csv"
    errores_validacion_xlsx = SALIDA_DIR / "errores_validacion_top_candidatos_cnn.xlsx"

    errores_validacion_df.to_csv(errores_validacion_csv, index=False, encoding="utf-8-sig")
    errores_validacion_df.to_excel(errores_validacion_xlsx, index=False)

    print("\nAlgunos candidatos fallaron en validación:")
    display(errores_validacion_df)

# ------------------------------------------------------------
# 6. Seleccionar modelo final estable
# ------------------------------------------------------------
mejor_fila = resultados_validacion_df.iloc[0]

modelo_final_nombre = mejor_fila["modelo"]
config_final_nombre = mejor_fila["configuracion"]
checkpoint_final_original = Path(str(mejor_fila["checkpoint_path"]))

print("\n" + "=" * 80)
print("MODELO FINAL MÁS ESTABLE")
print("=" * 80)
print("Modelo:", modelo_final_nombre)
print("Configuración:", config_final_nombre)
print("F1 macro test:", mejor_fila["f1_macro_test"])
print("F1 macro validación:", mejor_fila["f1_macro_validacion"])
print("Brecha test-validación:", mejor_fila["brecha_abs_test_validacion"])
print("Recall mínimo por clase:", mejor_fila["recall_minimo_clase_validacion"])
print("Puntaje estabilidad:", mejor_fila["puntaje_estabilidad"])

display(pd.DataFrame([mejor_fila]))

# ------------------------------------------------------------
# 7. Crear checkpoint final estable
# ------------------------------------------------------------
checkpoint_original = cargar_checkpoint_seguro(checkpoint_final_original)

checkpoint_final_estable_path = SALIDA_DIR / "modelo_final_estable_cnn.pt"

torch.save(
    {
        "modelo": checkpoint_original["modelo"],
        "configuracion": checkpoint_original["configuracion"],
        "config": checkpoint_original["config"],
        "config_validacion": {
            **checkpoint_original["config"],
            "batch_size": BATCH_VALIDACION_SEGURO
        },
        "state_dict": checkpoint_original["state_dict"],
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "resultado_estabilidad": mejor_fila.to_dict(),
        "resultados_validacion_candidatos": resultados_validacion_df.to_dict(orient="records"),
        "criterio_seleccion": (
            "Modelo elegido por estabilidad: combina F1 macro en validación, "
            "brecha entre test y validación y balance mínimo por clase."
        )
    },
    checkpoint_final_estable_path
)

del checkpoint_original
limpiar_memoria()

print("\nCheckpoint del modelo final estable guardado en:")
print(checkpoint_final_estable_path)

# ------------------------------------------------------------
# 8. Guardar matriz de confusión como texto y tabla simple
# ------------------------------------------------------------
matriz_final_txt = str(mejor_fila["matriz_confusion"])

matriz_final_path = SALIDA_DIR / "matriz_confusion_modelo_final_estable.txt"

with open(matriz_final_path, "w", encoding="utf-8") as f:
    f.write(matriz_final_txt)

print("\nMatriz de confusión guardada en:")
print(matriz_final_path)

print("\nCELDA 12 terminada correctamente.")
print("Ahora ejecuta la CELDA 12B para aplicar el modelo final estable a todas las fotos.")

Validación forzada en CPU para evitar caída del kernel.
Candidatos encontrados:


,experimento_id,etapa,modelo,img_size,batch_size,augmentation,epochs_head,epochs_fine,lr_head,lr_fine,...,recall_macro,f1_macro,configuracion,paciencia,device_usado_celda_11,cuda_disponible,gpu_nombre,ranking_test,checkpoint_path,nota
0,ARQ_cnn_simple_config_5_256_monstruo_controlado,comparacion_arquitectura_config,cnn_simple,256,64,crop_suave,8,25,0.0008,0.000005,...,0.830889,0.823989,config_5_256_monstruo_controlado,5,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,1,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...
1,ARQ_cnn_simple_config_base_224_limpia,comparacion_arquitectura_config,cnn_simple,224,64,limpia,5,12,0.0010,0.000010,...,0.804890,0.803089,config_base_224_limpia,4,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,2,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...
2,ARQ_cnn_simple_config_4_192_limpia,comparacion_arquitectura_config,cnn_simple,192,64,limpia,4,10,0.0010,0.000020,...,0.802241,0.795971,config_4_192_limpia,4,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,3,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...
3,ARQ_convnext_tiny_config_4_192_limpia,comparacion_arquitectura_config,convnext_tiny,192,64,limpia,4,10,0.0010,0.000020,...,0.790065,0.790065,config_4_192_limpia,4,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,4,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...
4,ARQ_cnn_simple_config_3_256_crop,comparacion_arquitectura_config,cnn_simple,256,32,crop_suave,5,14,0.0010,0.000005,...,0.798043,0.787834,config_3_256_crop,5,cuda,True,NVIDIA GeForce RTX 5070 Ti Laptop GPU,5,salida_airbnb_fotos\checkpoints_candidatos_cnn...,Candidato para validación final. No es necesar...



Evaluando candidato
Modelo: cnn_simple
Configuración: config_5_256_monstruo_controlado
F1 macro test: 0.8239887352790578
Batch validación: 2

Evaluando candidato
Modelo: cnn_simple
Configuración: config_base_224_limpia
F1 macro test: 0.8030894857822573
Batch validación: 2

Evaluando candidato
Modelo: cnn_simple
Configuración: config_4_192_limpia
F1 macro test: 0.7959706197692467
Batch validación: 2

Evaluando candidato
Modelo: convnext_tiny
Configuración: config_4_192_limpia
F1 macro test: 0.7900652412634843
Batch validación: 2
ConvNeXt Tiny con pesos preentrenados.

Evaluando candidato
Modelo: cnn_simple
Configuración: config_3_256_crop
F1 macro test: 0.7878336615906871
Batch validación: 2

Comparación de candidatos en validación:


,ranking_test,modelo,configuracion,checkpoint_path,f1_macro_test,f1_macro_validacion,brecha_abs_test_validacion,accuracy_validacion,precision_macro_validacion,recall_macro_validacion,...,batch_size_validacion,augmentation,epochs_head,epochs_fine,lr_head,lr_fine,weight_decay,paciencia,matriz_confusion,ranking_estabilidad
0,4,convnext_tiny,config_4_192_limpia,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.790065,0.764663,0.025403,0.764706,0.764660,0.764790,...,2,limpia,4,10,0.0010,0.000020,0.00010,4,"[[172, 54], [50, 166]]",1
1,3,cnn_simple,config_4_192_limpia,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.795971,0.766921,0.029050,0.769231,0.785788,0.771878,...,2,limpia,4,10,0.0010,0.000020,0.00010,4,"[[148, 78], [24, 192]]",2
2,2,cnn_simple,config_base_224_limpia,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.803089,0.762763,0.040327,0.764706,0.778559,0.767146,...,2,limpia,5,12,0.0010,0.000010,0.00010,4,"[[149, 77], [27, 189]]",3
3,5,cnn_simple,config_3_256_crop,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.787834,0.762200,0.025634,0.769231,0.815799,0.773517,...,2,crop_suave,5,14,0.0010,0.000005,0.00005,5,"[[132, 94], [8, 208]]",4
4,1,cnn_simple,config_5_256_monstruo_controlado,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.823989,0.757996,0.065993,0.760181,0.775026,0.762721,...,2,crop_suave,8,25,0.0008,0.000005,0.00010,5,"[[147, 79], [27, 189]]",5



Métricas de candidatos guardadas:
salida_airbnb_fotos\metricas_validacion_top_candidatos_cnn.csv
salida_airbnb_fotos\metricas_validacion_top_candidatos_cnn.xlsx

MODELO FINAL MÁS ESTABLE
Modelo: convnext_tiny
Configuración: config_4_192_limpia
F1 macro test: 0.7900652412634843
F1 macro validación: 0.7646625163826999
Brecha test-validación: 0.025402724880784433
Recall mínimo por clase: 0.7610619469026548
Puntaje estabilidad: 0.8062894111819985


,ranking_test,modelo,configuracion,checkpoint_path,f1_macro_test,f1_macro_validacion,brecha_abs_test_validacion,accuracy_validacion,precision_macro_validacion,recall_macro_validacion,...,batch_size_validacion,augmentation,epochs_head,epochs_fine,lr_head,lr_fine,weight_decay,paciencia,matriz_confusion,ranking_estabilidad
0,4,convnext_tiny,config_4_192_limpia,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.790065,0.764663,0.025403,0.764706,0.76466,0.76479,...,2,limpia,4,10,0.001,0.00002,0.0001,4,"[[172, 54], [50, 166]]",1



Checkpoint del modelo final estable guardado en:
salida_airbnb_fotos\modelo_final_estable_cnn.pt

Matriz de confusión guardada en:
salida_airbnb_fotos\matriz_confusion_modelo_final_estable.txt

CELDA 12 terminada correctamente.
Ahora ejecuta la CELDA 12B para aplicar el modelo final estable a todas las fotos.


In [28]:
# ============================================================
# CELDA 12B - Porcentajes finales por Airbnb usando TODAS las fotos
# ============================================================
# Esta celda aplica el MODELO FINAL ESTABLE a todas las fotos disponibles
# de train + test + validacion.
#
# Por cada alojamiento calcula:
# - numero real de imagenes por encima/debajo de la mediana
# - numero predicho de imagenes por encima/debajo de la mediana
# - proporcion real de imagenes por encima/debajo de la mediana
# - proporcion predicha de imagenes por encima/debajo de la mediana
#
# ============================================================

import gc
import copy
import pandas as pd
import torch
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 0. Configuracion de prediccion
# ------------------------------------------------------------
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# predecir_dataframe usa la variable global device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USAR_AMP = bool(device.type == "cuda" and globals().get("USAR_AMP", True))

print("Memoria limpiada antes de predecir todas las fotos.")
print("Dispositivo para prediccion final:", device)
print("Mixed precision en prediccion final:", USAR_AMP)

# ------------------------------------------------------------
# 1. Cargar el modelo final estable
# ------------------------------------------------------------
checkpoint_final_estable_path = SALIDA_DIR / "modelo_final_estable_cnn.pt"

if not checkpoint_final_estable_path.exists():
    raise FileNotFoundError(
        f"No encontre el modelo final estable:\n{checkpoint_final_estable_path}\n\n"
        "Primero debes ejecutar la CELDA 12 para elegir el modelo mas estable."
    )

print("Cargando modelo final estable...")

try:
    checkpoint_final = torch.load(
        checkpoint_final_estable_path,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    checkpoint_final = torch.load(
        checkpoint_final_estable_path,
        map_location="cpu"
    )

modelo_final_nombre = checkpoint_final["modelo"]
config_final_nombre = checkpoint_final["configuracion"]

if "config_validacion" in checkpoint_final:
    config_prediccion_final = copy.deepcopy(checkpoint_final["config_validacion"])
else:
    config_prediccion_final = copy.deepcopy(checkpoint_final["config"])

config_prediccion_final["batch_size"] = min(
    int(config_prediccion_final.get("batch_size", 8)),
    8
)

modelo_final = crear_modelo(modelo_final_nombre)
modelo_final.load_state_dict(checkpoint_final["state_dict"])
modelo_final = modelo_final.to(device)
modelo_final.eval()

print("Modelo final estable cargado correctamente.")
print("Modelo:", modelo_final_nombre)
print("Configuracion:", config_final_nombre)
print("Batch usado para prediccion:", config_prediccion_final["batch_size"])

# ------------------------------------------------------------
# 2. Unir fotos de train, test y validacion
# ------------------------------------------------------------
# Esta version usa solo las fotos que ya entraron al flujo CNN:
# df_train + df_test + df_validacion.
# Por eso el universo coincide con el dataset etiquetado usado por el modelo.
df_train_temp = df_train.copy()
df_train_temp["grupo_origen"] = "train"

df_test_temp = df_test.copy()
df_test_temp["grupo_origen"] = "test"

df_validacion_temp = df_validacion.copy()
df_validacion_temp["grupo_origen"] = "validacion"

df_todas_fotos = pd.concat(
    [df_train_temp, df_test_temp, df_validacion_temp],
    ignore_index=True
)

if ID_COL not in df_todas_fotos.columns:
    raise ValueError(f"No existe la columna ID_COL={ID_COL} en df_todas_fotos.")

if "label_calidad" not in df_todas_fotos.columns:
    raise ValueError("No existe la columna label_calidad en df_todas_fotos.")

if "path_modelo" not in df_todas_fotos.columns:
    raise ValueError("No existe la columna path_modelo en df_todas_fotos.")

df_todas_fotos[ID_COL] = df_todas_fotos[ID_COL].astype(str).str.strip()
df_todas_fotos["path_modelo"] = df_todas_fotos["path_modelo"].astype(str)
df_todas_fotos["label_calidad"] = df_todas_fotos["label_calidad"].astype(str).str.lower().str.strip()

df_todas_fotos = df_todas_fotos[df_todas_fotos["label_calidad"].isin(["alta", "media"])].copy()
df_todas_fotos = df_todas_fotos.drop_duplicates(subset=["path_modelo"]).copy()

df_todas_fotos["existe_imagen_12b"] = df_todas_fotos["path_modelo"].apply(lambda x: Path(str(x)).exists())
df_todas_fotos = df_todas_fotos[df_todas_fotos["existe_imagen_12b"]].copy()

print("\nTotal de fotos para prediccion final:", len(df_todas_fotos))
print("Total de Airbnb con fotos:", df_todas_fotos[ID_COL].nunique())

print("\nDistribucion de fotos usadas:")
display(
    df_todas_fotos
    .groupby(["grupo_origen", "label_calidad"])
    .size()
    .reset_index(name="n_fotos")
)

# ------------------------------------------------------------
# 3. Aplicar el modelo final estable a todas las fotos
# ------------------------------------------------------------
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

df_todas_pred = predecir_dataframe(
    model=modelo_final,
    dataframe=df_todas_fotos,
    config=config_prediccion_final
)

print("\nPredicciones generadas:", df_todas_pred.shape)
display(df_todas_pred.head())

# ------------------------------------------------------------
# 4. Calcular conteos y proporciones por Airbnb
# ------------------------------------------------------------
# En este notebook, label "alta" significa calidad por encima o igual a la mediana.
# label "media" significa calidad por debajo de la mediana.

def conteos_mediana_por_airbnb(df_base, columna_label, prefijo):
    conteo = (
        df_base
        .groupby(ID_COL)[columna_label]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["alta", "media"]:
        if col not in conteo.columns:
            conteo[col] = 0

    conteo[f"n_imagenes_{prefijo}_total"] = conteo["alta"] + conteo["media"]
    conteo[f"n_imagenes_{prefijo}_encima_mediana"] = conteo["alta"]
    conteo[f"n_imagenes_{prefijo}_debajo_mediana"] = conteo["media"]

    conteo[f"prop_imagenes_{prefijo}_encima_mediana"] = (
        conteo[f"n_imagenes_{prefijo}_encima_mediana"]
        / conteo[f"n_imagenes_{prefijo}_total"]
    )

    conteo[f"prop_imagenes_{prefijo}_debajo_mediana"] = (
        conteo[f"n_imagenes_{prefijo}_debajo_mediana"]
        / conteo[f"n_imagenes_{prefijo}_total"]
    )

    return conteo[
        [
            ID_COL,
            f"n_imagenes_{prefijo}_total",
            f"n_imagenes_{prefijo}_encima_mediana",
            f"n_imagenes_{prefijo}_debajo_mediana",
            f"prop_imagenes_{prefijo}_encima_mediana",
            f"prop_imagenes_{prefijo}_debajo_mediana"
        ]
    ].copy()

conteo_pred = conteos_mediana_por_airbnb(
    df_base=df_todas_pred,
    columna_label="label_calidad_modelo_final",
    prefijo="pred"
)

conteo_real = conteos_mediana_por_airbnb(
    df_base=df_todas_pred,
    columna_label="label_calidad",
    prefijo="real"
)

prob_promedio = (
    df_todas_pred
    .groupby(ID_COL)["prob_alta"]
    .mean()
    .reset_index()
    .rename(columns={"prob_alta": "prob_alta_promedio"})
)

proporciones_finales_airbnb = (
    conteo_real
    .merge(conteo_pred, on=ID_COL, how="outer")
    .merge(prob_promedio, on=ID_COL, how="left")
)

proporciones_finales_airbnb["dif_prop_pred_vs_real_encima_mediana"] = (
    proporciones_finales_airbnb["prop_imagenes_pred_encima_mediana"]
    - proporciones_finales_airbnb["prop_imagenes_real_encima_mediana"]
)

# ------------------------------------------------------------
# 5. Ordenar columnas finales
# ------------------------------------------------------------
columnas_finales = [
    ID_COL,
    "n_imagenes_real_total",
    "n_imagenes_real_encima_mediana",
    "n_imagenes_real_debajo_mediana",
    "prop_imagenes_real_encima_mediana",
    "prop_imagenes_real_debajo_mediana",
    "n_imagenes_pred_total",
    "n_imagenes_pred_encima_mediana",
    "n_imagenes_pred_debajo_mediana",
    "prop_imagenes_pred_encima_mediana",
    "prop_imagenes_pred_debajo_mediana",
    "dif_prop_pred_vs_real_encima_mediana",
    "prob_alta_promedio"
]

proporciones_finales_airbnb = proporciones_finales_airbnb[columnas_finales].copy()

columnas_prop = [
    c for c in proporciones_finales_airbnb.columns
    if c.startswith("prop_") or c.startswith("dif_prop_") or c == "prob_alta_promedio"
]
proporciones_finales_airbnb[columnas_prop] = proporciones_finales_airbnb[columnas_prop].round(4)

proporciones_finales_airbnb = proporciones_finales_airbnb.sort_values(
    "prop_imagenes_pred_encima_mediana",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. Guardar resultados
# ------------------------------------------------------------
pred_todas_csv = SALIDA_DIR / "predicciones_todas_las_fotos_modelo_final_estable.csv"
pred_todas_xlsx = SALIDA_DIR / "predicciones_todas_las_fotos_modelo_final_estable.xlsx"

prop_final_csv = SALIDA_DIR / "proporciones_finales_calidad_por_airbnb_modelo_final_estable.csv"
prop_final_xlsx = SALIDA_DIR / "proporciones_finales_calidad_por_airbnb_modelo_final_estable.xlsx"

def preparar_salida_con_id_excel(df_salida):
    df_salida = df_salida.copy()
    df_salida[ID_COL] = df_salida[ID_COL].astype(str).str.strip()

    id_excel_col = f"{ID_COL}_excel"
    if id_excel_col not in df_salida.columns:
        pos = list(df_salida.columns).index(ID_COL) + 1
        df_salida.insert(pos, id_excel_col, df_salida[ID_COL].apply(lambda x: f'="{x}"'))

    return df_salida


def guardar_xlsx_con_id_texto(df_salida, ruta_xlsx):
    with pd.ExcelWriter(ruta_xlsx, engine="openpyxl") as writer:
        df_salida.to_excel(writer, index=False, sheet_name="resultados")
        ws = writer.sheets["resultados"]

        columnas_texto = {ID_COL, f"{ID_COL}_excel"}
        for col_idx, celda_header in enumerate(ws[1], start=1):
            if celda_header.value in columnas_texto:
                for row_idx in range(2, ws.max_row + 1):
                    celda = ws.cell(row=row_idx, column=col_idx)
                    celda.value = str(celda.value)
                    celda.number_format = "@"


df_todas_pred_salida = preparar_salida_con_id_excel(df_todas_pred)
proporciones_finales_airbnb_salida = preparar_salida_con_id_excel(proporciones_finales_airbnb)

df_todas_pred_salida.to_csv(pred_todas_csv, index=False, encoding="utf-8-sig")
guardar_xlsx_con_id_texto(df_todas_pred_salida, pred_todas_xlsx)

proporciones_finales_airbnb_salida.to_csv(prop_final_csv, index=False, encoding="utf-8-sig")
guardar_xlsx_con_id_texto(proporciones_finales_airbnb_salida, prop_final_xlsx)

print("\nConteos y proporciones finales por Airbnb usando modelo final estable y etiqueta real:")
display(proporciones_finales_airbnb_salida)

print("\nCantidad de Airbnb en proporciones finales:")
print(proporciones_finales_airbnb_salida[ID_COL].nunique())

print("\nArchivos guardados:")
print(pred_todas_csv)
print(pred_todas_xlsx)
print(prop_final_csv)
print(prop_final_xlsx)

# ------------------------------------------------------------
# 7. Lectura rapida
# ------------------------------------------------------------
print("\nLectura:")
print("n_imagenes_real_encima_mediana / debajo_mediana = conteo real por alojamiento.")
print("n_imagenes_pred_encima_mediana / debajo_mediana = conteo predicho por el modelo.")
print("prop_imagenes_real_encima_mediana y prop_imagenes_pred_encima_mediana son proporciones entre 0 y 1.")
print(f"Para Excel: usa el XLSX o la columna {ID_COL}_excel del CSV para evitar notacion cientifica.")


Memoria limpiada antes de predecir todas las fotos.
Dispositivo para prediccion final: cuda
Mixed precision en prediccion final: True
Cargando modelo final estable...
ConvNeXt Tiny con pesos preentrenados.
Modelo final estable cargado correctamente.
Modelo: convnext_tiny
Configuracion: config_4_192_limpia
Batch usado para prediccion: 2

Total de fotos para prediccion final: 4480
Total de Airbnb con fotos: 52

Distribucion de fotos usadas:


,grupo_origen,label_calidad,n_fotos
0,test,alta,252
1,test,media,250
2,train,alta,1536
3,train,media,1496
4,validacion,alta,468
5,validacion,media,478


C:\Users\andre\AppData\Local\Temp\ipykernel_5108\845020441.py:1067: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USAR_AMP):



Predicciones generadas: (4480, 39)


,id_airbnb,nro_foto,image_path,ruta_local,path_modelo,label_ambiente,split,metodo_extraccion,existe_imagen,registro_id,...,label_calidad,existe,label_idx,grupo_origen,existe_imagen_12b,label_calidad_real,label_calidad_predicha,prob_media,prob_alta,label_calidad_modelo_final
0,1089836481621094489,1,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1089836481621094489||img\1089836481621094489\1...,...,alta,True,1,train,True,alta,alta,0.080647,0.919353,alta
1,1089836481621094489,2,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1089836481621094489||img\1089836481621094489\1...,...,alta,True,1,train,True,alta,alta,0.057124,0.942876,alta
2,1089836481621094489,3,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1089836481621094489||img\1089836481621094489\1...,...,alta,True,1,train,True,alta,alta,0.015815,0.984185,alta
3,1089836481621094489,4,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1089836481621094489||img\1089836481621094489\1...,...,media,True,0,train,True,media,media,0.739986,0.260013,media
4,1089836481621094489,5,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,img\1089836481621094489\1089836481621094489_fo...,NaN,NaN,manifest_reconstruido_desde_carpetas,True,1089836481621094489||img\1089836481621094489\1...,...,media,True,0,train,True,media,media,0.896160,0.103839,media



Conteos y proporciones finales por Airbnb usando modelo final estable y etiqueta real:


,id_airbnb,id_airbnb_excel,n_imagenes_real_total,n_imagenes_real_encima_mediana,n_imagenes_real_debajo_mediana,prop_imagenes_real_encima_mediana,prop_imagenes_real_debajo_mediana,n_imagenes_pred_total,n_imagenes_pred_encima_mediana,n_imagenes_pred_debajo_mediana,prop_imagenes_pred_encima_mediana,prop_imagenes_pred_debajo_mediana,dif_prop_pred_vs_real_encima_mediana,prob_alta_promedio
0,1488558453969582434,"=""1488558453969582434""",20,10,10,0.5000,0.5000,20,16,4,0.8000,0.2000,0.3000,0.6489
1,1207414356654599414,"=""1207414356654599414""",122,84,38,0.6885,0.3115,122,84,38,0.6885,0.3115,0.0000,0.6728
2,1417419722718950261,"=""1417419722718950261""",76,52,24,0.6842,0.3158,76,52,24,0.6842,0.3158,0.0000,0.6383
3,1500873674449478593,"=""1500873674449478593""",78,52,26,0.6667,0.3333,78,52,26,0.6667,0.3333,0.0000,0.6191
4,1194885970937542129,"=""1194885970937542129""",70,48,22,0.6857,0.3143,70,46,24,0.6571,0.3429,-0.0286,0.6430
5,1089836481621094489,"=""1089836481621094489""",64,36,28,0.5625,0.4375,64,42,22,0.6562,0.3438,0.0938,0.6051
6,1453092459905498863,"=""1453092459905498863""",64,42,22,0.6562,0.3438,64,42,22,0.6562,0.3438,0.0000,0.6482
7,846313424333805576,"=""846313424333805576""",69,47,22,0.6812,0.3188,69,45,24,0.6522,0.3478,-0.0290,0.6466
8,51764629,"=""51764629""",108,72,36,0.6667,0.3333,108,70,38,0.6481,0.3519,-0.0185,0.6467
9,1247380646917486240,"=""1247380646917486240""",136,94,42,0.6912,0.3088,136,88,48,0.6471,0.3529,-0.0441,0.6373



Cantidad de Airbnb en proporciones finales:
52

Archivos guardados:
salida_airbnb_fotos\predicciones_todas_las_fotos_modelo_final_estable.csv
salida_airbnb_fotos\predicciones_todas_las_fotos_modelo_final_estable.xlsx
salida_airbnb_fotos\proporciones_finales_calidad_por_airbnb_modelo_final_estable.csv
salida_airbnb_fotos\proporciones_finales_calidad_por_airbnb_modelo_final_estable.xlsx

Lectura:
n_imagenes_real_encima_mediana / debajo_mediana = conteo real por alojamiento.
n_imagenes_pred_encima_mediana / debajo_mediana = conteo predicho por el modelo.
prop_imagenes_real_encima_mediana y prop_imagenes_pred_encima_mediana son proporciones entre 0 y 1.
Para Excel: usa el XLSX o la columna id_airbnb_excel del CSV para evitar notacion cientifica.


### Celda 13 — Gráfico comparativo

Este gráfico resume visualmente cuál modelo obtuvo mejor F1-score en `test`. Sirve para el notebook y también para explicar la selección del modelo en el informe.

In [29]:
# ============================================================
# CELDA 13 - Resumen CNN para la rúbrica
# VERSIÓN ULTRA SEGURA
# ============================================================
# Esta celda resume la comparación de arquitecturas y la selección
# del modelo final estable.
#
# Versión ligera para evitar caída del kernel:
# - No usa torch.
# - No carga checkpoints .pt.
# - No usa matplotlib.
# - No genera gráficos pesados.
# - Solo lee archivos CSV ya generados por las celdas 11 y 12.
#
# Sirve para explicar:
# - qué modelos se compararon,
# - qué configuraciones se probaron,
# - qué modelo fue el mejor en test,
# - qué modelo fue elegido finalmente por estabilidad en validación,
# - por qué no se eligió solo por test.
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 0. Asegurar carpeta de salida
# ------------------------------------------------------------
try:
    SALIDA_DIR
except NameError:
    SALIDA_DIR = Path("salida_airbnb_fotos")

SALIDA_DIR = Path(SALIDA_DIR)

# ------------------------------------------------------------
# 1. Rutas de archivos generados por celdas previas
# ------------------------------------------------------------
resultados_path = SALIDA_DIR / "resultados_comparacion_arquitecturas_cnn.csv"
arquitecturas_path = SALIDA_DIR / "ventajas_desventajas_arquitecturas_cnn.csv"
hiperparametros_path = SALIDA_DIR / "hiperparametros_cnn_rubrica.csv"
validacion_candidatos_path = SALIDA_DIR / "metricas_validacion_top_candidatos_cnn.csv"

# ------------------------------------------------------------
# 2. Validar existencia de archivos
# ------------------------------------------------------------
archivos_necesarios = {
    "resultados de test": resultados_path,
    "ventajas/desventajas de arquitecturas": arquitecturas_path,
    "hiperparámetros": hiperparametros_path,
    "métricas de validación de candidatos": validacion_candidatos_path
}

faltantes = []

for nombre, ruta in archivos_necesarios.items():
    if not ruta.exists():
        faltantes.append((nombre, ruta))

if len(faltantes) > 0:
    print("Faltan archivos necesarios para esta celda:")
    for nombre, ruta in faltantes:
        print("-", nombre, ":", ruta)

    raise FileNotFoundError(
        "Primero ejecuta las celdas 11 y 12 corregidas para generar estos archivos."
    )

# ------------------------------------------------------------
# 3. Cargar archivos livianos
# ------------------------------------------------------------
resultados_test_df = pd.read_csv(resultados_path)
tabla_arquitecturas_cnn = pd.read_csv(arquitecturas_path)
tabla_hiperparametros_cnn = pd.read_csv(hiperparametros_path)
metricas_validacion_top_df = pd.read_csv(validacion_candidatos_path)

print("Archivos cargados correctamente.")

# ------------------------------------------------------------
# 4. Preparar tabla comparativa de TEST
# ------------------------------------------------------------
if "test_loss" not in resultados_test_df.columns and "loss" in resultados_test_df.columns:
    resultados_test_df["test_loss"] = resultados_test_df["loss"]

if "configuracion" not in resultados_test_df.columns:
    resultados_test_df["configuracion"] = "config_no_especificada"

columnas_posibles_test = [
    "modelo",
    "configuracion",
    "test_loss",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "img_size",
    "batch_size",
    "augmentation",
    "epochs_head",
    "epochs_fine",
    "lr_head",
    "lr_fine",
    "weight_decay",
    "paciencia"
]

columnas_test_existentes = [
    c for c in columnas_posibles_test
    if c in resultados_test_df.columns
]

comparacion_test_cnn = resultados_test_df[columnas_test_existentes].copy()

comparacion_test_cnn = comparacion_test_cnn.sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

comparacion_test_cnn = comparacion_test_cnn.merge(
    tabla_arquitecturas_cnn,
    on="modelo",
    how="left"
)

print("\nTOP resultados en TEST:")
display(comparacion_test_cnn.head(10))

# ------------------------------------------------------------
# 5. Preparar tabla de estabilidad en VALIDACIÓN
# ------------------------------------------------------------
if "ranking_estabilidad" in metricas_validacion_top_df.columns:
    metricas_validacion_top_df = metricas_validacion_top_df.sort_values(
        "ranking_estabilidad",
        ascending=True
    ).reset_index(drop=True)
else:
    metricas_validacion_top_df = metricas_validacion_top_df.sort_values(
        ["puntaje_estabilidad", "f1_macro_validacion"],
        ascending=[False, False]
    ).reset_index(drop=True)

print("\nComparación de candidatos en VALIDACIÓN:")
display(metricas_validacion_top_df)

# ------------------------------------------------------------
# 6. Identificar mejor modelo en test y modelo final estable
# ------------------------------------------------------------
mejor_test = comparacion_test_cnn.iloc[0]
mejor_final = metricas_validacion_top_df.iloc[0]

modelo_mejor_test = mejor_test["modelo"]
config_mejor_test = mejor_test["configuracion"]
f1_mejor_test = float(mejor_test["f1_macro"])

modelo_final = mejor_final["modelo"]
config_final = mejor_final["configuracion"]
f1_test_final = float(mejor_final["f1_macro_test"])
f1_validacion_final = float(mejor_final["f1_macro_validacion"])
brecha_final = float(mejor_final["brecha_abs_test_validacion"])
puntaje_estabilidad_final = float(mejor_final["puntaje_estabilidad"])

print("\nModelo con mejor resultado en TEST:")
print("Modelo:", modelo_mejor_test)
print("Configuración:", config_mejor_test)
print("F1 macro test:", round(f1_mejor_test, 4))

print("\nModelo final elegido por ESTABILIDAD:")
print("Modelo:", modelo_final)
print("Configuración:", config_final)
print("F1 macro test:", round(f1_test_final, 4))
print("F1 macro validación:", round(f1_validacion_final, 4))
print("Brecha test-validación:", round(brecha_final, 4))
print("Puntaje estabilidad:", round(puntaje_estabilidad_final, 4))

# ------------------------------------------------------------
# 7. Tabla resumen final para la rúbrica
# ------------------------------------------------------------
resumen_final_rubrica = pd.DataFrame([
    {
        "criterio": "Mejor candidato en test",
        "modelo": modelo_mejor_test,
        "configuracion": config_mejor_test,
        "f1_macro_test": f1_mejor_test,
        "f1_macro_validacion": None,
        "brecha_test_validacion": None,
        "puntaje_estabilidad": None,
        "comentario": "Este fue el mejor resultado preliminar en test, pero no necesariamente el más estable."
    },
    {
        "criterio": "Modelo final estable",
        "modelo": modelo_final,
        "configuracion": config_final,
        "f1_macro_test": f1_test_final,
        "f1_macro_validacion": f1_validacion_final,
        "brecha_test_validacion": brecha_final,
        "puntaje_estabilidad": puntaje_estabilidad_final,
        "comentario": "Este fue elegido considerando test, validación y estabilidad."
    }
])

print("\nResumen final para la rúbrica:")
display(resumen_final_rubrica)

# ------------------------------------------------------------
# 8. Guardar tablas finales
# ------------------------------------------------------------
comparacion_test_csv = SALIDA_DIR / "comparacion_test_cnn_rubrica_ligera.csv"
comparacion_test_xlsx = SALIDA_DIR / "comparacion_test_cnn_rubrica_ligera.xlsx"

comparacion_estabilidad_csv = SALIDA_DIR / "comparacion_estabilidad_cnn_rubrica_ligera.csv"
comparacion_estabilidad_xlsx = SALIDA_DIR / "comparacion_estabilidad_cnn_rubrica_ligera.xlsx"

resumen_final_csv = SALIDA_DIR / "resumen_modelo_final_cnn_rubrica_ligera.csv"
resumen_final_xlsx = SALIDA_DIR / "resumen_modelo_final_cnn_rubrica_ligera.xlsx"

comparacion_test_cnn.to_csv(comparacion_test_csv, index=False, encoding="utf-8-sig")
metricas_validacion_top_df.to_csv(comparacion_estabilidad_csv, index=False, encoding="utf-8-sig")
resumen_final_rubrica.to_csv(resumen_final_csv, index=False, encoding="utf-8-sig")

# Guardar Excel con try para que no mate la celda si openpyxl falla
try:
    comparacion_test_cnn.to_excel(comparacion_test_xlsx, index=False)
    metricas_validacion_top_df.to_excel(comparacion_estabilidad_xlsx, index=False)
    resumen_final_rubrica.to_excel(resumen_final_xlsx, index=False)

    print("\nArchivos Excel guardados correctamente.")
except Exception as e:
    print("\nNo se pudieron guardar los Excel, pero los CSV sí quedaron guardados.")
    print("Error:", e)

print("\nArchivos guardados:")
print(comparacion_test_csv)
print(comparacion_estabilidad_csv)
print(resumen_final_csv)

# ------------------------------------------------------------
# 9. Texto interpretativo para la rúbrica
# ------------------------------------------------------------
modelos_evaluados = sorted(comparacion_test_cnn["modelo"].dropna().unique().tolist())
configs_evaluadas = sorted(comparacion_test_cnn["configuracion"].dropna().unique().tolist())

texto_interpretacion_cnn = f"""
Interpretación del módulo CNN:

Se compararon diferentes arquitecturas de redes neuronales convolucionales para clasificar
la calidad visual de las fotografías de Airbnb en dos clases: alta y media. Las arquitecturas
evaluadas fueron: {", ".join(modelos_evaluados)}.

Además, se probaron distintas configuraciones de entrenamiento: {", ".join(configs_evaluadas)}.
Estas configuraciones variaron aspectos como el tamaño de imagen, batch size, número de épocas,
tasa de aprendizaje, regularización y estrategia de aumento de datos.

La métrica principal utilizada fue el F1 macro, porque permite evaluar el desempeño del modelo
considerando ambas clases por igual. Esto es importante porque no basta con acertar la clase
más frecuente; el modelo debe reconocer tanto fotos de calidad alta como fotos de calidad media.

En una primera etapa, los modelos se compararon usando el conjunto de test. El mejor candidato
en test fue {modelo_mejor_test}, con la configuración {config_mejor_test}, alcanzando un F1 macro
de {f1_mejor_test:.3f}.

Sin embargo, para reducir el riesgo de sobreajuste, la selección final no se hizo únicamente con
el resultado de test. Los mejores candidatos fueron evaluados también en el conjunto de validación.
De esta forma, se buscó elegir un modelo que no solo tenga buen desempeño en test, sino que también
mantenga un desempeño consistente en datos no vistos.

El modelo final seleccionado fue {modelo_final}, con la configuración {config_final}. Este modelo
obtuvo un F1 macro en test de {f1_test_final:.3f} y un F1 macro en validación de
{f1_validacion_final:.3f}. La diferencia absoluta entre test y validación fue de {brecha_final:.3f}
y su puntaje de estabilidad fue {puntaje_estabilidad_final:.3f}.

Por ello, el modelo final no fue elegido solamente por tener el mayor desempeño en test, sino por
mostrar una mejor estabilidad entre test y validación. Esta decisión permite sustentar mejor la
generalización del modelo y reduce el riesgo de seleccionar una arquitectura sobreajustada.
"""

print("\nTexto interpretativo:")
print(texto_interpretacion_cnn)

texto_path = SALIDA_DIR / "interpretacion_cnn_rubrica_estabilidad_ligera.txt"

with open(texto_path, "w", encoding="utf-8") as f:
    f.write(texto_interpretacion_cnn)

print("\nTexto interpretativo guardado en:")
print(texto_path)

print("\nCELDA 13 terminada correctamente en versión ligera.")

Archivos cargados correctamente.

TOP resultados en TEST:


,modelo,configuracion,test_loss,accuracy,precision_macro,recall_macro,f1_macro,img_size,batch_size,augmentation,epochs_head,epochs_fine,lr_head,lr_fine,weight_decay,paciencia,tipo,ventaja,desventaja
0,cnn_simple,config_5_256_monstruo_controlado,0.459666,0.824,0.829952,0.830889,0.823989,256,64,crop_suave,8,25,0.0008,0.000005,0.00010,5,CNN desde cero,Es fácil de explicar y sirve como línea base.,Tiene menos conocimiento previo y puede rendir...
1,cnn_simple,config_base_224_limpia,0.479740,0.804,0.802564,0.804890,0.803089,224,64,limpia,5,12,0.0010,0.000010,0.00010,4,CNN desde cero,Es fácil de explicar y sirve como línea base.,Tiene menos conocimiento previo y puede rendir...
2,cnn_simple,config_4_192_limpia,0.489928,0.796,0.801016,0.802241,0.795971,192,64,limpia,4,10,0.0010,0.000020,0.00010,4,CNN desde cero,Es fácil de explicar y sirve como línea base.,Tiene menos conocimiento previo y puede rendir...
3,convnext_tiny,config_4_192_limpia,0.503435,0.792,0.790065,0.790065,0.790065,192,64,limpia,4,10,0.0010,0.000020,0.00010,4,CNN moderna preentrenada,Modelo moderno con buena capacidad de represen...,Es más pesado y puede demorar más en entrenami...
4,cnn_simple,config_3_256_crop,0.517383,0.788,0.802280,0.798043,0.787834,256,32,crop_suave,5,14,0.0010,0.000005,0.00005,5,CNN desde cero,Es fácil de explicar y sirve como línea base.,Tiene menos conocimiento previo y puede rendir...
5,cnn_simple,config_2_224_color,0.488382,0.784,0.791293,0.791293,0.784000,224,32,base_color,6,15,0.0008,0.000008,0.00010,5,CNN desde cero,Es fácil de explicar y sirve como línea base.,Tiene menos conocimiento previo y puede rendir...
6,mobilenet_v2,config_3_256_crop,0.521756,0.780,0.777985,0.779116,0.778437,256,32,crop_suave,5,14,0.0010,0.000005,0.00005,5,CNN preentrenada liviana,Es rápida y consume pocos recursos.,Puede capturar menos detalles que modelos más ...
7,convnext_tiny,config_3_256_crop,0.504061,0.776,0.773913,0.774692,0.774252,256,32,crop_suave,5,14,0.0010,0.000005,0.00005,5,CNN moderna preentrenada,Modelo moderno con buena capacidad de represen...,Es más pesado y puede demorar más en entrenami...
8,resnet18,config_4_192_limpia,0.620400,0.772,0.769943,0.771042,0.770380,192,64,limpia,4,10,0.0010,0.000020,0.00010,4,CNN preentrenada intermedia,Equilibra rendimiento y costo computacional.,Puede no captar detalles finos si la diferenci...
9,mobilenet_v2,config_base_224_limpia,0.544472,0.772,0.769943,0.771042,0.770380,224,64,limpia,5,12,0.0010,0.000010,0.00010,4,CNN preentrenada liviana,Es rápida y consume pocos recursos.,Puede capturar menos detalles que modelos más ...



Comparación de candidatos en VALIDACIÓN:


,ranking_test,modelo,configuracion,checkpoint_path,f1_macro_test,f1_macro_validacion,brecha_abs_test_validacion,accuracy_validacion,precision_macro_validacion,recall_macro_validacion,...,batch_size_validacion,augmentation,epochs_head,epochs_fine,lr_head,lr_fine,weight_decay,paciencia,matriz_confusion,ranking_estabilidad
0,4,convnext_tiny,config_4_192_limpia,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.790065,0.819223,0.029157,0.819239,0.819217,0.819230,...,2,limpia,4,10,0.0010,0.000020,0.00010,4,"[[392, 86], [85, 383]]",1
1,3,cnn_simple,config_4_192_limpia,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.795971,0.788871,0.007100,0.790698,0.803468,0.791769,...,2,limpia,4,10,0.0010,0.000020,0.00010,4,"[[330, 148], [50, 418]]",2
2,2,cnn_simple,config_base_224_limpia,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.803089,0.786323,0.016767,0.787526,0.796018,0.788408,...,2,limpia,5,12,0.0010,0.000010,0.00010,4,"[[337, 141], [60, 408]]",3
3,1,cnn_simple,config_5_256_monstruo_controlado,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.823989,0.787928,0.036060,0.789641,0.801564,0.790679,...,2,crop_suave,8,25,0.0008,0.000005,0.00010,5,"[[331, 147], [52, 416]]",4
4,5,cnn_simple,config_3_256_crop,salida_airbnb_fotos\checkpoints_candidatos_cnn...,0.787834,0.783279,0.004555,0.788584,0.824216,0.790326,...,2,crop_suave,5,14,0.0010,0.000005,0.00005,5,"[[299, 179], [21, 447]]",5



Modelo con mejor resultado en TEST:
Modelo: cnn_simple
Configuración: config_5_256_monstruo_controlado
F1 macro test: 0.824

Modelo final elegido por ESTABILIDAD:
Modelo: convnext_tiny
Configuración: config_4_192_limpia
F1 macro test: 0.7901
F1 macro validación: 0.8192
Brecha test-validación: 0.0292
Puntaje estabilidad: 0.8495

Resumen final para la rúbrica:


,criterio,modelo,configuracion,f1_macro_test,f1_macro_validacion,brecha_test_validacion,puntaje_estabilidad,comentario
0,Mejor candidato en test,cnn_simple,config_5_256_monstruo_controlado,0.823989,NaN,NaN,NaN,Este fue el mejor resultado preliminar en test...
1,Modelo final estable,convnext_tiny,config_4_192_limpia,0.790065,0.819223,0.029157,0.849462,"Este fue elegido considerando test, validación..."



Archivos Excel guardados correctamente.

Archivos guardados:
salida_airbnb_fotos\comparacion_test_cnn_rubrica_ligera.csv
salida_airbnb_fotos\comparacion_estabilidad_cnn_rubrica_ligera.csv
salida_airbnb_fotos\resumen_modelo_final_cnn_rubrica_ligera.csv

Texto interpretativo:

Interpretación del módulo CNN:

Se compararon diferentes arquitecturas de redes neuronales convolucionales para clasificar
la calidad visual de las fotografías de Airbnb en dos clases: alta y media. Las arquitecturas
evaluadas fueron: cnn_simple, convnext_tiny, efficientnet_b0, mobilenet_v2, resnet18.

Además, se probaron distintas configuraciones de entrenamiento: config_2_224_color, config_3_256_crop, config_4_192_limpia, config_5_256_monstruo_controlado, config_base_224_limpia.
Estas configuraciones variaron aspectos como el tamaño de imagen, batch size, número de épocas,
tasa de aprendizaje, regularización y estrategia de aumento de datos.

La métrica principal utilizada fue el F1 macro, porque permite evaluar

## Bloque final CNN: evidencia para la rúbrica y uso operativo

Las siguientes celdas convierten los resultados técnicos en salidas fáciles de reportar:

- tablas de arquitectura, hiperparámetros y métricas;
- análisis de falsos positivos y falsos negativos;
- proporciones de fotos por encima de la mediana (`alta`) y por debajo de la mediana (`media`) por Airbnb;
- predicción de una nueva tanda desde otro Excel tipo G4.


### Celda 14 — Tablas para explicar la CNN en el informe

Esta celda genera tablas listas para copiar al informe: hiperparámetros, arquitecturas comparadas, ventajas/desventajas y resumen de métricas. También produce un texto corto para explicar el módulo CNN a alguien que no conoce el tema.


In [30]:
# ============================================================
# CELDA 14 - Resumen técnico CNN para rúbrica
# VERSIÓN AJUSTADA AL MODELO FINAL ESTABLE
# ============================================================
# Esta celda crea tablas simples para explicar:
# - particiones usadas por la CNN,
# - arquitecturas comparadas,
# - hiperparámetros probados,
# - comparación entre test y validación,
# - criterio de selección del modelo final estable.
#
# Versión segura:
# - No carga modelos.
# - No usa torch.
# - No genera gráficos.
# - Solo lee tablas CSV generadas previamente.
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# ------------------------------------------------------------
# 0. Carpeta de salida
# ------------------------------------------------------------
SALIDA_DIR = Path("salida_airbnb_fotos")
SALIDA_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Cargar dataset base para resumen de particiones
# ------------------------------------------------------------
dataset_cnn_path = SALIDA_DIR / "dataset_cnn_calidad_etiquetado.csv"

if "df" in globals():
    df_cnn_resumen = df.copy()
elif dataset_cnn_path.exists():
    df_cnn_resumen = pd.read_csv(dataset_cnn_path)
else:
    raise FileNotFoundError(
        f"No encontré el dataset CNN:\n{dataset_cnn_path}\n\n"
        "Ejecuta primero la Celda 8 o la Celda 9."
    )

# Validar columnas necesarias
columnas_necesarias = ["split_calidad", "label_calidad"]

for col in columnas_necesarias:
    if col not in df_cnn_resumen.columns:
        raise ValueError(
            f"Falta la columna '{col}' en el dataset CNN. "
            "Revisa que la Celda 8 haya generado correctamente el dataset."
        )

# ------------------------------------------------------------
# 2. Resumen de particiones
# ------------------------------------------------------------
resumen_splits = (
    df_cnn_resumen
    .groupby(["split_calidad", "label_calidad"])
    .size()
    .reset_index(name="n_fotos")
)

print("Resumen de particiones usadas por la CNN:")
display(resumen_splits)

# ------------------------------------------------------------
# 3. Cargar resultados generados por celdas 11, 12 y 13
# ------------------------------------------------------------
hiperparametros_path = SALIDA_DIR / "hiperparametros_cnn_rubrica.csv"
arquitecturas_path = SALIDA_DIR / "ventajas_desventajas_arquitecturas_cnn.csv"
resultados_test_path = SALIDA_DIR / "resultados_comparacion_arquitecturas_cnn.csv"
validacion_top_path = SALIDA_DIR / "metricas_validacion_top_candidatos_cnn.csv"

archivos_requeridos = {
    "hiperparámetros": hiperparametros_path,
    "arquitecturas": arquitecturas_path,
    "resultados de test": resultados_test_path,
    "validación de candidatos": validacion_top_path
}

faltantes = []

for nombre, ruta in archivos_requeridos.items():
    if not ruta.exists():
        faltantes.append((nombre, ruta))

if len(faltantes) > 0:
    print("Faltan archivos necesarios:")
    for nombre, ruta in faltantes:
        print("-", nombre, ":", ruta)

    raise FileNotFoundError(
        "Ejecuta primero las celdas 11, 12 y 13 corregidas."
    )

tabla_hiperparametros_cnn = pd.read_csv(hiperparametros_path)
tabla_arquitecturas_cnn = pd.read_csv(arquitecturas_path)
resultados_test_df = pd.read_csv(resultados_test_path)
metricas_validacion_top_df = pd.read_csv(validacion_top_path)

# ------------------------------------------------------------
# 4. Tabla de hiperparámetros
# ------------------------------------------------------------
print("Hiperparámetros probados en la comparación CNN:")
display(tabla_hiperparametros_cnn)

# ------------------------------------------------------------
# 5. Tabla de arquitecturas comparadas
# ------------------------------------------------------------
print("Arquitecturas CNN comparadas:")
display(tabla_arquitecturas_cnn)

# ------------------------------------------------------------
# 6. Resumen de resultados en test
# ------------------------------------------------------------
if "configuracion" not in resultados_test_df.columns:
    resultados_test_df["configuracion"] = "config_no_especificada"

columnas_test = [
    "modelo",
    "configuracion",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "img_size",
    "batch_size",
    "augmentation",
    "epochs_head",
    "epochs_fine",
    "lr_head",
    "lr_fine",
    "weight_decay",
    "paciencia"
]

columnas_test_existentes = [
    col for col in columnas_test
    if col in resultados_test_df.columns
]

resumen_test = resultados_test_df[columnas_test_existentes].copy()

resumen_test = resumen_test.sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

print("Resumen de desempeño en TEST:")
display(resumen_test.head(10))

# ------------------------------------------------------------
# 7. Resumen de estabilidad en validación
# ------------------------------------------------------------
if "ranking_estabilidad" in metricas_validacion_top_df.columns:
    resumen_estabilidad = metricas_validacion_top_df.sort_values(
        "ranking_estabilidad",
        ascending=True
    ).reset_index(drop=True)
else:
    resumen_estabilidad = metricas_validacion_top_df.sort_values(
        ["puntaje_estabilidad", "f1_macro_validacion"],
        ascending=[False, False]
    ).reset_index(drop=True)

print("Resumen de estabilidad en VALIDACIÓN:")
display(resumen_estabilidad)

# ------------------------------------------------------------
# 8. Identificar modelo preliminar y modelo final estable
# ------------------------------------------------------------
mejor_test = resumen_test.iloc[0]
mejor_estable = resumen_estabilidad.iloc[0]

modelo_mejor_test = mejor_test["modelo"]
config_mejor_test = mejor_test["configuracion"]
f1_mejor_test = float(mejor_test["f1_macro"])

modelo_final = mejor_estable["modelo"]
config_final = mejor_estable["configuracion"]
f1_test_final = float(mejor_estable["f1_macro_test"])
f1_validacion_final = float(mejor_estable["f1_macro_validacion"])
brecha_final = float(mejor_estable["brecha_abs_test_validacion"])
puntaje_estabilidad_final = float(mejor_estable["puntaje_estabilidad"])

criterio_seleccion = pd.DataFrame([
    {
        "etapa": "Selección preliminar",
        "criterio": "Mayor F1 macro en test",
        "modelo": modelo_mejor_test,
        "configuracion": config_mejor_test,
        "f1_macro_test": f1_mejor_test,
        "f1_macro_validacion": None,
        "brecha_test_validacion": None,
        "puntaje_estabilidad": None,
        "interpretacion": "Sirve como primer filtro, pero no garantiza estabilidad."
    },
    {
        "etapa": "Selección final",
        "criterio": "Estabilidad entre test y validación",
        "modelo": modelo_final,
        "configuracion": config_final,
        "f1_macro_test": f1_test_final,
        "f1_macro_validacion": f1_validacion_final,
        "brecha_test_validacion": brecha_final,
        "puntaje_estabilidad": puntaje_estabilidad_final,
        "interpretacion": "Se eligió porque mantiene mejor desempeño en validación y reduce riesgo de sobreajuste."
    }
])

print("Criterio de selección del modelo CNN:")
display(criterio_seleccion)

# ------------------------------------------------------------
# 9. Guardar tablas técnicas
# ------------------------------------------------------------
resumen_splits_csv = SALIDA_DIR / "cnn_resumen_splits.csv"
hiperparametros_csv = SALIDA_DIR / "cnn_hiperparametros_finales.csv"
arquitecturas_csv = SALIDA_DIR / "cnn_arquitecturas_comparadas_finales.csv"
resumen_test_csv = SALIDA_DIR / "cnn_resumen_test.csv"
resumen_estabilidad_csv = SALIDA_DIR / "cnn_resumen_estabilidad_validacion.csv"
criterio_csv = SALIDA_DIR / "cnn_criterio_seleccion_modelo_final.csv"

resumen_splits.to_csv(resumen_splits_csv, index=False, encoding="utf-8-sig")
tabla_hiperparametros_cnn.to_csv(hiperparametros_csv, index=False, encoding="utf-8-sig")
tabla_arquitecturas_cnn.to_csv(arquitecturas_csv, index=False, encoding="utf-8-sig")
resumen_test.to_csv(resumen_test_csv, index=False, encoding="utf-8-sig")
resumen_estabilidad.to_csv(resumen_estabilidad_csv, index=False, encoding="utf-8-sig")
criterio_seleccion.to_csv(criterio_csv, index=False, encoding="utf-8-sig")

# Guardar también en Excel si es posible
try:
    resumen_splits.to_excel(SALIDA_DIR / "cnn_resumen_splits.xlsx", index=False)
    tabla_hiperparametros_cnn.to_excel(SALIDA_DIR / "cnn_hiperparametros_finales.xlsx", index=False)
    tabla_arquitecturas_cnn.to_excel(SALIDA_DIR / "cnn_arquitecturas_comparadas_finales.xlsx", index=False)
    resumen_test.to_excel(SALIDA_DIR / "cnn_resumen_test.xlsx", index=False)
    resumen_estabilidad.to_excel(SALIDA_DIR / "cnn_resumen_estabilidad_validacion.xlsx", index=False)
    criterio_seleccion.to_excel(SALIDA_DIR / "cnn_criterio_seleccion_modelo_final.xlsx", index=False)

    print("\nArchivos Excel guardados correctamente.")
except Exception as e:
    print("\nNo se pudieron guardar algunos Excel, pero los CSV sí fueron guardados.")
    print("Error:", e)

print("\nArchivos CSV guardados:")
print(resumen_splits_csv)
print(hiperparametros_csv)
print(arquitecturas_csv)
print(resumen_test_csv)
print(resumen_estabilidad_csv)
print(criterio_csv)

# ------------------------------------------------------------
# 10. Texto corto para informe
# ------------------------------------------------------------
modelos_evaluados = sorted(resumen_test["modelo"].dropna().unique().tolist())

if "configuracion" in resumen_test.columns:
    configuraciones_evaluadas = sorted(resumen_test["configuracion"].dropna().unique().tolist())
else:
    configuraciones_evaluadas = []

texto_cnn = f"""
### Explicación simple del módulo CNN

El módulo CNN evalúa automáticamente las fotografías de los anuncios de Airbnb. La entrada
del modelo es una imagen y la salida es una clasificación de calidad visual en dos clases:
**alta** o **media**. Las imágenes no evaluables se excluyen porque no muestran claramente
el inmueble o no aportan información visual útil.

Para este módulo se compararon distintas arquitecturas de redes neuronales convolucionales:
{", ".join(modelos_evaluados)}. Además, se probaron diferentes configuraciones de entrenamiento,
incluyendo variaciones en tamaño de imagen, batch size, número de épocas, tasa de aprendizaje,
regularización y estrategia de aumento de datos.

La métrica principal utilizada fue el **F1 macro**, porque permite evaluar el desempeño del
modelo considerando ambas clases por igual. Esto es importante porque no basta con acertar la
clase más frecuente; el modelo debe reconocer tanto fotos de calidad alta como fotos de calidad
media.

Primero se identificó el mejor candidato en el conjunto de test. En esa etapa, el mejor resultado
fue obtenido por **{modelo_mejor_test}** con la configuración **{config_mejor_test}**, alcanzando
un F1 macro en test de **{f1_mejor_test:.3f}**.

Sin embargo, para reducir el riesgo de sobreajuste, la selección final no se realizó únicamente
por el resultado en test. Los mejores candidatos fueron evaluados también en el conjunto de
validación. De esta manera, se eligió el modelo que mostró mayor estabilidad entre test y
validación.

El modelo final seleccionado fue **{modelo_final}** con la configuración **{config_final}**.
Este modelo obtuvo un F1 macro en test de **{f1_test_final:.3f}** y un F1 macro en validación
de **{f1_validacion_final:.3f}**. La diferencia absoluta entre test y validación fue de
**{brecha_final:.3f}**, y su puntaje de estabilidad fue **{puntaje_estabilidad_final:.3f}**.

Por ello, el modelo final no fue elegido solo por ser el mejor en test, sino por mantener un
desempeño más consistente en validación. Esta decisión permite justificar mejor la capacidad
de generalización del modelo y reduce el riesgo de seleccionar una arquitectura sobreajustada.
"""

display(Markdown(texto_cnn))

texto_path = SALIDA_DIR / "texto_explicativo_cnn_para_informe.md"

with open(texto_path, "w", encoding="utf-8") as f:
    f.write(texto_cnn)

print("\nTexto explicativo guardado en:")
print(texto_path)

print("\nCELDA 14 terminada correctamente.")

Resumen de particiones usadas por la CNN:


,split_calidad,label_calidad,n_fotos
0,test,alta,252
1,test,media,250
2,train,alta,1536
3,train,media,1496
4,validacion,alta,468
5,validacion,media,478


Hiperparámetros probados en la comparación CNN:


,configuracion,hiperparametro,valor,explicacion
0,config_base_224_limpia,img_size,224,Tamaño al que se redimensionan las imágenes an...
1,config_base_224_limpia,batch_size,64,Cantidad de imágenes procesadas en cada paso d...
2,config_base_224_limpia,augmentation,limpia,Estrategia de transformación de imágenes duran...
3,config_base_224_limpia,epochs_head,5,Épocas para entrenar la cabeza clasificadora d...
4,config_base_224_limpia,epochs_fine,12,Épocas para ajustar las últimas capas del mode...
5,config_base_224_limpia,lr_head,0.001,Tasa de aprendizaje usada al entrenar la cabez...
6,config_base_224_limpia,lr_fine,1e-05,Tasa de aprendizaje usada en el fine-tuning.
7,config_base_224_limpia,weight_decay,0.0001,Regularización para reducir sobreajuste.
8,config_base_224_limpia,paciencia,4,Número de épocas sin mejora antes de detener e...
9,config_2_224_color,img_size,224,Tamaño al que se redimensionan las imágenes an...


Arquitecturas CNN comparadas:


...

Criterio de selección del modelo CNN:


,etapa,criterio,modelo,configuracion,f1_macro_test,f1_macro_validacion,brecha_test_validacion,puntaje_estabilidad,interpretacion
0,Selección preliminar,Mayor F1 macro en test,cnn_simple,config_5_256_monstruo_controlado,0.823989,NaN,NaN,NaN,"Sirve como primer filtro, pero no garantiza es..."
1,Selección final,Estabilidad entre test y validación,convnext_tiny,config_4_192_limpia,0.790065,0.819223,0.029157,0.849462,Se eligió porque mantiene mejor desempeño en v...



Archivos Excel guardados correctamente.

Archivos CSV guardados:
salida_airbnb_fotos\cnn_resumen_splits.csv
salida_airbnb_fotos\cnn_hiperparametros_finales.csv
salida_airbnb_fotos\cnn_arquitecturas_comparadas_finales.csv
salida_airbnb_fotos\cnn_resumen_test.csv
salida_airbnb_fotos\cnn_resumen_estabilidad_validacion.csv
salida_airbnb_fotos\cnn_criterio_seleccion_modelo_final.csv



### Explicación simple del módulo CNN

El módulo CNN evalúa automáticamente las fotografías de los anuncios de Airbnb. La entrada
del modelo es una imagen y la salida es una clasificación de calidad visual en dos clases:
**alta** o **media**. Las imágenes no evaluables se excluyen porque no muestran claramente
el inmueble o no aportan información visual útil.

Para este módulo se compararon distintas arquitecturas de redes neuronales convolucionales:
cnn_simple, convnext_tiny, efficientnet_b0, mobilenet_v2, resnet18. Además, se probaron diferentes configuraciones de entrenamiento,
incluyendo variaciones en tamaño de imagen, batch size, número de épocas, tasa de aprendizaje,
regularización y estrategia de aumento de datos.

La métrica principal utilizada fue el **F1 macro**, porque permite evaluar el desempeño del
modelo considerando ambas clases por igual. Esto es importante porque no basta con acertar la
clase más frecuente; el modelo debe reconocer tanto fotos de calidad alta como fotos de calidad
media.

Primero se identificó el mejor candidato en el conjunto de test. En esa etapa, el mejor resultado
fue obtenido por **cnn_simple** con la configuración **config_5_256_monstruo_controlado**, alcanzando
un F1 macro en test de **0.824**.

Sin embargo, para reducir el riesgo de sobreajuste, la selección final no se realizó únicamente
por el resultado en test. Los mejores candidatos fueron evaluados también en el conjunto de
validación. De esta manera, se eligió el modelo que mostró mayor estabilidad entre test y
validación.

El modelo final seleccionado fue **convnext_tiny** con la configuración **config_4_192_limpia**.
Este modelo obtuvo un F1 macro en test de **0.790** y un F1 macro en validación
de **0.819**. La diferencia absoluta entre test y validación fue de
**0.029**, y su puntaje de estabilidad fue **0.849**.

Por ello, el modelo final no fue elegido solo por ser el mejor en test, sino por mantener un
desempeño más consistente en validación. Esta decisión permite justificar mejor la capacidad
de generalización del modelo y reduce el riesgo de seleccionar una arquitectura sobreajustada.



Texto explicativo guardado en:
salida_airbnb_fotos\texto_explicativo_cnn_para_informe.md

CELDA 14 terminada correctamente.


### Celda 15 — Indicadores por Airbnb con el modelo ganador

Esta celda aplica el mejor modelo sobre las fotos de `validacion` y resume el resultado por Airbnb. Conserva la etiqueta humana como referencia, pero crea una etiqueta operativa basada en la predicción del modelo.

Salidas principales:

| Columna | Significado |
|---|---|
| `n_fotos_modelo` | Fotos evaluadas del Airbnb. |
| `n_alta_modelo` | Fotos predichas en o por encima de la mediana visual. |
| `n_media_modelo` | Fotos predichas por debajo de la mediana visual. |
| `prop_fotos_buena_calidad` | Porcentaje de fotos `alta`. |
| `prop_fotos_calidad_media` | Porcentaje de fotos `media`. |


In [31]:
# ============================================================
# CELDA 15 - Indicadores por Airbnb en validacion
# ============================================================
# Predice las fotos de validacion y resume, por alojamiento:
# - numero real de imagenes encima/debajo de la mediana
# - numero predicho de imagenes encima/debajo de la mediana
# - proporcion real y predicha de imagenes encima/debajo de la mediana
#
# No depende de objetos previos del kernel para predecir.
# Si df_validacion no existe, lo carga desde dataset_cnn_validacion.csv.
# ============================================================

import copy
import numpy as np
import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from IPython.display import display

SALIDA_DIR = Path("salida_airbnb_fotos")
VALIDACION_CSV = SALIDA_DIR / "dataset_cnn_validacion.csv"
CHECKPOINT_FINAL = SALIDA_DIR / "modelo_final_estable_cnn.pt"
CHECKPOINT_GANADOR_TEST = SALIDA_DIR / "modelo_ganador_experimentos_cnn.pt"

# ------------------------------------------------------------
# 0. Validar dependencias
# ------------------------------------------------------------
for nombre in ["crear_modelo", "crear_transforms"]:
    if nombre not in globals():
        raise NameError(
            f"Falta {nombre}. Ejecuta primero las celdas 10/10A antes de correr esta celda."
        )

if "ID_COL" not in globals():
    ID_COL = "id_airbnb"

# ------------------------------------------------------------
# 1. Cargar validacion si no existe en memoria
# ------------------------------------------------------------
if "df_validacion" not in globals():
    if not VALIDACION_CSV.exists():
        raise FileNotFoundError(
            f"No existe {VALIDACION_CSV}. Ejecuta primero la celda que genera dataset_cnn_validacion.csv."
        )
    df_validacion = pd.read_csv(VALIDACION_CSV)
    print("df_validacion no estaba en memoria. Se cargo desde:")
    print(VALIDACION_CSV)
else:
    df_validacion = df_validacion.copy()

if ID_COL not in df_validacion.columns:
    if "id_airbnb" in df_validacion.columns:
        ID_COL = "id_airbnb"
    elif "ID Airbnb" in df_validacion.columns:
        ID_COL = "ID Airbnb"
    else:
        raise ValueError("No encontre columna de ID Airbnb en df_validacion.")

if "path_modelo" not in df_validacion.columns:
    if "image_path" in df_validacion.columns:
        df_validacion["path_modelo"] = df_validacion["image_path"].astype(str)
    elif "ruta_imagen_final" in df_validacion.columns:
        df_validacion["path_modelo"] = df_validacion["ruta_imagen_final"].astype(str)
    elif "ruta_imagen" in df_validacion.columns:
        df_validacion["path_modelo"] = df_validacion["ruta_imagen"].astype(str)
    else:
        raise ValueError("df_validacion no tiene path_modelo/image_path/ruta_imagen.")

if "label_calidad" not in df_validacion.columns:
    if "calidad_final" in df_validacion.columns:
        df_validacion["label_calidad"] = df_validacion["calidad_final"].astype(str)
    else:
        raise ValueError("df_validacion no tiene label_calidad ni calidad_final.")

df_validacion[ID_COL] = df_validacion[ID_COL].astype(str).str.strip()
df_validacion["path_modelo"] = df_validacion["path_modelo"].astype(str)
df_validacion["label_calidad"] = df_validacion["label_calidad"].astype(str).str.lower().str.strip()
df_validacion = df_validacion[df_validacion["label_calidad"].isin(["alta", "media"])].copy()
df_validacion = df_validacion.drop_duplicates(subset=["path_modelo"]).copy()

df_validacion["existe_imagen"] = df_validacion["path_modelo"].apply(lambda x: Path(str(x)).exists())
df_validacion = df_validacion[df_validacion["existe_imagen"]].copy()

if len(df_validacion) == 0:
    raise ValueError("No quedaron fotos validas en df_validacion despues de filtrar etiquetas/rutas.")

print("Fotos de validacion:", len(df_validacion))
print("Alojamientos en validacion:", df_validacion[ID_COL].nunique())

# ------------------------------------------------------------
# 2. Cargar modelo desde checkpoint
# ------------------------------------------------------------
def cargar_checkpoint_seguro(path_checkpoint):
    try:
        return torch.load(path_checkpoint, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path_checkpoint, map_location="cpu")

checkpoint_path = CHECKPOINT_FINAL if CHECKPOINT_FINAL.exists() else CHECKPOINT_GANADOR_TEST

if not checkpoint_path.exists():
    raise FileNotFoundError(
        "No encontre checkpoint final/ganador en disco. Ejecuta primero la celda 11 o la celda 12."
    )

checkpoint = cargar_checkpoint_seguro(checkpoint_path)
modelo_eval_nombre = checkpoint["modelo"]
config_eval = copy.deepcopy(checkpoint.get("config_validacion", checkpoint.get("config")))

if config_eval is None:
    config_eval = {"img_size": 224, "augmentation": "limpia", "batch_size": 8}

if "label_to_idx" not in checkpoint or "idx_to_label" not in checkpoint:
    raise ValueError("El checkpoint no contiene label_to_idx/idx_to_label.")

label_to_idx = checkpoint["label_to_idx"]
idx_to_label = checkpoint["idx_to_label"]

modelo_eval = crear_modelo(modelo_eval_nombre)
modelo_eval.load_state_dict(checkpoint["state_dict"])

print("Modelo cargado desde:")
print(checkpoint_path)
print("Modelo:", modelo_eval_nombre)
print("Config:", config_eval)

# ------------------------------------------------------------
# 3. Predecir validacion foto por foto
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USAR_AMP = bool(device.type == "cuda" and globals().get("USAR_AMP", True))

modelo_eval = modelo_eval.to(device)
modelo_eval.eval()

_, eval_transform = crear_transforms(
    img_size=config_eval["img_size"],
    augmentation=config_eval.get("augmentation", "limpia")
)

pred_labels = []
prob_altas = []

with torch.no_grad():
    for _, row in df_validacion.iterrows():
        image = Image.open(row["path_modelo"]).convert("RGB")
        image_tensor = eval_transform(image).unsqueeze(0).to(device)

        if USAR_AMP:
            with torch.amp.autocast("cuda"):
                output = modelo_eval(image_tensor)
        else:
            output = modelo_eval(image_tensor)

        probs = torch.softmax(output, dim=1).detach().cpu().numpy()[0]
        pred_idx = int(np.argmax(probs))
        pred_label = idx_to_label[pred_idx]

        pred_labels.append(pred_label)
        prob_altas.append(float(probs[label_to_idx["alta"]]))

df_validacion_pred = df_validacion.reset_index(drop=True).copy()
df_validacion_pred["label_calidad_real"] = df_validacion_pred["label_calidad"]
df_validacion_pred["label_calidad_predicha"] = pred_labels
df_validacion_pred["label_calidad_final_modelo"] = pred_labels
df_validacion_pred["prob_alta"] = prob_altas

# ------------------------------------------------------------
# 4. Conteos y proporciones por Airbnb
# ------------------------------------------------------------
def resumen_mediana_por_airbnb(df_base, columna_label, prefijo):
    resumen = (
        df_base
        .groupby(ID_COL)[columna_label]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["alta", "media"]:
        if col not in resumen.columns:
            resumen[col] = 0

    resumen[f"n_imagenes_{prefijo}_total"] = resumen["alta"] + resumen["media"]
    resumen[f"n_imagenes_{prefijo}_encima_mediana"] = resumen["alta"]
    resumen[f"n_imagenes_{prefijo}_debajo_mediana"] = resumen["media"]
    resumen[f"prop_imagenes_{prefijo}_encima_mediana"] = (
        resumen[f"n_imagenes_{prefijo}_encima_mediana"]
        / resumen[f"n_imagenes_{prefijo}_total"]
    )
    resumen[f"prop_imagenes_{prefijo}_debajo_mediana"] = (
        resumen[f"n_imagenes_{prefijo}_debajo_mediana"]
        / resumen[f"n_imagenes_{prefijo}_total"]
    )

    return resumen[
        [
            ID_COL,
            f"n_imagenes_{prefijo}_total",
            f"n_imagenes_{prefijo}_encima_mediana",
            f"n_imagenes_{prefijo}_debajo_mediana",
            f"prop_imagenes_{prefijo}_encima_mediana",
            f"prop_imagenes_{prefijo}_debajo_mediana"
        ]
    ].copy()

resumen_real = resumen_mediana_por_airbnb(df_validacion_pred, "label_calidad_real", "real")
resumen_pred = resumen_mediana_por_airbnb(df_validacion_pred, "label_calidad_predicha", "pred")

prob_promedio = (
    df_validacion_pred
    .groupby(ID_COL)["prob_alta"]
    .mean()
    .reset_index()
    .rename(columns={"prob_alta": "prob_alta_promedio"})
)

resumen_validacion_modelo = (
    resumen_real
    .merge(resumen_pred, on=ID_COL, how="outer")
    .merge(prob_promedio, on=ID_COL, how="left")
)

resumen_validacion_modelo["dif_prop_pred_vs_real_encima_mediana"] = (
    resumen_validacion_modelo["prop_imagenes_pred_encima_mediana"]
    - resumen_validacion_modelo["prop_imagenes_real_encima_mediana"]
)

columnas_prop = [
    c for c in resumen_validacion_modelo.columns
    if c.startswith("prop_") or c.startswith("dif_prop_") or c == "prob_alta_promedio"
]
resumen_validacion_modelo[columnas_prop] = resumen_validacion_modelo[columnas_prop].round(4)

columnas_finales = [
    ID_COL,
    "n_imagenes_real_total",
    "n_imagenes_real_encima_mediana",
    "n_imagenes_real_debajo_mediana",
    "prop_imagenes_real_encima_mediana",
    "prop_imagenes_real_debajo_mediana",
    "n_imagenes_pred_total",
    "n_imagenes_pred_encima_mediana",
    "n_imagenes_pred_debajo_mediana",
    "prop_imagenes_pred_encima_mediana",
    "prop_imagenes_pred_debajo_mediana",
    "dif_prop_pred_vs_real_encima_mediana",
    "prob_alta_promedio"
]

resumen_validacion_modelo = resumen_validacion_modelo[columnas_finales].copy()

resumen_validacion_modelo = resumen_validacion_modelo.sort_values(
    "prop_imagenes_pred_encima_mediana",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Guardar resultados
# ------------------------------------------------------------
pred_csv = SALIDA_DIR / "validacion_predicciones_modelo_final.csv"
pred_xlsx = SALIDA_DIR / "validacion_predicciones_modelo_final.xlsx"
resumen_csv = SALIDA_DIR / "proporciones_calidad_por_airbnb_validacion_modelo.csv"
resumen_xlsx = SALIDA_DIR / "proporciones_calidad_por_airbnb_validacion_modelo.xlsx"

df_validacion_pred.to_csv(pred_csv, index=False, encoding="utf-8-sig")
df_validacion_pred.to_excel(pred_xlsx, index=False)
resumen_validacion_modelo.to_csv(resumen_csv, index=False, encoding="utf-8-sig")
resumen_validacion_modelo.to_excel(resumen_xlsx, index=False)

print("\nPredicciones de validacion guardadas:")
print(pred_csv)
print(pred_xlsx)

print("\nIndicadores por Airbnb en validacion:")
display(resumen_validacion_modelo)

print("\nArchivos guardados:")
print(resumen_csv)
print(resumen_xlsx)


Fotos de validacion: 946
Alojamientos en validacion: 10
ConvNeXt Tiny con pesos preentrenados.
Modelo cargado desde:
salida_airbnb_fotos\modelo_final_estable_cnn.pt
Modelo: convnext_tiny
Config: {'img_size': 192, 'batch_size': 2, 'augmentation': 'limpia', 'epochs_head': 4, 'epochs_fine': 10, 'lr_head': 0.001, 'lr_fine': 2e-05, 'weight_decay': 0.0001, 'paciencia': 4}

Predicciones de validacion guardadas:
salida_airbnb_fotos\validacion_predicciones_modelo_final.csv
salida_airbnb_fotos\validacion_predicciones_modelo_final.xlsx

Indicadores por Airbnb en validacion:


,id_airbnb,n_imagenes_real_total,n_imagenes_real_encima_mediana,n_imagenes_real_debajo_mediana,prop_imagenes_real_encima_mediana,prop_imagenes_real_debajo_mediana,n_imagenes_pred_total,n_imagenes_pred_encima_mediana,n_imagenes_pred_debajo_mediana,prop_imagenes_pred_encima_mediana,prop_imagenes_pred_debajo_mediana,dif_prop_pred_vs_real_encima_mediana,prob_alta_promedio
0,1422671086540527802,68,28,40,0.4118,0.5882,68,44,24,0.6471,0.3529,0.2353,0.5891
1,1202250552806361613,146,100,46,0.6849,0.3151,146,88,58,0.6027,0.3973,-0.0822,0.6283
2,1412366769894758932,158,84,74,0.5316,0.4684,158,88,70,0.5570,0.4430,0.0253,0.5324
3,1158688913933082401,66,31,35,0.4697,0.5303,66,35,31,0.5303,0.4697,0.0606,0.4733
4,742541719033641514,98,52,46,0.5306,0.4694,98,50,48,0.5102,0.4898,-0.0204,0.4950
5,976060981254094446,140,68,72,0.4857,0.5143,140,66,74,0.4714,0.5286,-0.0143,0.4860
6,827026659851983853,51,26,25,0.5098,0.4902,51,22,29,0.4314,0.5686,-0.0784,0.5004
7,1648119432301960151,127,53,74,0.4173,0.5827,127,53,74,0.4173,0.5827,0.0000,0.4232
8,1118142286350817818,62,16,46,0.2581,0.7419,62,18,44,0.2903,0.7097,0.0323,0.3327
9,1086343485196324895,30,10,20,0.3333,0.6667,30,4,26,0.1333,0.8667,-0.2000,0.2749



Archivos guardados:
salida_airbnb_fotos\proporciones_calidad_por_airbnb_validacion_modelo.csv
salida_airbnb_fotos\proporciones_calidad_por_airbnb_validacion_modelo.xlsx


### Celda 16 — Nueva tanda desde otro Excel tipo G4

Usa esta celda cuando tengas nuevos anuncios. La entrada es un Excel parecido al original, con hoja `Principal` y columnas `URL Canónica` e `ID Airbnb`.

La celda hace cuatro cosas:

1. lee el nuevo Excel;
2. descarga fotos desde el recorrido de Airbnb;
3. aplica el modelo ganador a cada foto;
4. devuelve un Excel enriquecido con proporción de fotos de buena calidad y proporción de fotos de calidad media por Airbnb.

Para evitar usar por error el Excel antiguo, pon el nombre del nuevo archivo en `NUEVO_EXCEL_PATH`, por ejemplo `G4_nueva_tanda.xlsx`.


In [ ]:
# ============================================================
# CELDA 16 - Aplicar modelo CNN ganador a una nueva tanda tipo G4
# ============================================================
# Esta celda sirve para usar el mejor modelo CNN ya entrenado sobre una nueva tanda.
#
# ¿Qué espera?
# 1. Un Excel nuevo parecido al G4.
# 2. Una carpeta con fotos descargadas por ID Airbnb.
#
# Estructura esperada de fotos:
#
# nueva_tanda_fotos/
#     708973846265169428/
#         foto_001.jpg
#         foto_002.jpg
#     1135034995673292208/
#         foto_001.jpg
#         foto_002.jpg
#
# Resultado:
# - Devuelve el mismo Excel original, pero con columnas nuevas:
#   n_fotos_modelo
#   n_fotos_buena_calidad
#   n_fotos_calidad_media
#   prop_fotos_buena_calidad
#   prop_fotos_calidad_media
#   prob_alta_promedio
#   estado_procesamiento_fotos
#
# Importante:
# Si un Airbnb del Excel no tiene fotos, igual aparecerá en el resultado final.
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch

# ------------------------------------------------------------
# 1. Configuración de entrada
# ------------------------------------------------------------
# Cambia el nombre del Excel si tu archivo nuevo tiene otro nombre.
# Debe estar en la misma carpeta del notebook o colocar la ruta completa.

RUTA_EXCEL_NUEVA_TANDA = Path("G4_nueva_tanda.xlsx")

# Carpeta donde estarán las fotos de la nueva tanda.
# Debe tener subcarpetas por ID Airbnb.
CARPETA_FOTOS_NUEVA_TANDA = Path("nueva_tanda_fotos")

# Carpeta de salida
SALIDA_NUEVA_TANDA = SALIDA_DIR / "predicciones_nueva_tanda"
SALIDA_NUEVA_TANDA.mkdir(parents=True, exist_ok=True)

print("Excel nueva tanda:", RUTA_EXCEL_NUEVA_TANDA)
print("Carpeta fotos nueva tanda:", CARPETA_FOTOS_NUEVA_TANDA)
print("Carpeta salida:", SALIDA_NUEVA_TANDA)

# ------------------------------------------------------------
# 2. Validar archivos de entrada
# ------------------------------------------------------------
if not RUTA_EXCEL_NUEVA_TANDA.exists():
    raise FileNotFoundError(
        f"No encontré el Excel de nueva tanda: {RUTA_EXCEL_NUEVA_TANDA}\n"
        "Coloca el archivo en la carpeta del notebook o cambia RUTA_EXCEL_NUEVA_TANDA."
    )

if not CARPETA_FOTOS_NUEVA_TANDA.exists():
    raise FileNotFoundError(
        f"No encontré la carpeta de fotos: {CARPETA_FOTOS_NUEVA_TANDA}\n"
        "Crea una carpeta llamada nueva_tanda_fotos con subcarpetas por ID Airbnb."
    )

# ------------------------------------------------------------
# 3. Cargar Excel nuevo
# ------------------------------------------------------------
df_nuevo_excel = pd.read_excel(RUTA_EXCEL_NUEVA_TANDA)

print("Filas del Excel nuevo:", len(df_nuevo_excel))
display(df_nuevo_excel.head())

# ------------------------------------------------------------
# 4. Detectar columna ID Airbnb
# ------------------------------------------------------------
def detectar_columna_id_local(df):
    """
    Detecta la columna que identifica al Airbnb.
    Se hace flexible porque el Excel puede venir con distintos nombres.
    """

    posibles = [
        "id_airbnb",
        "ID Airbnb",
        "ID_Airbnb",
        "airbnb_id",
        "room_id",
        "listing_id",
        "id",
        "ID"
    ]

    for col in posibles:
        if col in df.columns:
            return col

    # Si no encuentra por nombre exacto, busca columnas que parezcan ID
    for col in df.columns:
        col_lower = str(col).lower()
        if "airbnb" in col_lower and "id" in col_lower:
            return col
        if "room" in col_lower and "id" in col_lower:
            return col
        if "listing" in col_lower and "id" in col_lower:
            return col

    raise ValueError(
        "No encontré la columna de ID Airbnb en el Excel nuevo. "
        "Renombra la columna como id_airbnb o ID Airbnb."
    )

ID_COL_NUEVO = detectar_columna_id_local(df_nuevo_excel)

print("Columna ID detectada:", ID_COL_NUEVO)

# Pasar ID a texto para evitar errores en cruces
df_nuevo_excel[ID_COL_NUEVO] = df_nuevo_excel[ID_COL_NUEVO].astype(str).str.strip()

ids_excel = df_nuevo_excel[ID_COL_NUEVO].dropna().astype(str).str.strip().unique().tolist()

print("Airbnb únicos en Excel nuevo:", len(ids_excel))

# ------------------------------------------------------------
# 5. Cargar modelo ganador si no está en memoria
# ------------------------------------------------------------
checkpoint_path = SALIDA_DIR / "modelo_ganador_experimentos_cnn.pt"

if "modelo_ganador" not in globals() or "config_ganadora" not in globals():
    print("Cargando modelo ganador desde checkpoint...")

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"No encontré el modelo ganador en: {checkpoint_path}\n"
            "Primero debes ejecutar la celda de entrenamiento CNN y guardar el modelo ganador."
        )

    checkpoint = torch.load(checkpoint_path, map_location=device)

    modelo_ganador_nombre = checkpoint["modelo"]
    config_ganadora = checkpoint["config"]

    modelo_ganador = crear_modelo(modelo_ganador_nombre).to(device)
    modelo_ganador.load_state_dict(checkpoint["state_dict"])

print("Modelo ganador cargado:", modelo_ganador_nombre)
print("Configuración ganadora:", config_ganadora)

modelo_ganador.eval()

# ------------------------------------------------------------
# 6. Crear manifest de fotos de la nueva tanda
# ------------------------------------------------------------
# Esta parte recorre nueva_tanda_fotos/<ID Airbnb>/ y encuentra imágenes.
# Solo se consideran los IDs que están en el Excel nuevo.

extensiones_validas = {".jpg", ".jpeg", ".png", ".webp"}

registros_fotos = []

for id_airbnb in ids_excel:
    carpeta_id = CARPETA_FOTOS_NUEVA_TANDA / str(id_airbnb)

    if not carpeta_id.exists():
        continue

    fotos = [
        p for p in carpeta_id.rglob("*")
        if p.is_file() and p.suffix.lower() in extensiones_validas
    ]

    for foto in fotos:
        registros_fotos.append(
            {
                ID_COL_NUEVO: str(id_airbnb),
                "path_modelo": str(foto),
                "nombre_archivo": foto.name
            }
        )

df_manifest_nueva_tanda = pd.DataFrame(registros_fotos)

manifest_csv = SALIDA_NUEVA_TANDA / "manifest_fotos_nueva_tanda.csv"
manifest_xlsx = SALIDA_NUEVA_TANDA / "manifest_fotos_nueva_tanda.xlsx"

df_manifest_nueva_tanda.to_csv(manifest_csv, index=False, encoding="utf-8-sig")
df_manifest_nueva_tanda.to_excel(manifest_xlsx, index=False)

print("Fotos encontradas para nueva tanda:", len(df_manifest_nueva_tanda))

if len(df_manifest_nueva_tanda) > 0:
    print("Airbnb con al menos una foto:", df_manifest_nueva_tanda[ID_COL_NUEVO].nunique())
    display(df_manifest_nueva_tanda.head())
else:
    print("No se encontraron fotos para los IDs del Excel.")
    print("Se generará un Excel final con todos los Airbnb, pero sin proporciones calculadas.")

# ------------------------------------------------------------
# 7. Función para predecir una foto
# ------------------------------------------------------------
_, eval_transform_nueva = crear_transforms(
    img_size=config_ganadora["img_size"],
    augmentation=config_ganadora["augmentation"]
)

def predecir_foto_nueva_tanda(path_imagen):
    """
    Aplica el modelo ganador a una imagen.
    Devuelve etiqueta predicha y probabilidades.
    """

    image = Image.open(path_imagen).convert("RGB")
    image_tensor = eval_transform_nueva(image).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast(device_type="cuda", enabled=USAR_AMP):
            output = modelo_ganador(image_tensor)
            probs = torch.softmax(output, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    pred_label = idx_to_label[pred_idx]

    return pred_label, float(probs[0]), float(probs[1])

# ------------------------------------------------------------
# 8. Aplicar modelo a todas las fotos encontradas
# ------------------------------------------------------------
if len(df_manifest_nueva_tanda) > 0:

    predicciones = []

    for i, row in df_manifest_nueva_tanda.iterrows():
        pred_label, prob_media, prob_alta = predecir_foto_nueva_tanda(row["path_modelo"])

        predicciones.append(
            {
                ID_COL_NUEVO: row[ID_COL_NUEVO],
                "path_modelo": row["path_modelo"],
                "nombre_archivo": row["nombre_archivo"],
                "label_calidad_predicha": pred_label,
                "label_calidad_modelo_final": pred_label,
                "prob_media": prob_media,
                "prob_alta": prob_alta
            }
        )

    df_pred_nueva_tanda = pd.DataFrame(predicciones)

else:
    df_pred_nueva_tanda = pd.DataFrame(
        columns=[
            ID_COL_NUEVO,
            "path_modelo",
            "nombre_archivo",
            "label_calidad_predicha",
            "label_calidad_modelo_final",
            "prob_media",
            "prob_alta"
        ]
    )

predicciones_csv = SALIDA_NUEVA_TANDA / "predicciones_fotos_nueva_tanda.csv"
predicciones_xlsx = SALIDA_NUEVA_TANDA / "predicciones_fotos_nueva_tanda.xlsx"

df_pred_nueva_tanda.to_csv(predicciones_csv, index=False, encoding="utf-8-sig")
df_pred_nueva_tanda.to_excel(predicciones_xlsx, index=False)

print("Predicciones generadas:", len(df_pred_nueva_tanda))

if len(df_pred_nueva_tanda) > 0:
    display(df_pred_nueva_tanda.head())

# ------------------------------------------------------------
# 9. Calcular proporciones por Airbnb usando las fotos predichas
# ------------------------------------------------------------
if len(df_pred_nueva_tanda) > 0:

    conteo_nueva = (
        df_pred_nueva_tanda
        .groupby(ID_COL_NUEVO)["label_calidad_modelo_final"]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )

    # Asegurar columnas aunque alguna clase no aparezca
    if "alta" not in conteo_nueva.columns:
        conteo_nueva["alta"] = 0

    if "media" not in conteo_nueva.columns:
        conteo_nueva["media"] = 0

    conteo_nueva["n_fotos_modelo"] = conteo_nueva["alta"] + conteo_nueva["media"]

    conteo_nueva["prop_fotos_buena_calidad"] = (
        conteo_nueva["alta"] / conteo_nueva["n_fotos_modelo"]
    )

    conteo_nueva["prop_fotos_calidad_media"] = (
        conteo_nueva["media"] / conteo_nueva["n_fotos_modelo"]
    )

    conteo_nueva = conteo_nueva.rename(
        columns={
            "alta": "n_fotos_buena_calidad",
            "media": "n_fotos_calidad_media"
        }
    )

    prob_prom_nueva = (
        df_pred_nueva_tanda
        .groupby(ID_COL_NUEVO)["prob_alta"]
        .mean()
        .reset_index()
        .rename(columns={"prob_alta": "prob_alta_promedio"})
    )

    proporciones_nueva_tanda = conteo_nueva.merge(
        prob_prom_nueva,
        on=ID_COL_NUEVO,
        how="left"
    )

else:
    proporciones_nueva_tanda = pd.DataFrame(
        columns=[
            ID_COL_NUEVO,
            "n_fotos_buena_calidad",
            "n_fotos_calidad_media",
            "n_fotos_modelo",
            "prop_fotos_buena_calidad",
            "prop_fotos_calidad_media",
            "prob_alta_promedio"
        ]
    )

# Asegurar ID como texto
proporciones_nueva_tanda[ID_COL_NUEVO] = proporciones_nueva_tanda[ID_COL_NUEVO].astype(str)

prop_nueva_csv = SALIDA_NUEVA_TANDA / "proporciones_calidad_por_airbnb_nueva_tanda.csv"
prop_nueva_xlsx = SALIDA_NUEVA_TANDA / "proporciones_calidad_por_airbnb_nueva_tanda.xlsx"

proporciones_nueva_tanda.to_csv(prop_nueva_csv, index=False, encoding="utf-8-sig")
proporciones_nueva_tanda.to_excel(prop_nueva_xlsx, index=False)

print("Airbnb con proporciones calculadas:", proporciones_nueva_tanda[ID_COL_NUEVO].nunique())

if len(proporciones_nueva_tanda) > 0:
    display(proporciones_nueva_tanda.head())

# ------------------------------------------------------------
# 10. Unir proporciones al Excel original conservando TODOS los Airbnb
# ------------------------------------------------------------
df_nuevo_excel_final = df_nuevo_excel.copy()

df_nuevo_excel_final[ID_COL_NUEVO] = (
    df_nuevo_excel_final[ID_COL_NUEVO]
    .astype(str)
    .str.strip()
)

proporciones_nueva_tanda[ID_COL_NUEVO] = (
    proporciones_nueva_tanda[ID_COL_NUEVO]
    .astype(str)
    .str.strip()
)

df_nueva_tanda_final = df_nuevo_excel_final.merge(
    proporciones_nueva_tanda,
    on=ID_COL_NUEVO,
    how="left"
)

# ------------------------------------------------------------
# 11. Completar Airbnb sin fotos procesadas
# ------------------------------------------------------------
columnas_conteo = [
    "n_fotos_modelo",
    "n_fotos_buena_calidad",
    "n_fotos_calidad_media"
]

for col in columnas_conteo:
    if col not in df_nueva_tanda_final.columns:
        df_nueva_tanda_final[col] = 0

    df_nueva_tanda_final[col] = (
        df_nueva_tanda_final[col]
        .fillna(0)
        .astype(int)
    )

columnas_prop = [
    "prop_fotos_buena_calidad",
    "prop_fotos_calidad_media",
    "prob_alta_promedio"
]

for col in columnas_prop:
    if col not in df_nueva_tanda_final.columns:
        df_nueva_tanda_final[col] = np.nan

# Para proporciones, si no hay fotos, dejamos 0.
# Para prob_alta_promedio, dejamos NaN porque no hubo predicción real.
df_nueva_tanda_final["prop_fotos_buena_calidad"] = (
    df_nueva_tanda_final["prop_fotos_buena_calidad"]
    .fillna(0)
)

df_nueva_tanda_final["prop_fotos_calidad_media"] = (
    df_nueva_tanda_final["prop_fotos_calidad_media"]
    .fillna(0)
)

df_nueva_tanda_final["estado_procesamiento_fotos"] = np.where(
    df_nueva_tanda_final["n_fotos_modelo"] > 0,
    "con_fotos_procesadas",
    "sin_fotos_procesadas"
)

# ------------------------------------------------------------
# 12. Validaciones finales
# ------------------------------------------------------------
print("\nValidación de cantidad de Airbnb:")
print("Airbnb únicos en Excel original:", df_nuevo_excel[ID_COL_NUEVO].nunique())
print("Airbnb únicos en resultado final:", df_nueva_tanda_final[ID_COL_NUEVO].nunique())

print("\nEstado de procesamiento:")
display(
    df_nueva_tanda_final["estado_procesamiento_fotos"]
    .value_counts()
    .reset_index()
    .rename(
        columns={
            "index": "estado",
            "estado_procesamiento_fotos": "cantidad"
        }
    )
)

# ------------------------------------------------------------
# 13. Guardar Excel final enriquecido
# ------------------------------------------------------------
excel_final_nueva_tanda = SALIDA_NUEVA_TANDA / "excel_nueva_tanda_con_proporciones_cnn.xlsx"
csv_final_nueva_tanda = SALIDA_NUEVA_TANDA / "excel_nueva_tanda_con_proporciones_cnn.csv"

df_nueva_tanda_final.to_excel(excel_final_nueva_tanda, index=False)
df_nueva_tanda_final.to_csv(csv_final_nueva_tanda, index=False, encoding="utf-8-sig")

print("\nResultado final de nueva tanda guardado en:")
print(excel_final_nueva_tanda)
print(csv_final_nueva_tanda)

display(df_nueva_tanda_final.head())

### Celda 17 — Texto base de conclusiones

Esta celda genera un texto inicial para el informe. Resume qué hizo el módulo CNN, qué métricas usó, qué significan los errores y cómo se podría mejorar el sistema.


In [32]:
# ============================================================
# CELDA 17 - Texto automático de conclusiones CNN
# Genera un resumen breve para adaptar al informe final.
# ============================================================
from IPython.display import Markdown, display

try:
    mejor_modelo_texto = modelo_ganador_nombre
except NameError:
    mejor_modelo_texto = "modelo ganador"

texto_conclusiones_cnn = f"""
## Conclusiones del módulo CNN

El componente CNN permite evaluar automáticamente la calidad visual de las fotografías de los anuncios de Airbnb. Para esta etapa se trabajó con dos clases: **alta** y **media** utilidad visual. Las imágenes no evaluables fueron excluidas del entrenamiento porque no representan espacios del inmueble.

Se compararon distintas variantes de modelos: una CNN simple entrenada desde cero y modelos preentrenados mediante transfer learning. Esta comparación evita escoger una arquitectura de manera arbitraria. El modelo seleccionado fue **{mejor_modelo_texto}**, debido a que obtuvo el mejor desempeño comparativo según las métricas calculadas.

Las métricas usadas fueron accuracy, precision, recall, F1-score y matriz de confusión. El F1-score es especialmente importante porque resume el equilibrio entre precision y recall. En este problema, un falso positivo ocurre cuando una foto de calidad media se clasifica como alta, lo que puede sobreestimar la calidad visual del anuncio. Un falso negativo ocurre cuando una foto de calidad alta se clasifica como media, lo que puede subestimar el atractivo visual del inmueble.

La principal fortaleza del módulo es que transforma muchas fotos en indicadores resumidos por Airbnb, como la proporción de fotos de buena calidad y de calidad media. Una debilidad es que el dataset visual aún es pequeño, por lo que el modelo puede mejorar si se incorporan más imágenes etiquetadas. Como oportunidad de mejora, se recomienda ampliar el banco de fotos, revisar casos ambiguos y reentrenar periódicamente el modelo con nuevos anuncios.
"""

display(Markdown(texto_conclusiones_cnn))

with open(SALIDA_DIR / "conclusiones_cnn_para_informe.md", "w", encoding="utf-8") as f:
    f.write(texto_conclusiones_cnn)

print("Conclusiones guardadas en:", SALIDA_DIR / "conclusiones_cnn_para_informe.md")



## Conclusiones del módulo CNN

El componente CNN permite evaluar automáticamente la calidad visual de las fotografías de los anuncios de Airbnb. Para esta etapa se trabajó con dos clases: **alta** y **media** utilidad visual. Las imágenes no evaluables fueron excluidas del entrenamiento porque no representan espacios del inmueble.

Se compararon distintas variantes de modelos: una CNN simple entrenada desde cero y modelos preentrenados mediante transfer learning. Esta comparación evita escoger una arquitectura de manera arbitraria. El modelo seleccionado fue **modelo ganador**, debido a que obtuvo el mejor desempeño comparativo según las métricas calculadas.

Las métricas usadas fueron accuracy, precision, recall, F1-score y matriz de confusión. El F1-score es especialmente importante porque resume el equilibrio entre precision y recall. En este problema, un falso positivo ocurre cuando una foto de calidad media se clasifica como alta, lo que puede sobreestimar la calidad visual del anuncio. Un falso negativo ocurre cuando una foto de calidad alta se clasifica como media, lo que puede subestimar el atractivo visual del inmueble.

La principal fortaleza del módulo es que transforma muchas fotos en indicadores resumidos por Airbnb, como la proporción de fotos de buena calidad y de calidad media. Una debilidad es que el dataset visual aún es pequeño, por lo que el modelo puede mejorar si se incorporan más imágenes etiquetadas. Como oportunidad de mejora, se recomienda ampliar el banco de fotos, revisar casos ambiguos y reentrenar periódicamente el modelo con nuevos anuncios.


Conclusiones guardadas en: salida_airbnb_fotos\conclusiones_cnn_para_informe.md


In [ ]:
from pathlib import Path

SALIDA_DIR = Path("salida_airbnb_fotos")
archivo_etiquetas = SALIDA_DIR / "etiquetas_calidad_fotos.csv"

if archivo_etiquetas.exists():
    archivo_etiquetas.unlink()
    print("Archivo de etiquetas eliminado. Puedes re-etiquetar desde cero.")
else:
    print("No había archivo de etiquetas.")

## Guía rápida: ¿desde dónde empezar?

Si ya tengo la carpeta de imágenes descargada, **no debo volver a ejecutar las celdas de descarga o scraping de Airbnb**. Esto evita perder tiempo, duplicar imágenes o volver a descargar fotos que ya existen.

## Caso 1: ya tengo las imágenes, pero no tengo el manifest

Empezar desde la **Celda 6**.

La Celda 6 sirve para crear el archivo de control con las rutas de imágenes y los grupos de trabajo.

## Caso 2: ya existe el manifest, pero falta etiquetar

Si ya existe este archivo:

`salida_airbnb_fotos/dataset_manifest_con_split_calidad.csv`

empezar desde la **Celda 7**.

La Celda 7 sirve para etiquetar manualmente las fotos.

## Caso 3: ya terminé de etiquetar

Si ya existe este archivo:

`salida_airbnb_fotos/etiquetas_calidad_fotos.csv`

y ya terminé de calificar las imágenes, empezar desde la **Celda 8**.

La Celda 8 consolida las etiquetas y prepara el dataset para la CNN.

## Caso 4: ya tengo el dataset CNN listo

Si ya existe este archivo:

`salida_airbnb_fotos/dataset_cnn_calidad_etiquetado.csv`

empezar desde la **Celda 9**.

La Celda 9 prepara las imágenes y los DataLoaders para entrenar el modelo.

## Regla simple

* Tengo fotos, pero no manifest → **Celda 6**
* Tengo manifest, pero faltan etiquetas → **Celda 7**
* Tengo etiquetas completas → **Celda 8**
* Tengo dataset CNN listo → **Celda 9**

## Importante

No volver a ejecutar las celdas de descarga si las fotos ya están en la carpeta local. Solo se deben volver a correr si se quiere descargar una nueva tanda de imágenes o corregir la descarga.
